In [ ]:
%%writefile /kaggle/working/run_lexical.py
"""
Subprocess Dual-Output Lexical Engine
====================================
Subprocess memory isolation running 78,482-row 4-View LinearSVC (Output 1)
and 5-View BPE + SGD + Style Twin + Specialist Complementary Ensemble (Output 2).
"""

import os
import gc
import re
import sys
import time
import hashlib
import unicodedata
import collections
import subprocess
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler

LABELS = [
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5", "claude-fable-5",
    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3"
]
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}

FAMILIES = {
    "gpt": [0, 1, 2, 3, 4],
    "gemini": [5, 6, 7, 8],
    "claude": [9, 10, 11, 12, 13],
    "glm": [15, 16],
}

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - c)
    return e / (np.sum(e, axis=axis, keepdims=True) + 1e-15)

def log_softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable log-softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    return x - c - np.log(np.sum(np.exp(x - c), axis=axis, keepdims=True) + 1e-15)

def _text_hash(text: str) -> str:
    """Normalized SHA1 hash of whitespace-collapsed text to guarantee zero exact/near-exact leakage."""
    norm = " ".join(str(text).strip().split())
    return hashlib.sha1(norm.encode("utf-8", errors="ignore")).hexdigest()

def _norm_text_key(value: Any) -> str:
    """Normalized NFKC string key for exact deduplication."""
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', str(value or ''))).strip()

FUNCTION_WORDS = set(
    "a an the and or but if while although because as so than that which who whom whose "
    "this these those it its they them their theirs he him his she her hers we us our ours "
    "you your yours i me my mine in on at to from with without for of by over under between "
    "through into about is are was were be been being have has had do does did can could "
    "will would shall should may might must not no nor all any some each every both either "
    "neither other another such only also just very too more most less least here there "
    "when where why how what whether then now still yet even much many few little "
    "am having s t re ve ll d m don doesn didn isn aren wasn weren hasn haven hadn won wouldn couldn shouldn".split()
)
STOP_WORDS = FUNCTION_WORDS

def shape_text(t: str) -> str:
    """Replaces alphabetic character blocks with X (uppercase) and x (lowercase)."""
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m.group(0)[0].isupper() else "x", str(t))

def function_text(t: str) -> str:
    """Masks non-stopwords and non-punctuation to CONTENT, preserving function and punctuation syntax."""
    tokens = re.findall(r"(?u)\b\w+\b|[^\w\s]", str(t))
    return " ".join([w.lower() if (w.lower() in STOP_WORDS or not w.isalnum()) else "CONTENT" for w in tokens])

_RE_CODE_BLOCK = re.compile(r"```[\s\S]*?```|`[^`\n]+`")
_RE_STRAIGHT_CONTR = re.compile(r"[A-Za-z]'[A-Za-z]")
_RE_CURLY_CONTR = re.compile(r"[A-Za-z]\u2019[A-Za-z]")

def extract_handwriting_and_orthographic_features(texts: List[str]) -> Tuple[np.ndarray, np.ndarray]:
    """
    Single-pass extractor returning:
    - feats: (N, 26) float32 continuous punctuation, contraction, and structural densities
    - rules: (N, 4) float32 binary indicators for the 4 discrete orthographic invariants:
        col 0: any curly quote (\u2018, \u2019, \u201c, \u201d)
        col 1: >= 2 em-dashes (\u2014) in non-code prose
        col 2: >= 2 straight contractions [A-Za-z]'[A-Za-z] in non-code prose AND 0 curly \u2019
        col 3: >= 1 curly contraction [A-Za-z]\u2019[A-Za-z] in non-code prose
    """
    N = len(texts)
    feats = np.zeros((N, 26), dtype=np.float32)
    rules = np.zeros((N, 4), dtype=np.float32)

    for i, raw_t in enumerate(texts):
        t = str(raw_t)
        prose = _RE_CODE_BLOCK.sub(" ", t)
        t_len = max(len(t), 1)
        words = t.split()
        n_words = max(len(words), 1)

        raw_curly_sq = t.count("\u2018") + t.count("\u2019")
        raw_curly_dq = t.count("\u201c") + t.count("\u201d")
        prose_em_dash = prose.count("\u2014")
        n_str_c = len(_RE_STRAIGHT_CONTR.findall(prose))
        n_cur_c = len(_RE_CURLY_CONTR.findall(prose))

        # 4 Discrete Structural Rule Flags
        rules[i, 0] = 1.0 if (raw_curly_sq + raw_curly_dq) > 0 else 0.0
        rules[i, 1] = 1.0 if prose_em_dash >= 2 else 0.0
        rules[i, 2] = 1.0 if (n_str_c >= 2 and raw_curly_sq == 0) else 0.0
        rules[i, 3] = 1.0 if n_cur_c >= 1 else 0.0

        # 26 Continuous Density Features
        c_colon = t.count(":") / n_words
        c_semi = t.count(";") / n_words
        c_em_dash = t.count("\u2014") / n_words
        c_double_hyph = t.count("--") / n_words
        c_curly_sq = raw_curly_sq / n_words
        c_straight_sq = t.count("'") / n_words
        c_curly_dq = raw_curly_dq / n_words
        c_straight_dq = t.count('"') / n_words
        c_paren = (t.count("(") + t.count(")")) / n_words
        c_comma = t.count(",") / n_words
        c_excl = t.count("!") / n_words
        c_ques = t.count("?") / n_words
        c_ellipsis = (t.count("...") + t.count("\u2026")) / n_words
        c_hyphen = t.count("-") / n_words
        n_straight_contr = n_str_c / n_words
        n_curly_contr = n_cur_c / n_words

        c_newline = t.count("\n") / n_words
        c_bullet = (t.count("\n- ") + t.count("\n* ")) / n_words
        c_upper = sum(1 for ch in t if ch.isupper()) / t_len
        c_digits = sum(1 for ch in t if ch.isdigit()) / t_len

        w_lens = [len(w) for w in words]
        mean_w_len = float(np.mean(w_lens)) if w_lens else 0.0
        std_w_len = float(np.std(w_lens)) if len(w_lens) > 1 else 0.0

        sents = [s.strip() for s in t.replace("!", ".").replace("?", ".").split(".") if s.strip()]
        s_lens = [len(s.split()) for s in sents]
        mean_s_len = float(np.mean(s_lens)) if s_lens else 0.0
        std_s_len = float(np.std(s_lens)) if len(s_lens) > 1 else 0.0
        burstiness = float(std_s_len / max(mean_s_len, 1e-5))
        ttr = float(len(set(words)) / n_words)

        feats[i] = [
            c_colon, c_semi, c_em_dash, c_double_hyph, c_curly_sq, c_straight_sq,
            c_curly_dq, c_straight_dq, c_paren, c_comma, c_excl, c_ques,
            c_ellipsis, c_hyphen, n_straight_contr, n_curly_contr, c_newline,
            c_bullet, c_upper, c_digits, mean_w_len, std_w_len, mean_s_len,
            std_s_len, burstiness, ttr
        ]
    return feats, rules

def extract_handwriting_features(texts: List[str]) -> np.ndarray:
    """Backward-compatible wrapper returning only the (N, 26) continuous matrix."""
    feats, _ = extract_handwriting_and_orthographic_features(texts)
    return feats

def fit_handwriting_model(
    hw_tr: np.ndarray,
    fit_y: np.ndarray,
    hw_te: np.ndarray,
    df_tr: pd.DataFrame,
    groups_085: np.ndarray,
    twin_rows_085: np.ndarray,
    len_tr_raw: int
) -> np.ndarray:
    """
    Fits dedicated LogisticRegression plain (C=0.25) and prompt-centered (C=0.15)
    classifiers strictly on the 26 continuous handwriting features.
    Returns log_p_handwriting (N_test, 18).
    """
    clf_hw_plain = LogisticRegression(C=0.25, max_iter=1000, random_state=42, class_weight="balanced")
    clf_hw_plain.fit(hw_tr, fit_y)
    lp_hw = log_softmax(clf_hw_plain.decision_function(hw_te))

    if twin_rows_085.any() and len_tr_raw >= 18:
        hw_tr_c = hw_tr[:len_tr_raw].copy()
        for p in np.unique(df_tr["prompt_id"].values):
            m_p = (df_tr["prompt_id"].values == p)
            hw_tr_c[m_p] -= hw_tr_c[m_p].mean(axis=0, keepdims=True)
        clf_hw_twin = LogisticRegression(C=0.15, max_iter=1000, random_state=42, class_weight="balanced")
        clf_hw_twin.fit(hw_tr_c, fit_y[:len_tr_raw])

        hw_te_c = hw_te.copy()
        for g in np.unique(groups_085[twin_rows_085]):
            m_g = (groups_085 == g)
            hw_te_c[m_g] -= hw_te_c[m_g].mean(axis=0, keepdims=True)
        lp_hw_twin = log_softmax(clf_hw_twin.decision_function(hw_te_c[twin_rows_085]))
        lp_hw[twin_rows_085] = log_softmax(0.70 * lp_hw[twin_rows_085] + 0.30 * lp_hw_twin)

    return lp_hw

def train_lexical_engine(
    fit_texts: List[str],
    fit_y: np.ndarray,
    fit_sw: Optional[np.ndarray] = None,
    df_tr: Optional[pd.DataFrame] = None,
    qwen_tok_dir: Optional[str] = None
) -> Dict[str, Any]:
    """
    Trains all TF-IDF vectorizers, linear classifiers, handwriting density models,
    and specialist refiners offline on train + synthetic pool data.
    Returns a self-contained bundle dictionary that can be saved with save_lexical_bundle.
    """
    if fit_sw is None:
        fit_sw = np.ones(len(fit_texts), dtype=np.float32)
    len_tr_raw = len(df_tr) if df_tr is not None else 0

    VIEWS_4V = [
        ("char", 3, 5, 250000, 1.0),
        ("word", 1, 3, 180000, 0.7),
        ("shape", 2, 6, 60000, 0.5),
        ("function", 1, 3, 50000, 0.5),
    ]

    fit_sh = [shape_text(t) for t in fit_texts]
    fit_fn = [function_text(t) for t in fit_texts]

    vectorizers_4v = []
    tr_mats_4v = []
    for mode, mn, mx, mf, sc in VIEWS_4V:
        analyzer = "char" if mode == "shape" else ("word" if mode == "function" else mode)
        opts = dict(
            analyzer=analyzer,
            ngram_range=(mn, mx),
            lowercase=False,
            sublinear_tf=True,
            min_df=2 if len(fit_texts) >= 50 else 1,
            max_features=mf,
            dtype=np.float32,
        )
        if analyzer == "word":
            opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"

        v = TfidfVectorizer(**opts)
        if mode == "shape":
            tr_m = v.fit_transform(fit_sh) * np.float32(sc)
        elif mode == "function":
            tr_m = v.fit_transform(fit_fn) * np.float32(sc)
        else:
            tr_m = v.fit_transform(fit_texts) * np.float32(sc)
        vectorizers_4v.append(v)
        tr_mats_4v.append(tr_m)

    X_tr_4v = sparse.hstack(tr_mats_4v, format="csr")

    # OUTPUT 1: Pure 4-View LinearSVC (C=0.5, class_weight='balanced', source sample weights)
    svc_main = LinearSVC(C=0.5, random_state=42, max_iter=5000, tol=1e-3, class_weight="balanced")
    svc_main.fit(X_tr_4v, fit_y, sample_weight=fit_sw)

    # OUTPUT 2: Stamatatos Text Distortion Model (keeps top-100 words, masks rest with '*')
    wre = re.compile(r"[A-Za-z’']+")
    cnt = collections.Counter(w.lower() for t in fit_texts for w in wre.findall(t))
    distort_keep = {w for w, _ in cnt.most_common(100)}
    dist_fn = lambda arr: [wre.sub(lambda m_: m_.group(0) if m_.group(0).lower() in distort_keep else '*', t) for t in arr]

    D_tr = dist_fn(fit_texts)
    vs_dist = [
        TfidfVectorizer(sublinear_tf=True, lowercase=False, analyzer='char', ngram_range=(2, 6), min_df=2 if len(fit_texts) >= 50 else 1),
        TfidfVectorizer(sublinear_tf=True, lowercase=True, token_pattern=r"[^\s]+", ngram_range=(1, 3), min_df=2 if len(fit_texts) >= 50 else 1)
    ]
    Xa_dist = sparse.hstack([v_.fit_transform(D_tr) for v_ in vs_dist], format="csr")

    lr_distort = LogisticRegression(C=10.0, max_iter=3000, random_state=42)
    lr_distort.fit(Xa_dist, fit_y, sample_weight=fit_sw)

    # OUTPUT 3: Continuous Handwriting Density Model
    hw_tr_raw, _ = extract_handwriting_and_orthographic_features(fit_texts)
    scaler = StandardScaler()
    hw_tr = np.clip(scaler.fit_transform(hw_tr_raw), -5.0, 5.0).astype(np.float32)

    clf_hw_plain = LogisticRegression(C=0.25, max_iter=1000, random_state=42, class_weight="balanced")
    clf_hw_plain.fit(hw_tr, fit_y)

    clf_hw_twin = None
    if df_tr is not None and len_tr_raw >= 18 and "prompt_id" in df_tr.columns:
        hw_tr_c = hw_tr[:len_tr_raw].copy()
        for p in np.unique(df_tr["prompt_id"].values):
            m_p = (df_tr["prompt_id"].values == p)
            hw_tr_c[m_p] -= hw_tr_c[m_p].mean(axis=0, keepdims=True)
        clf_hw_twin = LogisticRegression(C=0.15, max_iter=1000, random_state=42, class_weight="balanced")
        clf_hw_twin.fit(hw_tr_c, fit_y[:len_tr_raw])

    # 5th View: BPE or Character-n-gram
    tok = None
    if qwen_tok_dir and os.path.exists(qwen_tok_dir):
        try:
            from transformers import AutoTokenizer
            tok = AutoTokenizer.from_pretrained(qwen_tok_dir, use_fast=True)
        except Exception:
            tok = None

    if tok is not None:
        def batch_bpe_strings(texts: List[str], batch_sz: int = 4000) -> List[str]:
            out_strs = []
            for s in range(0, len(texts), batch_sz):
                batch_ids = tok(
                    texts[s : s + batch_sz],
                    add_special_tokens=False,
                    truncation=True,
                    max_length=512,
                )["input_ids"]
                out_strs.extend(" ".join(map(str, ids)) for ids in batch_ids)
            return out_strs

        fit_bpe_strs = batch_bpe_strings(fit_texts)
        v_5th = TfidfVectorizer(
            analyzer="word", token_pattern=r"\S+", lowercase=False,
            ngram_range=(1, 3), max_features=75000, sublinear_tf=True, dtype=np.float32
        )
        tr_5th = v_5th.fit_transform(fit_bpe_strs) * 0.8
        is_bpe = True
        del fit_bpe_strs
        gc.collect()
    else:
        v_5th = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), max_features=75000, sublinear_tf=True, dtype=np.float32)
        tr_5th = v_5th.fit_transform(fit_texts) * 0.8
        is_bpe = False

    X_tr_5v = sparse.hstack([X_tr_4v, tr_5th], format="csr")

    svc_comp = LinearSVC(C=0.35, random_state=42, max_iter=3000, tol=1e-3, class_weight="balanced")
    svc_comp.fit(X_tr_5v, fit_y, sample_weight=fit_sw)

    sgd_comp = SGDClassifier(loss='log_loss', alpha=1e-4, max_iter=1500, random_state=42)
    sgd_comp.fit(X_tr_5v, fit_y, sample_weight=fit_sw)

    svc_twin = None
    if df_tr is not None and len_tr_raw >= 18 and "prompt_id" in df_tr.columns:
        X_train_only = X_tr_5v[:len_tr_raw]
        u_tr, inv_tr = np.unique(df_tr['prompt_id'].values, return_inverse=True)
        counts_tr = np.bincount(inv_tr)
        vals_tr = (1.0 / counts_tr[inv_tr]).astype(np.float32)
        G = sparse.csr_matrix((vals_tr, (np.arange(len(inv_tr)), inv_tr)), shape=(len(inv_tr), len(u_tr)), dtype=np.float32)
        X_train_c = X_train_only - G @ (G.T @ X_train_only)

        svc_twin = LinearSVC(C=0.30, random_state=42, max_iter=2500)
        svc_twin.fit(X_train_c, fit_y[:len_tr_raw])
        del X_train_only, G, X_train_c
        gc.collect()

    specialist_svcs = {}
    for fam_name, fam_cols in FAMILIES.items():
        fam_mask = np.isin(fit_y, fam_cols)
        if fam_mask.sum() > 50:
            sub_y = fit_y[fam_mask]
            sub_lab2i = {c: i for i, c in enumerate(fam_cols)}
            sub_y_mapped = np.array([sub_lab2i[y] for y in sub_y])

            spec_svc = LinearSVC(C=0.40, random_state=42, max_iter=2000, tol=1e-3, class_weight="balanced")
            spec_svc.fit(X_tr_5v[fam_mask], sub_y_mapped)
            specialist_svcs[fam_name] = spec_svc

    bundle = {
        "views_4v_specs": VIEWS_4V,
        "vectorizers_4v": vectorizers_4v,
        "svc_main": svc_main,
        "distort_keep": sorted(list(distort_keep)),
        "vs_dist": vs_dist,
        "lr_distort": lr_distort,
        "hw_scaler": scaler,
        "clf_hw_plain": clf_hw_plain,
        "clf_hw_twin": clf_hw_twin,
        "is_bpe": is_bpe,
        "v_5th": v_5th,
        "svc_comp": svc_comp,
        "sgd_comp": sgd_comp,
        "svc_twin": svc_twin,
        "specialist_svcs": specialist_svcs,
        "families": FAMILIES,
        "labels": LABELS,
    }
    return bundle

def save_lexical_bundle(bundle: Dict[str, Any], path: Path | str) -> None:
    """Serializes the lexical bundle to disk using compressed joblib."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(bundle, path, compress=3)
    sz_mb = path.stat().st_size / (1024 * 1024)
    print(f"[Lexical] Saved lexical bundle to {path} ({sz_mb:.2f} MB)")

def load_lexical_bundle(path: Path | str) -> Dict[str, Any]:
    """Loads a pre-trained lexical bundle from disk."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Lexical bundle not found at {path}")
    t0 = time.time()
    bundle = joblib.load(path)
    print(f"[Lexical] Loaded lexical bundle from {path} in {time.time() - t0:.2f}s")
    return bundle

def predict_lexical_engine(
    bundle: Dict[str, Any],
    test_texts: List[str],
    unseen_texts: Optional[List[str]] = None,
    qwen_tok_dir: Optional[str] = None,
    working_dir: Optional[Path | str] = None,
) -> Dict[str, np.ndarray]:
    """
    Executes fast inference using a pre-trained lexical bundle on test_texts and optional unseen_texts.
    Takes ~10-20 seconds on CPU.
    """
    t0 = time.time()
    unseen_texts = unseen_texts or []

    # 1. Compute stopword TF-IDF distance matrix and topic clusters (t=0.85) on test
    if len(test_texts) >= 2:
        Xt_tfidf = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=100000, dtype=np.float32).fit_transform(test_texts)
        D = np.clip(1.0 - (Xt_tfidf @ Xt_tfidf.T).toarray(), 0.0, None)
        np.fill_diagonal(D, 0.0)
        cond_d = squareform(D, checks=False)
        groups_085 = fcluster(linkage(cond_d, method='average'), 0.85, criterion='distance')
    else:
        D = np.zeros((len(test_texts), len(test_texts)), dtype=np.float32)
        groups_085 = np.ones(len(test_texts), dtype=int)

    sz_085 = np.bincount(groups_085)[groups_085] if len(groups_085) > 0 else np.array([], dtype=int)
    twin_rows_085 = (sz_085 >= 2) & (sz_085 <= 18) if len(sz_085) > 0 else np.zeros(0, dtype=bool)

    # Stopword TF-IDF and clusters on unseen holdout benchmark
    if len(unseen_texts) >= 2:
        Xu_tfidf = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=100000, dtype=np.float32).fit_transform(unseen_texts)
        D_u = np.clip(1.0 - (Xu_tfidf @ Xu_tfidf.T).toarray(), 0.0, None)
        np.fill_diagonal(D_u, 0.0)
        cond_du = squareform(D_u, checks=False)
        groups_085_u = fcluster(linkage(cond_du, method='average'), 0.85, criterion='distance')
    else:
        D_u = np.zeros((len(unseen_texts), len(unseen_texts)), dtype=np.float32)
        groups_085_u = np.ones(len(unseen_texts), dtype=int)
    sz_085_u = np.bincount(groups_085_u)[groups_085_u] if len(groups_085_u) > 0 else np.array([], dtype=int)
    twin_rows_085_u = (sz_085_u >= 2) & (sz_085_u <= 18) if len(sz_085_u) > 0 else np.zeros(0, dtype=bool)

    # 2. 4-View Transform
    test_sh = [shape_text(t) for t in test_texts]
    unseen_sh = [shape_text(t) for t in unseen_texts] if unseen_texts else []
    test_fn = [function_text(t) for t in test_texts]
    unseen_fn = [function_text(t) for t in unseen_texts] if unseen_texts else []

    te_mats_4v = []
    unseen_mats_4v = []
    for (mode, mn, mx, mf, sc), v in zip(bundle["views_4v_specs"], bundle["vectorizers_4v"]):
        if mode == "shape":
            te_m = v.transform(test_sh) * np.float32(sc)
            u_m = v.transform(unseen_sh) * np.float32(sc) if unseen_texts else None
        elif mode == "function":
            te_m = v.transform(test_fn) * np.float32(sc)
            u_m = v.transform(unseen_fn) * np.float32(sc) if unseen_texts else None
        else:
            te_m = v.transform(test_texts) * np.float32(sc)
            u_m = v.transform(unseen_texts) * np.float32(sc) if unseen_texts else None
        te_mats_4v.append(te_m)
        if u_m is not None:
            unseen_mats_4v.append(u_m)

    X_te_4v = sparse.hstack(te_mats_4v, format="csr")
    X_unseen_4v = sparse.hstack(unseen_mats_4v, format="csr") if unseen_mats_4v else None

    # OUTPUT 1: Pure 4-View LinearSVC
    svc_main = bundle["svc_main"]
    logits_svc_main = svc_main.decision_function(X_te_4v)
    log_p_svc_main = log_softmax(logits_svc_main)

    if X_unseen_4v is not None and X_unseen_4v.shape[0] > 0:
        unseen_log_p_svc_main = log_softmax(svc_main.decision_function(X_unseen_4v))
    else:
        unseen_log_p_svc_main = np.zeros((0, K), dtype=np.float32)

    # OUTPUT 2: Text Distortion Model
    wre = re.compile(r"[A-Za-z’']+")
    distort_keep = set(bundle["distort_keep"])
    dist_fn = lambda arr: [wre.sub(lambda m_: m_.group(0) if m_.group(0).lower() in distort_keep else '*', t) for t in arr]

    D_te = dist_fn(test_texts)
    D_unseen = dist_fn(unseen_texts) if unseen_texts else []
    vs_dist = bundle["vs_dist"]
    lr_distort = bundle["lr_distort"]

    Xb_dist = sparse.hstack([v_.transform(D_te) for v_ in vs_dist], format="csr")
    Xu_dist = sparse.hstack([v_.transform(D_unseen) for v_ in vs_dist], format="csr") if unseen_texts else None

    P_distort = lr_distort.predict_proba(Xb_dist)
    log_p_distort = lr_distort.predict_log_proba(Xb_dist)
    if Xu_dist is not None and Xu_dist.shape[0] > 0:
        P_unseen_distort = lr_distort.predict_proba(Xu_dist)
        log_p_unseen_distort = lr_distort.predict_log_proba(Xu_dist)
    else:
        P_unseen_distort = None
        log_p_unseen_distort = None

    # OUTPUT 3: Continuous Handwriting & Orthographic Features
    scaler = bundle["hw_scaler"]
    clf_hw_plain = bundle["clf_hw_plain"]
    clf_hw_twin = bundle.get("clf_hw_twin")

    hw_te_raw, rule_flags_te = extract_handwriting_and_orthographic_features(test_texts)
    hw_te = np.clip(scaler.transform(hw_te_raw), -5.0, 5.0).astype(np.float32)
    lp_hw = log_softmax(clf_hw_plain.decision_function(hw_te))

    if clf_hw_twin is not None and twin_rows_085.any():
        hw_te_c = hw_te.copy()
        for g in np.unique(groups_085[twin_rows_085]):
            m_g = (groups_085 == g)
            hw_te_c[m_g] -= hw_te_c[m_g].mean(axis=0, keepdims=True)
        lp_hw_twin = log_softmax(clf_hw_twin.decision_function(hw_te_c[twin_rows_085]))
        lp_hw[twin_rows_085] = log_softmax(0.70 * lp_hw[twin_rows_085] + 0.30 * lp_hw_twin)

    if unseen_texts:
        hw_u_raw, rule_flags_u = extract_handwriting_and_orthographic_features(unseen_texts)
        hw_u = np.clip(scaler.transform(hw_u_raw), -5.0, 5.0).astype(np.float32)
        lp_hw_u = log_softmax(clf_hw_plain.decision_function(hw_u))
        if clf_hw_twin is not None and twin_rows_085_u.any():
            hw_u_c = hw_u.copy()
            for g in np.unique(groups_085_u[twin_rows_085_u]):
                m_g = (groups_085_u == g)
                hw_u_c[m_g] -= hw_u_c[m_g].mean(axis=0, keepdims=True)
            lp_hw_u_twin = log_softmax(clf_hw_twin.decision_function(hw_u_c[twin_rows_085_u]))
            lp_hw_u[twin_rows_085_u] = log_softmax(0.70 * lp_hw_u[twin_rows_085_u] + 0.30 * lp_hw_u_twin)
    else:
        rule_flags_u = np.zeros((0, 4), dtype=np.float32)
        lp_hw_u = np.zeros((0, K), dtype=np.float32)

    # 5th View Transform
    is_bpe = bundle.get("is_bpe", False)
    v_5th = bundle["v_5th"]

    if is_bpe:
        tok = None
        if qwen_tok_dir and os.path.exists(qwen_tok_dir):
            try:
                from transformers import AutoTokenizer
                tok = AutoTokenizer.from_pretrained(qwen_tok_dir, use_fast=True)
            except Exception:
                tok = None

        if tok is not None:
            def batch_bpe_strings(texts: List[str], batch_sz: int = 4000) -> List[str]:
                out_strs = []
                for s in range(0, len(texts), batch_sz):
                    batch_ids = tok(
                        texts[s : s + batch_sz],
                        add_special_tokens=False,
                        truncation=True,
                        max_length=512,
                    )["input_ids"]
                    out_strs.extend(" ".join(map(str, ids)) for ids in batch_ids)
                return out_strs

            test_bpe_strs = batch_bpe_strings(test_texts)
            unseen_bpe_strs = batch_bpe_strings(unseen_texts) if unseen_texts else []
            te_5th = v_5th.transform(test_bpe_strs) * 0.8
            unseen_5th = v_5th.transform(unseen_bpe_strs) * 0.8 if unseen_texts else None
            del test_bpe_strs, unseen_bpe_strs
        else:
            te_5th = v_5th.transform(test_texts) * 0.8
            unseen_5th = v_5th.transform(unseen_texts) * 0.8 if unseen_texts else None
    else:
        te_5th = v_5th.transform(test_texts) * 0.8
        unseen_5th = v_5th.transform(unseen_texts) * 0.8 if unseen_texts else None

    X_te_5v = sparse.hstack([X_te_4v, te_5th], format="csr")
    X_unseen_5v = sparse.hstack([X_unseen_4v, unseen_5th], format="csr") if unseen_5th is not None else None

    # Complementary Ensemble
    svc_comp = bundle["svc_comp"]
    sgd_comp = bundle["sgd_comp"]
    svc_twin = bundle.get("svc_twin")

    P_svc_comp = softmax(svc_comp.decision_function(X_te_5v), axis=1)
    P_sgd_comp = sgd_comp.predict_proba(X_te_5v)
    P_twin_comp = P_svc_comp.copy()

    if svc_twin is not None and twin_rows_085.any():
        X_te_twin = X_te_5v[twin_rows_085]
        u_t, inv_t = np.unique(groups_085[twin_rows_085], return_inverse=True)
        counts_t = np.bincount(inv_t)
        vals_t = (1.0 / counts_t[inv_t]).astype(np.float32)
        G_t = sparse.csr_matrix((vals_t, (np.arange(len(inv_t)), inv_t)), shape=(len(inv_t), len(u_t)), dtype=np.float32)
        X_te_twin_c = X_te_twin - G_t @ (G_t.T @ X_te_twin)
        P_twin_comp[twin_rows_085] = softmax(svc_twin.decision_function(X_te_twin_c), axis=1)
        del X_te_twin, G_t, X_te_twin_c

    P_lex_comp = 0.40 * P_svc_comp + 0.25 * P_sgd_comp + 0.15 * P_twin_comp + 0.20 * P_distort
    log_p_lex_comp = np.log(np.clip(P_lex_comp, 1e-9, 1.0))

    P_unseen_comp = None
    log_p_unseen_comp = None
    if X_unseen_5v is not None and X_unseen_5v.shape[0] > 0:
        P_unseen_svc = softmax(svc_comp.decision_function(X_unseen_5v), axis=1)
        P_unseen_sgd = sgd_comp.predict_proba(X_unseen_5v)
        P_twin_comp_u = P_unseen_svc.copy()
        if svc_twin is not None and len(twin_rows_085_u) > 0 and twin_rows_085_u.any():
            X_u_twin = X_unseen_5v[twin_rows_085_u]
            u_u, inv_u = np.unique(groups_085_u[twin_rows_085_u], return_inverse=True)
            counts_u = np.bincount(inv_u)
            vals_u = (1.0 / counts_u[inv_u]).astype(np.float32)
            G_u = sparse.csr_matrix((vals_u, (np.arange(len(inv_u)), inv_u)), shape=(len(inv_u), len(u_u)), dtype=np.float32)
            X_u_twin_c = X_u_twin - G_u @ (G_u.T @ X_u_twin)
            P_twin_comp_u[twin_rows_085_u] = softmax(svc_twin.decision_function(X_u_twin_c), axis=1)
            del X_u_twin, G_u, X_u_twin_c

        u_twin = P_twin_comp_u if P_twin_comp_u is not None else P_unseen_svc
        if P_unseen_distort is not None:
            P_unseen_comp = 0.40 * P_unseen_svc + 0.25 * P_unseen_sgd + 0.15 * u_twin + 0.20 * P_unseen_distort
        else:
            P_unseen_comp = 0.55 * P_unseen_svc + 0.45 * P_unseen_sgd
        log_p_unseen_comp = np.log(np.clip(P_unseen_comp, 1e-9, 1.0))

    # 4 Within-Family Specialist Refiners
    beta = 0.25
    specialist_svcs = bundle.get("specialist_svcs", {})
    families = bundle.get("families", FAMILIES)

    for fam_name, spec_svc in specialist_svcs.items():
        if fam_name not in families:
            continue
        fam_cols = families[fam_name]
        df_spec = spec_svc.decision_function(X_te_5v)
        logits_spec = np.column_stack([-df_spec, df_spec]) if df_spec.ndim == 1 else df_spec
        P_spec = softmax(logits_spec, axis=1)

        P_prior = P_lex_comp[:, fam_cols]
        P_prior_norm = P_prior / (P_prior.sum(axis=1, keepdims=True) + 1e-12)
        P_refined = (1.0 - beta) * P_prior_norm + beta * P_spec

        log_ratio = np.log(np.clip(P_refined, 1e-9, 1.0)) - np.log(np.clip(P_prior_norm, 1e-9, 1.0))
        log_p_lex_comp[:, fam_cols] += log_ratio

        if log_p_unseen_comp is not None and X_unseen_5v is not None and X_unseen_5v.shape[0] > 0:
            df_spec_u = spec_svc.decision_function(X_unseen_5v)
            logits_spec_u = np.column_stack([-df_spec_u, df_spec_u]) if df_spec_u.ndim == 1 else df_spec_u
            P_spec_u = softmax(logits_spec_u, axis=1)
            P_prior_u = P_unseen_comp[:, fam_cols]
            P_prior_u_norm = P_prior_u / (P_prior_u.sum(axis=1, keepdims=True) + 1e-12)
            P_refined_u = (1.0 - beta) * P_prior_u_norm + beta * P_spec_u
            log_ratio_u = np.log(np.clip(P_refined_u, 1e-9, 1.0)) - np.log(np.clip(P_prior_u_norm, 1e-9, 1.0))
            log_p_unseen_comp[:, fam_cols] += log_ratio_u

    P_lex_comp_final = softmax(log_p_lex_comp, axis=1) if len(test_texts) > 0 else np.zeros((0, K), dtype=np.float32)
    unseen_p_lex_comp_final = softmax(log_p_unseen_comp, axis=1) if (log_p_unseen_comp is not None and len(unseen_texts) > 0) else np.zeros((0, K), dtype=np.float32)

    # Save to disk if working_dir provided
    if working_dir is not None:
        working_dir = Path(working_dir)
        os.makedirs(working_dir, exist_ok=True)
        np.save(working_dir / 'log_p_svc_main.npy', log_p_svc_main)
        np.save(working_dir / 'p_lex_comp.npy', P_lex_comp_final)
        np.save(working_dir / 'D_tfidf.npy', D)
        np.save(working_dir / 'groups_085.npy', groups_085)
        np.save(working_dir / 'log_p_handwriting.npy', lp_hw)
        np.save(working_dir / 'rule_flags_test.npy', rule_flags_te)
        np.save(working_dir / 'unseen_log_p_svc_main.npy', unseen_log_p_svc_main)
        np.save(working_dir / 'unseen_p_lex_comp.npy', unseen_p_lex_comp_final)
        if 'D_u' in locals() and D_u is not None:
            np.save(working_dir / 'D_tfidf_unseen.npy', D_u)
        if 'groups_085_u' in locals() and groups_085_u is not None:
            np.save(working_dir / 'groups_085_unseen.npy', groups_085_u)
        if 'lp_hw_u' in locals() and lp_hw_u is not None:
            np.save(working_dir / 'log_p_handwriting_unseen.npy', lp_hw_u)
        if 'rule_flags_u' in locals() and rule_flags_u is not None:
            np.save(working_dir / 'rule_flags_unseen.npy', rule_flags_u)
        if log_p_distort is not None:
            np.save(working_dir / 'log_p_distort.npy', log_p_distort)
        if log_p_unseen_distort is not None:
            np.save(working_dir / 'unseen_log_p_distort.npy', log_p_unseen_distort)

    print(f"[Lexical] predict_lexical_engine completed in {time.time() - t0:.2f}s")
    return {
        "log_p_svc_main": log_p_svc_main,
        "p_lex_comp": P_lex_comp_final,
        "D_tfidf": D,
        "groups_085": groups_085,
        "log_p_handwriting": lp_hw,
        "rule_flags_test": rule_flags_te,
        "unseen_log_p_svc_main": unseen_log_p_svc_main,
        "unseen_p_lex_comp": unseen_p_lex_comp_final,
        "D_tfidf_unseen": D_u if ('D_u' in locals() and D_u is not None) else None,
        "groups_085_unseen": groups_085_u if ('groups_085_u' in locals() and groups_085_u is not None) else None,
        "log_p_handwriting_unseen": lp_hw_u if ('lp_hw_u' in locals() and lp_hw_u is not None) else None,
        "rule_flags_unseen": rule_flags_u if ('rule_flags_u' in locals() and rule_flags_u is not None) else None,
        "log_p_distort": log_p_distort,
        "unseen_log_p_distort": log_p_unseen_distort,
    }

def run_lexical_engine(
    train_path: Optional[Path] = None,
    test_path: Optional[Path] = None,
    pool_file: Optional[Path] = None,
    qwen_tok_dir: str = "",
    working_dir: Optional[Path] = None,
    bundle_path: Optional[Path | str] = None,
) -> Dict[str, np.ndarray]:
    """
    Runs the lexical engine with automatic pre-trained bundle auto-discovery.
    If a pre-trained bundle exists, executes pure inference in ~15-30s.
    Otherwise fits models from scratch on train + pool data.
    """
    working_dir = Path(working_dir) if working_dir else Path(".")
    working_dir.mkdir(parents=True, exist_ok=True)

    # 1. Search for pre-trained bundle
    bundle = None
    bundle_candidates = []
    if bundle_path:
        bundle_candidates.append(Path(bundle_path))
    if working_dir:
        bundle_candidates.append(working_dir / "lexical_bundle.joblib")
    bundle_candidates.extend([
        Path("local_data/statistical_heads/lexical_bundle.joblib"),
        Path("/kaggle/input/ntl-statistical-heads/lexical_bundle.joblib"),
    ])
    if Path("/kaggle/input").exists():
        for p in Path("/kaggle/input").rglob("lexical_bundle.joblib"):
            if p not in bundle_candidates:
                bundle_candidates.append(p)

    for cand in bundle_candidates:
        if cand.exists():
            try:
                bundle = load_lexical_bundle(cand)
                print(f"[Lexical] Using pre-trained bundle from {cand}")
                break
            except Exception as e:
                print(f"[Lexical] Could not load candidate bundle {cand}: {e}")

    # 2. Extract test and unseen texts
    if test_path and str(test_path).endswith('.parquet') and Path(test_path).exists():
        df_te = pd.read_parquet(test_path)
    elif test_path and Path(test_path).exists():
        df_te = pd.read_csv(test_path)
    else:
        df_te = pd.DataFrame({"text": []})
    test_texts = df_te['text'].fillna('').astype(str).tolist()

    unseen_parquet = working_dir / "unseen_holdout.parquet"
    unseen_texts: List[str] = []
    unseen_hashes = set()
    if unseen_parquet.exists():
        try:
            df_unseen_holdout = pd.read_parquet(unseen_parquet)
            if len(df_unseen_holdout) > 0 and "text" in df_unseen_holdout.columns:
                unseen_texts = df_unseen_holdout["text"].fillna("").astype(str).tolist()
                if "_hash" in df_unseen_holdout.columns:
                    unseen_hashes = set(df_unseen_holdout["_hash"].tolist())
                else:
                    unseen_hashes = set(_text_hash(t) for t in unseen_texts)
                print(f"[Lexical] Found unseen holdout set: {len(unseen_texts)} rows quarantined from training.")
        except Exception as e:
            print(f"[Lexical] Note: Could not read {unseen_parquet}: {e}")

    # 3. If pre-trained bundle found, predict immediately!
    if bundle is not None:
        return predict_lexical_engine(
            bundle=bundle,
            test_texts=test_texts,
            unseen_texts=unseen_texts,
            qwen_tok_dir=qwen_tok_dir,
            working_dir=working_dir,
        )

    # 4. Fall back to training from scratch
    print("[Lexical] No pre-trained bundle found. Training lexical engine from scratch...")
    if train_path is None or not Path(train_path).exists():
        raise FileNotFoundError(f"Training path {train_path} not found and no pre-trained bundle available.")

    df_tr = pd.read_csv(train_path)
    fit_texts = df_tr['text'].fillna('').astype(str).tolist()
    fit_y = np.array([LAB2I[m] for m in df_tr['model']])
    fit_sw = np.ones(len(df_tr), dtype=np.float32)  # Official train rows get 1.00
    seen_texts = {_norm_text_key(t) for t in fit_texts}

    # Load pool parquet files if present with source weights
    pool_candidates = [pool_file] if isinstance(pool_file, (str, Path)) else (pool_file if isinstance(pool_file, list) else [])
    for pf in pool_candidates:
        if pf and os.path.exists(pf):
            try:
                p_df = pd.read_parquet(pf, columns=['text', 'model'])
                p_df = p_df[p_df['model'].isin(LABELS)].dropna(subset=['text', 'model']).copy()
                pf_name = str(pf).lower().replace('\\', '/')
                # Matched clean gets 0.50, general scraped pool gets 0.35
                src_w = 0.50 if ('matched' in pf_name or 'clean' in pf_name) else 0.35

                keep_mask = []
                for t in p_df['text'].values:
                    k = _norm_text_key(t)
                    h = _text_hash(t)
                    ok = bool(k) and (k not in seen_texts) and (h not in unseen_hashes)
                    keep_mask.append(ok)
                    if ok:
                        seen_texts.add(k)
                p_df = p_df.loc[keep_mask].reset_index(drop=True)
                if len(p_df) > 0:
                    p_texts = p_df['text'].fillna('').astype(str).tolist()
                    p_y = p_df['model'].map(LAB2I).values
                    p_sw = np.full(len(p_df), src_w, dtype=np.float32)
                    fit_texts.extend(p_texts)
                    fit_y = np.r_[fit_y, p_y]
                    fit_sw = np.r_[fit_sw, p_sw]
                    print(f"[Lexical] Added {len(p_df)} rows from {Path(pf).name} with source weight {src_w}")
            except Exception as e:
                print(f"[Lexical] Note: Could not read pool {pf}: {e}")

    bundle = train_lexical_engine(
        fit_texts=fit_texts,
        fit_y=fit_y,
        fit_sw=fit_sw,
        df_tr=df_tr,
        qwen_tok_dir=qwen_tok_dir,
    )

    # Optional: save the newly trained bundle to working_dir
    try:
        save_lexical_bundle(bundle, working_dir / "lexical_bundle.joblib")
    except Exception as e:
        print(f"[Lexical] Note: Could not save bundle: {e}")

    return predict_lexical_engine(
        bundle=bundle,
        test_texts=test_texts,
        unseen_texts=unseen_texts,
        qwen_tok_dir=qwen_tok_dir,
        working_dir=working_dir,
    )

if __name__ == "__main__":
    working_dir = Path("/kaggle/working")
    if not working_dir.exists():
        working_dir = Path("local_data")
    working_dir.mkdir(parents=True, exist_ok=True)

    input_dir = Path("/kaggle/input")
    if not input_dir.exists():
        input_dir = Path(".")

    candidates = [
        input_dir / 'competitions' / 'name-that-llm',
        input_dir / 'name-that-llm',
        Path('local_data'),
        Path('.')
    ]
    train_path = None
    test_path = None
    for c in candidates:
        if (c / 'train.csv').exists() and train_path is None:
            train_path = c / 'train.csv'
        if (c / 'test.csv').exists() and test_path is None:
            test_path = c / 'test.csv'
    if train_path is None:
        train_path = Path('local_data') / 'train.csv'
    if test_path is None:
        test_path = Path('local_data') / 'test.csv'

    pool_files: List[Path] = []
    for pat in ['nlm_pool_v5.parquet', 'pool_v4.parquet', 'nlm_matched_clean.parquet', 'nlm_pool_v2.parquet']:
        for p in input_dir.rglob(pat):
            if p not in pool_files:
                pool_files.append(p)

    tok_paths = list(input_dir.rglob("vocab.json"))
    qwen_tok_dir = str(tok_paths[0].parent) if tok_paths else ""

    print(f"[Module 1 CLI] Running lexical engine: train={train_path}, test={test_path}, pools={[p.name for p in pool_files]}")
    run_lexical_engine(train_path, test_path, pool_files, qwen_tok_dir, working_dir)
    print("[Module 1 CLI] Lexical engine finished successfully.")


In [ ]:
# === MODULE: src/namethatllm/config.py ===
ENSEMBLE_CONFIG = {'version': '1.0.0', 'name': 'production_ensemble', 'description': 'Production Two-Tier Stack (85% Main Part-Two Baseline + 15% Prior-Scaled Complementary Stack + Unsupervised Spectral Bank)', 'probe_gate_min_bacc': 0.4, 'execution_mode': 'benchmark_only', 'run_benchmark': True, 'run_submission': False, 'enable_ablation_probe': True, 'labels': ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'], 'families': {'gpt': [0, 1, 2, 3, 4], 'gemini': [5, 6, 7, 8], 'claude': [9, 10, 11, 12, 13], 'glm': [15, 16]}, 'required_assets': {'require_pool_parquet': True, 'required_base_models': [{'name': 'qwen3_14b', 'family': 'qwen3', 'hidden_size': 5120, 'num_layers': 40}, {'name': 'qwen25_14b', 'family': 'qwen2', 'hidden_size': 5120, 'num_layers': 48}, {'name': 'qwen3_8b', 'family': 'qwen3', 'hidden_size': 4096, 'num_layers': 36}, {'name': 'deberta_v3_large', 'family': 'deberta', 'hidden_size': 1024, 'num_layers': 24}], 'required_checkpoints': ['qwen3_14b_p2', 'qwen25_14b_p2', 'deberta_p2', 'qwen3_8b_claude'], 'optional_checkpoints': ['qwen3_14b_v5', 'qwen3_14b_inhouse', 'qwen25_14b_inhouse', 'deberta_inhouse', 'qwen3_8b_v5', 'llama3_8b_v5', 'gemma2_9b_ts', 'llama31_8b_ts', 'mistral_7b_ts', 'bge_m3']}, 'model_weights': {'svc_4view': {'plain_weight': 0.3, 'twin_weight': 0.0}, 'qwen3_14b_v5': {'plain_weight': 0.48, 'twin_weight': 0.16}, 'qwen3_14b_p2': {'plain_weight': 0.42, 'twin_weight': 0.14}, 'qwen3_14b_inhouse': {'plain_weight': 0.4, 'twin_weight': 0.14}, 'qwen3_8b_v5': {'plain_weight': 0.32, 'twin_weight': 0.11}, 'qwen25_14b_p2': {'plain_weight': 0.22, 'twin_weight': 0.08}, 'qwen25_14b_inhouse': {'plain_weight': 0.22, 'twin_weight': 0.08}, 'deberta_p2': {'plain_weight': 0.22, 'twin_weight': 0.06}, 'llama3_8b_v5': {'plain_weight': 0.2, 'twin_weight': 0.07}, 'gemma2_9b_ts': {'plain_weight': 0.32, 'twin_weight': 0.1}, 'llama31_8b_ts': {'plain_weight': 0.3, 'twin_weight': 0.1}, 'mistral_7b_ts': {'plain_weight': 0.28, 'twin_weight': 0.08}, 'bge_m3': {'plain_weight': 0.2, 'twin_weight': 0.0}, 'qwen3_8b_claude': {'plain_weight': 0.0, 'twin_weight': 0.0}, 'lexical_5v': {'plain_weight': 0.1, 'twin_weight': 0.0}, 'wpca_dp': {'plain_weight': 0.25, 'twin_weight': 0.0}, 'spectral_bank': {'plain_weight': 0.1, 'twin_weight': 0.0}}, 'main_stack': {'linear_svc_weight': 1.0, 'members': {'lora_qwen3_14b': {'plain_weight': 0.4, 'twin_weight': 0.1, 'twin_clustering': 'groups_085'}, 'lora_qwen25_14b': {'plain_weight': 0.2, 'twin_weight': 0.1, 'twin_clustering': 'groups_085'}, 'deberta_v3_large': {'plain_weight': 0.25, 'twin_weight': 0.0}}}, 'complementary_stack': {'stream_weights': {'lora_and_twins': 0.35, 'lexical': 0.2, 'dp_posteriors': 0.3, 'spectral_bank': 0.15}, 'lora_members': {'lora_qwen3_14b': {'plain_weight': 0.3, 'twin_weight': 0.1}, 'lora14b_v1': {'plain_weight': 0.14, 'twin_weight': 0.05}, 'qwen3_14b_ts': {'plain_weight': 0.2, 'twin_weight': 0.08}, 'lora_qwen25_14b': {'plain_weight': 0.2, 'twin_weight': 0.08}, 'instyle': {'plain_weight': 0.22, 'twin_weight': 0.08}, 'qwen3_8b_ts': {'plain_weight': 0.2, 'twin_weight': 0.08}, 'ntl_claude_head': {'plain_weight': 0.14, 'twin_weight': 0.05}, 'deberta_v3_large': {'plain_weight': 0.18, 'twin_weight': 0.06}}, 'twin_resolutions': {'t85': 0.5, 't72': 0.3, 't60': 0.2}, 'dp_weights': {'q3_14b': 0.38, 'q25_14b': 0.24, 'deberta': 0.2, 'instyle': 0.1, 'part1_14b': 0.08, 'q3_14b_ts': 0.1, 'q3_8b_ts': 0.1, 'claude_8b': 0.04}, 'se_floors': {'qwen3_14b_p2': 0.1, 'qwen25_14b_p2': 0.1, 'deberta_p2': 0.1, 'qwen3_14b_p1': 0.35, 'qwen3_14b_ts': 0.25, 'instyle': 0.35, 'qwen3_8b_ts': 0.35, 'qwen3_8b_claude': 0.35}, 'prior_scales': {'qwen3_14b_p2': 1.0, 'qwen25_14b_p2': 0.85, 'deberta_p2': 0.8, 'qwen3_14b_p1': 0.5, 'qwen3_14b_ts': 0.5, 'instyle': 0.45, 'qwen3_8b_ts': 0.4, 'qwen3_8b_claude': 0.3}, 'spectral_bank': {'plain_C': 0.04, 'center_C': 0.03, 'center_blend': 0.35}, 'extract_train_wpca': False}, 'fusion': {'main_ratio': 0.85, 'comp_ratio': 0.15}, 'postprocess': {'claude_specialist_weight': 0.15, 'curly_quote_multiplier': 0.36787944117144233, 'multi_em_dash_log_penalty': 0.0, 'straight_contr_log_penalty': 0.0, 'curly_contr_log_penalty': 0.0, 'continuous_hw_weight': 0.0, 'sinkhorn_iterations': 100, 'sinkhorn_strength': 0.65, 'sinkhorn_target_mix': 0.35, 'enable_typobayes': True, 'typobayes_kappa': 24.0, 'typobayes_evidence_scale': 1.2, 'typobayes_evidence_cap': 0.3, 'max_cluster_size': 14}}

"""
Canonical Configuration, Constants & Path Resolution
====================================================
Master canonical label ordering, provider families, simplex helpers,
and Kaggle filesystem asset discovery.
"""

import json
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
import numpy as np

# Canonical 18-Class Labels (Strict order from NTL two-stage LoRA part two)
LABELS = [
    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra",
    "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash",
    "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5", "claude-fable-5",
    "qwen3.8-max", "glm-5.3", "glm-5.3-flash", "kimi-k3"
]
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}

FAMILIES = {
    "gpt": [0, 1, 2, 3, 4],
    "gemini": [5, 6, 7, 8],
    "claude": [9, 10, 11, 12, 13],
    "glm": [15, 16],
}

HARD_CLASS_INDICES = [1, 2, 3, 6, 7, 8, 13, 14, 15, 16, 17]

SPECIALIST_GROUPS = {
    "gpt": [0, 1, 2, 3, 4],
    "gpt_56_triad": [1, 2, 3],
    "gemini": [5, 6, 7, 8],
    "gemini_flash_triad": [6, 7, 8],
    "gemini_37_38_pair": [7, 8],
    "claude": [9, 10, 11, 12, 13],
    "glm": [15, 16],
    "glm_kimi_triad": [15, 16, 17],
    "frontier_and_fable": [10, 13, 14, 15, 16, 17],
    "kimi_gpt55_qwen": [0, 14, 17],
    "claude_glm": [9, 10, 11, 12, 13, 15, 16],
    "gpt_qwen_kimi": [0, 1, 2, 3, 4, 14, 17],
}

PRIOR_SCALES = {
    "qwen3_14b_p2": 1.00,
    "qwen3_14b_inhouse": 1.00,
    "qwen25_14b_p2": 0.85,
    "qwen25_14b_inhouse": 0.85,
    "deberta_p2": 0.80,
    "deberta_inhouse": 0.80,
    "deberta_gemini": 0.40,
    "deberta_glm": 0.35,
    "deberta_gpt": 0.35,
    "deberta_claude": 0.35,
    "deberta_claude_glm": 0.35,
    "deberta_gpt_qwen_kimi": 0.35,
    "qwen3_14b_p1": 0.50,
    "instyle": 0.45,
    "qwen3_8b_claude": 0.30,
    "qwen3_8b_inhouse": 0.30,
    "gemma2_9b_ts": 0.40,
    "llama31_8b_ts": 0.40,
    "mistral_7b_ts": 0.40,
    "bge_m3": 0.60,
}

def align_cols(arr: np.ndarray, src_labels: list) -> np.ndarray:
    """Aligns columns of array to canonical 18-class order."""
    src_list = [x.decode('utf-8') if isinstance(x, (bytes, np.bytes_)) else str(x) for x in src_labels]
    idx = [src_list.index(l) for l in LABELS]
    if arr.ndim == 1:
        return arr[idx]
    elif arr.ndim == 2:
        if arr.shape[0] == len(src_list) and arr.shape[1] != len(src_list):
            return arr[idx, :]
        elif arr.shape[1] == len(src_list):
            return arr[:, idx]
    return arr[..., idx]

def log_softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable log-softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    return x - c - np.log(np.sum(np.exp(x - c), axis=axis, keepdims=True) + 1e-15)

def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax."""
    c = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - c)
    return e / (np.sum(e, axis=axis, keepdims=True) + 1e-15)

def resolve_dataset_paths(input_dir: Optional[Path] = None) -> Tuple[Path, Path, Optional[Path]]:
    """Resolves train.csv, test.csv, and pool_v4.parquet across local and Kaggle environments."""
    if input_dir is None:
        input_dir = Path('/kaggle/input')
        if not input_dir.exists():
            input_dir = Path('.')

    candidates = [
        input_dir / 'competitions' / 'name-that-llm',
        input_dir / 'name-that-llm',
        Path('local_data'),
        Path('.')
    ]
    train_path = None
    test_path = None
    for c in candidates:
        if (c / 'train.csv').exists() and train_path is None:
            train_path = c / 'train.csv'
        if (c / 'test.csv').exists() and test_path is None:
            test_path = c / 'test.csv'

    if train_path is None:
        train_path = Path('local_data') / 'train.csv'
    if test_path is None:
        test_path = Path('local_data') / 'test.csv'

    pool_files = resolve_all_pool_files(input_dir)
    pool_file = pool_files[0] if pool_files else None

    return train_path, test_path, pool_file

def resolve_all_pool_files(input_dir: Optional[Path] = None) -> List[Path]:
    """Discovers all pretraining/unlabeled pool parquet files in priority order."""
    if input_dir is None:
        input_dir = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')
    pool_files: List[Path] = []
    pool_patterns = ['clean_master_pool.parquet', 'ntl_clean_master_pool.parquet', 'nlm_pool_v5.parquet', 'pool_v4.parquet', 'nlm_matched_clean.parquet', 'nlm_pool_v2.parquet']
    for pat in pool_patterns:
        for p in input_dir.rglob(pat):
            if p not in pool_files:
                pool_files.append(p)
    return pool_files

def find_base_model(input_dir: Path, model_family: str, hidden_sz: int, num_layers: int) -> Optional[Path]:
    """Discovers base model directories from Hugging Face config.json inspection."""
    candidates = []
    for cfg_p in input_dir.rglob('config.json'):
        try:
            cfg = json.load(open(cfg_p))
            m_type = str(cfg.get('model_type', '')).lower()
            h_dim = cfg.get('hidden_size', 0)
            n_lay = cfg.get('num_hidden_layers', 0)
            if model_family.lower() in m_type and h_dim == hidden_sz and n_lay == num_layers:
                p_str = str(cfg_p.parent).lower().replace('\\', '/')
                if model_family.lower() in p_str:
                    return cfg_p.parent
                candidates.append(cfg_p.parent)
        except Exception:
            continue
    return candidates[0] if candidates else None

class CheckpointInfo(tuple):
    """Tuple subclass supporting both 2-element (dim, keep) and 3-element (dim, keep, backbone) unpacking."""
    def __iter__(self):
        try:
            import sys
            import dis
            f = sys._getframe(1)
            for i in dis.get_instructions(f.f_code):
                if i.offset == f.f_lasti and i.opname.startswith('UNPACK_SEQUENCE'):
                    if i.argval == 2:
                        return iter(self[:2])
                    elif i.argval == 3:
                        return iter(self[:3])
        except Exception:
            pass
        return super().__iter__()

def _inspect_checkpoint_dim_and_keep(ckpt_dir: Path) -> Tuple[int, int]:
    """Returns (head_feature_dim, keep_layers) for checkpoint directory without full model loading."""
    dim = 0
    keep = 0
    backbone = ""
    meta_p = ckpt_dir / "meta.json"
    if meta_p.exists():
        try:
            m = json.load(open(meta_p))
            cfg = m.get("cfg", m)
            keep = int(cfg.get("keep", 0))
            backbone = str(m.get("backbone", cfg.get("backbone", ""))).lower()
        except Exception:
            pass

    adapter_p = ckpt_dir / "adapter_config.json"
    if adapter_p.exists():
        try:
            ac = json.load(open(adapter_p))
            base_model = str(ac.get("base_model_name_or_path", "")).lower()
            if not backbone and base_model:
                backbone = base_model
        except Exception:
            pass

    heads_p = ckpt_dir / "heads.safetensors"
    if heads_p.exists():
        try:
            from safetensors import safe_open
            with safe_open(str(heads_p), framework="pt", device="cpu") as f:
                for k in f.keys():
                    shape = f.get_tensor(k).shape
                    if len(shape) == 2 and (shape[0] == 18 or shape[1] == 18):
                        dim = int(shape[1] if shape[0] == 18 else shape[0])
                        break
        except Exception:
            pass
    elif (ckpt_dir / "heads.pt").exists():
        try:
            import torch
            sd = torch.load(ckpt_dir / "heads.pt", map_location="cpu", weights_only=False)
            if isinstance(sd, dict):
                for k, v in sd.items():
                    if hasattr(v, "shape") and len(v.shape) == 2 and (v.shape[0] == 18 or v.shape[1] == 18):
                        dim = int(v.shape[1] if v.shape[0] == 18 else v.shape[0])
                        break
        except Exception:
            pass
    return CheckpointInfo((dim, keep, backbone))

def find_checkpoints(input_dir: Path) -> Dict[str, Path]:
    """Finds all fine-tuned LoRA and head checkpoints across safetensors and PyTorch formats."""
    ckpts = {}

    # 1. Check for heads.safetensors and heads.pt files
    head_candidates = list(input_dir.rglob('heads.safetensors')) + list(input_dir.rglob('heads.pt'))
    for p in head_candidates:
        p_str = str(p).lower().replace('\\', '/')
        is_fold = ('_f0' in p_str or '_f1' in p_str or '_f2' in p_str or '_f3' in p_str or '_f4' in p_str)

        if 'lora_qwen3_14b' in p_str or 'qwen3_14b_part2' in p_str or ('14b' in p_str and 'part2' in p_str and 'qwen3' in p_str):
            ckpts['qwen3_14b_p2'] = p.parent
        elif 'lora_qwen25_14b' in p_str or 'qwen25_14b_part2' in p_str or ('14b' in p_str and 'part2' in p_str and 'qwen2' in p_str):
            ckpts['qwen25_14b_p2'] = p.parent
        elif 'q14v5' in p_str:
            if 'qwen3_14b_v5' not in ckpts or (not is_fold):
                ckpts['qwen3_14b_v5'] = p.parent
        elif 'ts_q3b14' in p_str:
            if 'qwen3_14b_ts' not in ckpts or (not is_fold):
                ckpts['qwen3_14b_ts'] = p.parent
        elif 'lora14b_v1' in p_str:
            ckpts['qwen3_14b_p1'] = p.parent
        elif any(token in p_str for token in ('inhouse', 'ntl-inhouse', 'export_lora24', 'export_champion', 'lora-peft-training', 'peft_training')):
            if "family_gate" in p_str or "/gate" in p_str or "\\gate" in p_str:
                continue
            dim, keep, backbone = _inspect_checkpoint_dim_and_keep(p.parent)
            if '2.5' in backbone or 'qwen25' in backbone or (dim == 40960 and (keep >= 30 or '25' in p_str or 'qwen25' in p_str)):
                ckpts['qwen25_14b_inhouse'] = p.parent
            elif dim == 40960 or '14b' in p_str or '14b' in backbone:
                ckpts['qwen3_14b_inhouse'] = p.parent
            elif dim == 8192 or 'deberta' in p_str or 'deberta' in backbone:
                ckpts['deberta_inhouse'] = p.parent
            elif dim in (32768, 28672) or ('8b' in p_str or '8b' in backbone) or ('14b' not in p_str and '14b' not in backbone and dim != 40960):
                ckpts['qwen3_8b_inhouse'] = p.parent
        elif 'q8v5' in p_str:
            if 'qwen3_8b_v5' not in ckpts or (not is_fold):
                ckpts['qwen3_8b_v5'] = p.parent
        elif 'ts_q3b8' in p_str or 'ts8_s7' in p_str or 'l24x_q3b8' in p_str:
            if 'qwen3_8b_ts' not in ckpts or (not is_fold):
                ckpts['qwen3_8b_ts'] = p.parent
        elif 'll8v5' in p_str:
            if 'llama3_8b_v5' not in ckpts or (not is_fold):
                ckpts['llama3_8b_v5'] = p.parent
        elif 'ts_llama8' in p_str:
            if 'llama3_8b_ts' not in ckpts or (not is_fold):
                ckpts['llama3_8b_ts'] = p.parent
        elif 'lora8b_v1' in p_str:
            ckpts['qwen3_8b_p1'] = p.parent
        elif 'ts_q25_7b' in p_str or 'l24x_q25_7b' in p_str:
            if 'qwen25_7b_ts' not in ckpts or (not is_fold):
                ckpts['qwen25_7b_ts'] = p.parent
        elif any(t in p_str for t in ('gemma2', 'gemma_2', 'gemma-2', 'ts_gemma', 'ts-gemma')):
            if 'gemma2_9b_ts' not in ckpts or (not is_fold):
                ckpts['gemma2_9b_ts'] = p.parent
        elif any(t in p_str for t in ('llama31', 'llama_31', 'llama-3.1', 'ts_llama31', 'ts-llama31')):
            if 'llama31_8b_ts' not in ckpts or (not is_fold):
                ckpts['llama31_8b_ts'] = p.parent
        elif any(t in p_str for t in ('mistral', 'ts_mistral', 'ts-mistral')):
            if 'mistral_7b_ts' not in ckpts or (not is_fold):
                ckpts['mistral_7b_ts'] = p.parent
        elif any(t in p_str for t in ('bge_m3', 'bge-m3', 'bgem3')):
            if 'bge_m3' not in ckpts or (not is_fold):
                ckpts['bge_m3'] = p.parent
        elif 'instyle' in p_str:
            ckpts['instyle'] = p.parent

    # 2. Check for Claude specialist head
    for p in input_dir.rglob('claude_head.safetensors'):
        ckpts['qwen3_8b_claude'] = p.parent
        break
    if 'qwen3_8b_claude' not in ckpts:
        for p in input_dir.rglob('readouts.npz'):
            if 'claude' in str(p).lower():
                ckpts['qwen3_8b_claude'] = p.parent
                break

    # 3. Check for DeBERTa sequence classification head
    for p in list(input_dir.rglob('head.safetensors')) + list(input_dir.rglob('heads.safetensors')):
        if 'deberta' in str(p).lower():
            ckpts['deberta_p2'] = p.parent
            break

    # 4. Fallback search for instyle
    if 'instyle' not in ckpts:
        for p in input_dir.rglob('*'):
            p_s = str(p).lower().replace('\\', '/')
            if 'instyle' in p_s and ((p / 'adapter_model.safetensors').exists() or (p / 'adapter' / 'adapter_model.safetensors').exists() or (p / 'heads.safetensors').exists() or (p / 'heads.pt').exists()):
                ckpts['instyle'] = p
                break

    # 5. Fallback search for q14v5 / ts_q3b14 adapter
    if 'qwen3_14b_ts' not in ckpts:
        for p in input_dir.rglob('adapter_model.safetensors'):
            p_s = str(p).lower().replace('\\', '/')
            if 'q14v5' in p_s or 'ts_q3b14' in p_s:
                ckpts['qwen3_14b_ts'] = p.parent
                break

    # 6. Fallback search for bge_m3 base or fine-tuned directory
    if 'bge_m3' not in ckpts:
        for p in input_dir.rglob('*'):
            p_s = str(p).lower().replace('\\', '/')
            if ('bge-m3' in p_s or 'bge_m3' in p_s or 'bgem3' in p_s) and ((p / 'config.json').exists() or (p / 'model.safetensors').exists() or (p / 'pytorch_model.bin').exists()):
                ckpts['bge_m3'] = p
                break

    return ckpts

def find_fold_adapters(input_dir: Optional[Path], prefix: str) -> List[Path]:
    """Scans input_dir for all adapter/head directories matching prefix, prefix_f*, prefix_fold*, prefix-f*."""
    if input_dir is None:
        input_dir = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('.')

    matches: Set[Path] = set()
    prefix_lower = prefix.lower()

    candidate_files = (
        list(input_dir.rglob('heads.safetensors')) +
        list(input_dir.rglob('heads.pt')) +
        list(input_dir.rglob('adapter_model.safetensors'))
    )
    for f in candidate_files:
        p = f.parent
        p_name = p.name.lower()
        if p_name in ('adapter', 'checkpoint'):
            p = p.parent
            p_name = p.name.lower()

        p_str = str(p).lower().replace('\\', '/')
        if prefix_lower in p_name or prefix_lower in p_str:
            # Check pattern: exact match or prefix followed by _ or - or fold
            tokens = p_name.replace('-', '_').split('_')
            matched = False
            for t in tokens:
                if t == prefix_lower or t.startswith(prefix_lower):
                    matched = True
                    break
            if not matched and prefix_lower in p_str:
                matched = True

            if matched:
                matches.add(p)

    return sorted(list(matches), key=lambda x: str(x).lower())

def validate_production_assets(
    input_dir: Path,
    train_path: Path,
    test_path: Path,
    pool_files: List[Path],
    ckpts: Dict[str, Path],
    cfg: Optional[Dict[str, Any]] = None,
) -> Dict[str, Path]:
    """
    Fail-closed pre-flight validation for production inference runs (when NTL_SMOKE != '1').
    Raises FileNotFoundError immediately at startup if any required dataset, pool parquet,
    base model trunk, or checkpoint is missing.
    Returns a dict of verified base_model paths keyed by name.
    """
    if cfg is None:
        cfg = {}
    req = cfg.get("required_assets", {})
    missing: List[str] = []

    # 1. Verify competition CSVs
    if not train_path or not Path(train_path).exists():
        missing.append(f"Competition train.csv not found at: {train_path}")
    if not test_path or not Path(test_path).exists():
        missing.append(f"Competition test.csv not found at: {test_path}")

    # 2. Verify public pool parquet(s)
    require_pool = req.get("require_pool_parquet", True)
    valid_pools = [p for p in pool_files if p is not None and Path(p).exists()]
    if require_pool and len(valid_pools) == 0:
        missing.append(f"No public pool parquet (pool_v4.parquet / nlm_pool_v5.parquet) found under {input_dir}")

    # 3. Verify all required base model trunks
    default_bases = [
        {"name": "qwen3_14b", "family": "qwen3", "hidden_size": 5120, "num_layers": 40},
        {"name": "qwen25_14b", "family": "qwen2", "hidden_size": 5120, "num_layers": 48},
        {"name": "qwen3_8b", "family": "qwen3", "hidden_size": 4096, "num_layers": 36},
        {"name": "deberta_v3_large", "family": "deberta", "hidden_size": 1024, "num_layers": 24},
    ]
    base_specs = req.get("required_base_models", default_bases)
    verified_bases: Dict[str, Path] = {}
    for b_spec in base_specs:
        b_path = find_base_model(
            input_dir,
            b_spec["family"],
            int(b_spec["hidden_size"]),
            int(b_spec["num_layers"]),
        )
        if b_path is None or not Path(b_path).exists():
            missing.append(
                f"Required base model '{b_spec['name']}' (family={b_spec['family']}, "
                f"hidden={b_spec['hidden_size']}, layers={b_spec['num_layers']}) not found under {input_dir}"
            )
        else:
            verified_bases[b_spec["name"]] = b_path

    # 4. Verify all required fine-tuned checkpoints
    default_req_ckpts = [
        "qwen3_14b_p2", "qwen25_14b_p2",
        "deberta_p2", "qwen3_8b_claude"
    ]
    req_ckpts = req.get("required_checkpoints", default_req_ckpts)
    for ck_name in req_ckpts:
        if ck_name not in ckpts or not Path(ckpts[ck_name]).exists():
            missing.append(f"Required checkpoint '{ck_name}' not found under {input_dir} (discovered: {list(ckpts.keys())})")

    if missing:
        error_msg = (
            "[PRE-FLIGHT ASSET AUDIT FAILED] Missing required inference assets:\n  - "
            + "\n  - ".join(missing)
        )
        raise FileNotFoundError(error_msg)

    print(
        f"[Pre-Flight Audit PASSED] Verified {len(valid_pools)} pool file(s), "
        f"{len(verified_bases)} base trunks, and {len(ckpts)} checkpoints: {list(ckpts.keys())}"
    )
    return verified_bases


In [ ]:
# === MODULE: src/namethatllm/wpca_gaussian.py ===
"""
Full-Rank wPCA & ExactAuthorPosterior (2^18 Bitmask DP)
======================================================
Two-way variance decomposition, Ledoit-Wolf shrinkage floors,
crash-proof cluster splitting (max_size <= 14), and exact forward-backward
bitmask DP author posteriors under the unique-authorship constraint.
"""

from functools import lru_cache
from typing import Tuple, List, Dict, Optional, Any
import numpy as np
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.special import logsumexp
from sklearn.covariance import ledoit_wolf_shrinkage


def _two_way(Z: np.ndarray, y: np.ndarray, pinv: np.ndarray, G: int, K_classes: int = 18, iters: int = 4) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Iterative two-way ANOVA decomposition separating author means and prompt random effects."""
    N, d = Z.shape
    mu = np.zeros((K_classes, d), dtype=np.float64)
    P = np.zeros((G, d), dtype=np.float64)
    ng = np.bincount(pinv, minlength=G).clip(min=1)
    for _ in range(iters):
        R = Z - P[pinv]
        for k in range(K_classes):
            idx = (y == k)
            if idx.any():
                mu[k] = R[idx].mean(axis=0)
        R = Z - mu[y]
        P[:] = 0.0
        np.add.at(P, pinv, R)
        P /= ng[:, None]
    return mu, P, ng

def _shrunk(S: np.ndarray, alpha: float) -> np.ndarray:
    """Shrinks covariance matrix S towards spherical target."""
    d = S.shape[0]
    tr = np.trace(S) / d
    return (1.0 - alpha) * S + alpha * tr * np.eye(d)

def fit_shared_gaussian_lowram(X_tr_f32: np.ndarray, y: np.ndarray, prompts: np.ndarray,
                               d_target: int = 256, se_floor: float = 0.10) -> dict:
    """
    Fits shared Gaussian whitener with within-prompt centering and Ledoit-Wolf shrinkage floor.
    Returns unsupervised top-64 within-prompt principal components V_top64 (zero label leakage!).
    """
    N, D = X_tr_f32.shape
    pu, pinv = np.unique(prompts, return_inverse=True)
    G = len(pu)

    mx = X_tr_f32.mean(axis=0, dtype=np.float64).astype(np.float32)
    sx = X_tr_f32.std(axis=0, dtype=np.float64).astype(np.float32)
    sx = np.where(sx > 1e-6, sx, 1.0).astype(np.float32)

    # Standardize and subtract prompt means in-place (Within-Prompt Whitened PCA!)
    R = (X_tr_f32 - mx) / sx
    Pm = np.zeros((G, D), dtype=np.float32)
    np.add.at(Pm, pinv, R)
    Pm /= np.bincount(pinv, minlength=G)[:, None].clip(min=1).astype(np.float32)
    R -= Pm[pinv]
    del Pm

    d = int(min(d_target, max(N - G - 1, 1), D))
    Gram = (R @ R.T).astype(np.float64)
    lam_r, U = np.linalg.eigh(Gram)
    idx_top = np.argsort(lam_r)[::-1][:d]
    lam_top = np.clip(lam_r[idx_top], 1e-10, None)
    U_f32 = U[:, idx_top].astype(np.float32)
    V_f32 = (R.T @ U_f32) / np.sqrt(lam_top).astype(np.float32)[None, :]
    del R, Gram, U, U_f32

    # Unsupervised within-prompt top-64 principal components (zero label leakage)
    V_top64 = V_f32[:, :min(64, d)].copy()

    Z = (((X_tr_f32 - mx) / sx) @ V_f32).astype(np.float64)

    mu, P, ng = _two_way(Z, y, pinv, G, K_classes=K, iters=4)
    E = Z - mu[y] - P[pinv]
    dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = (E.T @ E) / dof
    ae = max(float(ledoit_wolf_shrinkage(E, assume_centered=True)), se_floor)
    Se = _shrunk(Se_raw, ae)

    Pc = P - P.mean(0)
    Sp_raw = (Pc.T @ Pc) / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
    ap = float(ledoit_wolf_shrinkage(Pc, assume_centered=True))
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se))
    Sp = (1.0 - ap) * Sp_raw + ap * (tr / d) * np.eye(d)

    L = np.linalg.cholesky(Se)
    Li = np.linalg.inv(L)
    M = 0.5 * ((Li @ Sp @ Li.T) + (Li @ Sp @ Li.T).T)
    lam, Q = np.linalg.eigh(M)
    lam = np.clip(lam, 1e-4, None)
    A = Q.T @ Li
    W_f32 = (V_f32 @ A.T.astype(np.float32)).astype(np.float32)
    m_whitened = mu @ A.T

    return {
        "mx": mx, "sx": sx, "W": W_f32, "m": m_whitened, "lam": lam,
        "V_top64": V_top64, "T1": 2.40, "Tg": 2.35, "ae": ae
    }

def transform_lowram(X_f32: np.ndarray, prm: dict, batch_rows: int = 1000) -> np.ndarray:
    """Projects representations into whitened space."""
    N = len(X_f32)
    d = prm["W"].shape[1]
    U_out = np.empty((N, d), dtype=np.float64)
    mx, sx, W = prm["mx"], prm["sx"], prm["W"]
    for s in range(0, N, batch_rows):
        chunk = ((X_f32[s : s + batch_rows] - mx) / sx) @ W
        U_out[s : s + batch_rows] = chunk.astype(np.float64)
    return U_out

def split_oversized_clusters_dist(D: np.ndarray, groups: np.ndarray, max_cluster_size: int = 14) -> np.ndarray:
    """Recursively splits clusters with size > max_cluster_size to prevent bitmask DP overflow."""
    if len(groups) < 2:
        return groups.copy()
    new_groups = np.zeros(len(groups), dtype=np.int64)
    next_id = 1
    for g in np.unique(groups):
        rows = np.flatnonzero(groups == g)
        if len(rows) <= max_cluster_size:
            new_groups[rows] = next_id
            next_id += 1
        else:
            n_sub = int(np.ceil(len(rows) / max_cluster_size))
            D_sub = D[np.ix_(rows, rows)].copy()
            np.fill_diagonal(D_sub, 0.0)
            D_sub = np.clip(0.5 * (D_sub + D_sub.T), 0.0, None)
            d_condensed = squareform(D_sub, checks=False)
            sub_labels = fcluster(linkage(d_condensed, method='average'), t=n_sub, criterion='maxclust')
            for sg in np.unique(sub_labels):
                s_rows = rows[sub_labels == sg]
                for start in range(0, len(s_rows), max_cluster_size):
                    new_groups[s_rows[start : start + max_cluster_size]] = next_id
                    next_id += 1
    return new_groups

@lru_cache(maxsize=4)
def _bitmask_levels(k: int = 18):
    """Precomputes bitmasks partitioned by population count (number of set bits)."""
    masks = np.arange(1 << k, dtype=np.int64)
    counts = np.fromiter((int(x).bit_count() for x in masks), dtype=np.uint8, count=len(masks))
    return tuple(masks[counts == n] for n in range(k + 1))

class ExactAuthorPosterior:
    """Exact shared-prompt Gaussian posteriors under author uniqueness via 2^18 bitmask DP."""
    def __init__(self, m: np.ndarray, lam: np.ndarray):
        self.m = np.asarray(m, dtype=np.float64)
        self.lam = np.asarray(lam, dtype=np.float64)
        self.K, self.d = self.m.shape
        self.levels = _bitmask_levels(self.K)
        self._subset_means = {}

    def single(self, u: np.ndarray, temperature: float = 2.40) -> np.ndarray:
        diff = u[None, :] - self.m
        dist = np.sum((diff * diff) / (1.0 + self.lam)[None, :], axis=1)
        log_p = -0.5 * dist / max(float(temperature), 1e-4)
        return log_p - logsumexp(log_p)

    def group(self, U: np.ndarray, temperature: float = 2.35) -> np.ndarray:
        U = np.asarray(U, dtype=np.float64)
        n = len(U)
        if n <= 1 or n > self.K:
            return np.vstack([self.single(U[i], temperature=temperature) for i in range(n)])

        if n not in self._subset_means:
            if len(self._subset_means) >= 3:
                self._subset_means.clear()
            masks = self.levels[n]
            membership = ((masks[:, None] >> np.arange(self.K)) & 1).astype(np.float32)
            self._subset_means[n] = membership @ self.m.astype(np.float32)

        terminal = self.levels[n]
        gamma = self.lam / (1.0 + n * self.lam)
        residual = U.sum(axis=0, keepdims=True) - self._subset_means[n].astype(np.float64)
        correction = 0.5 * np.einsum("nd,d,nd->n", residual, gamma, residual) / max(float(temperature), 1e-4)
        correction -= correction.max()

        # Passage-specific unary log-likelihoods A[i, author]
        A = -0.5 * np.sum((U[:, None, :] - self.m[None, :, :]) ** 2, axis=2) / max(float(temperature), 1e-4)
        A -= A.max(axis=1, keepdims=True)

        forward = np.full(1 << self.K, -np.inf, dtype=np.float64)
        forward[0] = 0.0
        for count in range(1, n + 1):
            masks = self.levels[count]
            for author in range(self.K):
                bit = 1 << author
                chosen = masks[(masks & bit) != 0]
                forward[chosen] = np.logaddexp(forward[chosen], forward[chosen ^ bit] + A[count - 1, author])

        backward = np.full_like(forward, -np.inf)
        backward[terminal] = correction
        for count in range(n - 1, -1, -1):
            masks = self.levels[count]
            for author in range(self.K):
                bit = 1 << author
                chosen = masks[(masks & bit) == 0]
                backward[chosen] = np.logaddexp(backward[chosen], A[count, author] + backward[chosen | bit])

        result = np.empty((n, self.K), dtype=np.float64)
        for row in range(n):
            masks = self.levels[row]
            for author in range(self.K):
                bit = 1 << author
                chosen = masks[(masks & bit) == 0]
                result[row, author] = logsumexp(forward[chosen] + A[row, author] + backward[chosen | bit]) - backward[0]

        result -= logsumexp(result, axis=1, keepdims=True)
        return result

def scores_exact_dp(U_test: np.ndarray, groups: np.ndarray, prm: dict, D_dist: np.ndarray) -> np.ndarray:
    """Wraps ExactAuthorPosterior with oversized cluster splitting guardrail."""
    N = len(U_test)
    if N == 0:
        return np.zeros((0, K), dtype=np.float64)
    if N == 1:
        solver = ExactAuthorPosterior(prm["m"], prm["lam"])
        return solver.single(U_test[0], temperature=prm.get("T1", 2.40))[None, :]

    safe_groups = split_oversized_clusters_dist(D_dist, groups, max_cluster_size=14)
    solver = ExactAuthorPosterior(prm["m"], prm["lam"])
    T1 = prm.get("T1", 2.40)
    Tg = prm.get("Tg", 2.35)

    lp_dp = np.zeros((N, K), dtype=np.float64)
    for g in np.unique(safe_groups):
        idx_g = np.where(safe_groups == g)[0]
        if len(idx_g) == 1:
            lp_dp[idx_g] = solver.single(U_test[idx_g[0]], temperature=T1)
        else:
            lp_dp[idx_g] = solver.group(U_test[idx_g], temperature=Tg)
    return lp_dp


def save_wpca_params(prm: dict, path: Any):
    """Serializes fitted wPCA Gaussian whitener parameters to safetensors."""
    from pathlib import Path
    import torch
    from safetensors.torch import save_file

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sd = {
        "mx": torch.from_numpy(np.asarray(prm["mx"], dtype=np.float32)),
        "sx": torch.from_numpy(np.asarray(prm["sx"], dtype=np.float32)),
        "W": torch.from_numpy(np.asarray(prm["W"], dtype=np.float32)),
        "m": torch.from_numpy(np.asarray(prm["m"], dtype=np.float32)),
        "lam": torch.from_numpy(np.asarray(prm["lam"], dtype=np.float32)),
        "V_top64": torch.from_numpy(np.asarray(prm["V_top64"], dtype=np.float32)),
        "scalars": torch.tensor([float(prm.get("T1", 2.40)), float(prm.get("Tg", 2.35)), float(prm.get("ae", 0.10))], dtype=torch.float32),
    }
    save_file(sd, str(path))


def load_wpca_params(path: Any) -> dict:
    """Loads fitted wPCA parameters from safetensors."""
    from pathlib import Path
    from safetensors.torch import load_file

    sd = load_file(str(path))
    sc = sd["scalars"].cpu().numpy()
    return {
        "mx": sd["mx"].cpu().numpy(),
        "sx": sd["sx"].cpu().numpy(),
        "W": sd["W"].cpu().numpy(),
        "m": sd["m"].cpu().numpy().astype(np.float64),
        "lam": sd["lam"].cpu().numpy().astype(np.float64),
        "V_top64": sd["V_top64"].cpu().numpy(),
        "T1": float(sc[0]),
        "Tg": float(sc[1]),
        "ae": float(sc[2]),
    }


def train_and_save_wpca(
    Z_train: np.ndarray,
    y_train: np.ndarray,
    prompts_train: np.ndarray,
    output_path: Any,
    d_target: int = 256,
    se_floor: float = 0.10,
) -> dict:
    """
    Fits within-prompt whitened PCA (wPCA) on pre-extracted feature matrix and saves to safetensors.
    Zero GPU requirements; executes on CPU in seconds.
    """
    print(f"[wPCA Trainer] Fitting wPCA on shape {Z_train.shape} with {len(np.unique(prompts_train))} prompt groups...")
    prm = fit_shared_gaussian_lowram(
        X_tr_f32=np.asarray(Z_train, dtype=np.float32),
        y=np.asarray(y_train, dtype=int),
        prompts=np.asarray(prompts_train),
        d_target=d_target,
        se_floor=se_floor,
    )
    save_wpca_params(prm, output_path)
    print(f"[wPCA Trainer] Saved fitted wPCA parameters to {output_path}")
    return prm


def save_wpca_bundle(bundle: Dict[str, dict], path: Any):
    """Serializes a dictionary of {model_key: wpca_params} to a single safetensors file."""
    from pathlib import Path
    import torch
    from safetensors.torch import save_file

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sd = {}
    for model_key, prm in bundle.items():
        sd[f"{model_key}__mx"] = torch.from_numpy(np.asarray(prm["mx"], dtype=np.float32))
        sd[f"{model_key}__sx"] = torch.from_numpy(np.asarray(prm["sx"], dtype=np.float32))
        sd[f"{model_key}__W"] = torch.from_numpy(np.asarray(prm["W"], dtype=np.float32))
        sd[f"{model_key}__m"] = torch.from_numpy(np.asarray(prm["m"], dtype=np.float32))
        sd[f"{model_key}__lam"] = torch.from_numpy(np.asarray(prm["lam"], dtype=np.float32))
        sd[f"{model_key}__V_top64"] = torch.from_numpy(np.asarray(prm["V_top64"], dtype=np.float32))
        sd[f"{model_key}__scalars"] = torch.tensor(
            [float(prm.get("T1", 2.40)), float(prm.get("Tg", 2.35)), float(prm.get("ae", 0.10))],
            dtype=torch.float32
        )
    save_file(sd, str(path))
    sz_mb = path.stat().st_size / (1024 * 1024)
    print(f"[wPCA] Saved {len(bundle)} models to wPCA bundle at {path} ({sz_mb:.2f} MB)")


def load_wpca_bundle(path: Any) -> Dict[str, dict]:
    """Loads all model wPCA parameters from a single safetensors file."""
    from safetensors.torch import load_file
    sd = load_file(str(path))
    model_keys = set()
    for k in sd.keys():
        if "__" in k:
            model_keys.add(k.split("__")[0])
    bundle = {}
    for mk in sorted(model_keys):
        sc = sd[f"{mk}__scalars"].cpu().numpy()
        bundle[mk] = {
            "mx": sd[f"{mk}__mx"].cpu().numpy(),
            "sx": sd[f"{mk}__sx"].cpu().numpy(),
            "W": sd[f"{mk}__W"].cpu().numpy(),
            "m": sd[f"{mk}__m"].cpu().numpy().astype(np.float64),
            "lam": sd[f"{mk}__lam"].cpu().numpy().astype(np.float64),
            "V_top64": sd[f"{mk}__V_top64"].cpu().numpy(),
            "T1": float(sc[0]),
            "Tg": float(sc[1]),
            "ae": float(sc[2]),
        }
    print(f"[wPCA] Loaded {len(bundle)} models from wPCA bundle at {path}: {list(bundle.keys())}")
    return bundle


In [ ]:
# === MODULE: src/namethatllm/spectral_bank.py ===
"""
Leak-Free Prior-Scaled 384-Dim Joint Spectral Bank
=================================================
Constructs full-rank cross-backbone spectral representations from unsupervised
prompt-centered principal components V_top64, scaled by out-of-sample prior scales.
"""

from typing import List, Tuple, Dict, Any, Optional
import numpy as np
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression


def project_spectral_block(Z: np.ndarray, res: dict, prior_scale: float) -> np.ndarray:
    """
    Projects a single representation matrix onto unsupervised within-prompt V_top64
    and scales by out-of-sample prior scale.
    """
    if len(Z) == 0:
        d_out = res["V_top64"].shape[1] if "V_top64" in res else 64
        return np.zeros((0, d_out), dtype=np.float32)
    mx, sx, V64 = res["mx"], res["sx"], res["V_top64"]
    S = (((Z - mx) / sx) @ V64).astype(np.float32)
    return (S / np.linalg.norm(S, axis=1, keepdims=True).clip(min=1e-6)) * float(prior_scale)


def make_spectral_blocks(Z_tr: np.ndarray, Z_te: np.ndarray, res: dict, prior_scale: float) -> Tuple[np.ndarray, np.ndarray]:
    """
    Backward-compatible wrapper projecting train and test features onto V_top64.
    """
    return project_spectral_block(Z_tr, res, prior_scale), project_spectral_block(Z_te, res, prior_scale)


def fit_joint_spectral_bank(
    spectral_tr_blocks: List[np.ndarray],
    spectral_te_blocks: List[np.ndarray],
    y_sub: np.ndarray,
    prompts_sub: np.ndarray,
    D_tfidf: np.ndarray,
    D_neural_sum: np.ndarray,
    D_neural_weight_total: float,
    N_test: int,
    plain_C: float = 0.04,
    center_C: float = 0.03,
    center_blend: float = 0.35,
    max_cluster_size: int = 14,
    spectral_u_blocks: Optional[List[np.ndarray]] = None,
    D_tfidf_u: Optional[np.ndarray] = None,
    D_neural_sum_u: Optional[np.ndarray] = None,
    D_neural_weight_total_u: float = 0.0,
    N_unseen: int = 0,
    return_unseen: bool = False,
    save_path: Optional[Any] = None,
) -> Any:
    """
    Fits plain and prompt-centered logistic regressions on stacked unsupervised spectral features.
    Returns:
    - If return_unseen: (P_spectral, oof_bacc, P_spectral_u)
    - Otherwise: (P_spectral, oof_bacc)
    """
    if len(spectral_tr_blocks) == 0:
        p_te = np.ones((N_test, K), dtype=np.float64) / float(K)
        p_u = np.ones((N_unseen, K), dtype=np.float64) / float(K) if N_unseen > 0 else None
        return (p_te, 0.0, p_u) if return_unseen else (p_te, 0.0)

    X_spec_tr = np.hstack(spectral_tr_blocks)
    X_spec_te = np.hstack(spectral_te_blocks)

    # 5-fold GroupKFold OOF Balanced Accuracy (zero label leakage)
    gkf = GroupKFold(n_splits=5)
    oof_preds = np.zeros(len(y_sub), dtype=int)
    for tr_idx, val_idx in gkf.split(X_spec_tr, y_sub, groups=prompts_sub):
        clf_fold = LogisticRegression(C=plain_C, max_iter=2000, random_state=42)
        clf_fold.fit(X_spec_tr[tr_idx], y_sub[tr_idx])
        oof_preds[val_idx] = clf_fold.predict(X_spec_tr[val_idx])
    oof_bacc = float(balanced_accuracy_score(y_sub, oof_preds))

    # Plain Logistic Regression
    clf_plain = LogisticRegression(C=plain_C, max_iter=2000, random_state=42)
    clf_plain.fit(X_spec_tr, y_sub)
    lp_spec_plain = clf_plain.predict_log_proba(X_spec_te)
    if hasattr(clf_plain, 'classes_'):
        lp_spec_plain = align_cols(lp_spec_plain, [LABELS[c] for c in clf_plain.classes_])

    # Prompt-Centered Logistic Regression
    X_spec_tr_c = X_spec_tr.copy()
    for p in np.unique(prompts_sub):
        mask = (prompts_sub == p)
        X_spec_tr_c[mask] -= X_spec_tr_c[mask].mean(axis=0, keepdims=True)
    clf_center = LogisticRegression(C=center_C, max_iter=2000, random_state=42)
    clf_center.fit(X_spec_tr_c, y_sub)

    if save_path:
        spectral_params = {
            "W_plain": clf_plain.coef_,
            "b_plain": clf_plain.intercept_,
            "W_center": clf_center.coef_,
            "b_center": clf_center.intercept_,
            "center_blend": center_blend,
        }
        try:
            save_spectral_params(spectral_params, save_path)
            print(f"[Spectral Bank] Saved fitted spectral parameters to {save_path}")
        except Exception as e:
            print(f"[Spectral Bank] Warning: Could not save spectral params: {e}")

    # Prompt-centered test features
    X_spec_te_c = X_spec_te.copy()
    D_neural_final = D_neural_sum / max(1e-6, D_neural_weight_total) if D_neural_weight_total > 0 else D_tfidf
    D_hybrid_final = 0.55 * D_tfidf + 0.45 * D_neural_final
    g65_final = fcluster(linkage(squareform(D_hybrid_final, checks=False), method='average'), t=0.65, criterion='distance') if N_test >= 2 else np.ones(N_test, dtype=int)
    safe_groups = split_oversized_clusters_dist(D_hybrid_final, g65_final, max_cluster_size=max_cluster_size)

    if N_test >= 2 and len(np.unique(safe_groups)) > 0:
        for g in np.unique(safe_groups):
            mask = (safe_groups == g)
            if mask.sum() > 1:
                X_spec_te_c[mask] -= X_spec_te_c[mask].mean(axis=0, keepdims=True)
            else:
                X_spec_te_c[mask] = 0.0
    else:
        X_spec_te_c = np.zeros_like(X_spec_te)

    lp_spec_center = clf_center.predict_log_proba(X_spec_te_c)
    if hasattr(clf_center, 'classes_'):
        lp_spec_center = align_cols(lp_spec_center, [LABELS[c] for c in clf_center.classes_])

    lp_spec_joint = lp_spec_plain + center_blend * lp_spec_center
    P_spectral = softmax(lp_spec_joint, axis=1)

    # Symmetrical evaluation on unseen holdout benchmark
    P_spectral_u = None
    if spectral_u_blocks and len(spectral_u_blocks) == len(spectral_tr_blocks) and N_unseen > 0:
        X_spec_u = np.hstack(spectral_u_blocks)
        lp_spec_u_plain = clf_plain.predict_log_proba(X_spec_u)
        if hasattr(clf_plain, 'classes_'):
            lp_spec_u_plain = align_cols(lp_spec_u_plain, [LABELS[c] for c in clf_plain.classes_])

        X_spec_u_c = X_spec_u.copy()
        D_neural_u = D_neural_sum_u / max(1e-6, D_neural_weight_total_u) if (D_neural_weight_total_u > 0 and D_neural_sum_u is not None) else (D_tfidf_u if D_tfidf_u is not None else np.zeros((N_unseen, N_unseen), dtype=np.float32))
        D_hybrid_u = 0.55 * (D_tfidf_u if D_tfidf_u is not None else D_neural_u) + 0.45 * D_neural_u
        if N_unseen >= 2 and D_hybrid_u.shape[0] == N_unseen:
            g65_u = fcluster(linkage(squareform(D_hybrid_u, checks=False), method='average'), t=0.65, criterion='distance')
            safe_groups_u = split_oversized_clusters_dist(D_hybrid_u, g65_u, max_cluster_size=max_cluster_size)
            for g in np.unique(safe_groups_u):
                mask = (safe_groups_u == g)
                if mask.sum() > 1:
                    X_spec_u_c[mask] -= X_spec_u_c[mask].mean(axis=0, keepdims=True)
                else:
                    X_spec_u_c[mask] = 0.0
        else:
            X_spec_u_c = np.zeros_like(X_spec_u)

        lp_spec_u_center = clf_center.predict_log_proba(X_spec_u_c)
        if hasattr(clf_center, 'classes_'):
            lp_spec_u_center = align_cols(lp_spec_u_center, [LABELS[c] for c in clf_center.classes_])

        lp_spec_u_joint = lp_spec_u_plain + center_blend * lp_spec_u_center
        P_spectral_u = softmax(lp_spec_u_joint, axis=1)

    if return_unseen:
        return P_spectral, oof_bacc, P_spectral_u
    return P_spectral, oof_bacc


def save_spectral_params(prm: dict, path: Any):
    """Serializes fitted Spectral Bank linear heads to safetensors."""
    from pathlib import Path
    import torch
    from safetensors.torch import save_file

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sd = {
        "W_plain": torch.from_numpy(np.asarray(prm["W_plain"], dtype=np.float32)),
        "b_plain": torch.from_numpy(np.asarray(prm["b_plain"], dtype=np.float32)),
        "W_center": torch.from_numpy(np.asarray(prm["W_center"], dtype=np.float32)),
        "b_center": torch.from_numpy(np.asarray(prm["b_center"], dtype=np.float32)),
        "center_blend": torch.tensor([float(prm.get("center_blend", 0.35))], dtype=torch.float32),
    }
    save_file(sd, str(path))


def load_spectral_params(path: Any) -> dict:
    """Loads fitted Spectral Bank parameters from safetensors."""
    from safetensors.torch import load_file

    sd = load_file(str(path))
    return {
        "W_plain": sd["W_plain"].cpu().numpy(),
        "b_plain": sd["b_plain"].cpu().numpy(),
        "W_center": sd["W_center"].cpu().numpy(),
        "b_center": sd["b_center"].cpu().numpy(),
        "center_blend": float(sd["center_blend"].cpu().numpy()[0]),
        "labels": list(LABELS),
    }


def train_and_save_spectral_bank(
    spectral_tr_blocks: List[np.ndarray],
    y_sub: np.ndarray,
    prompts_sub: np.ndarray,
    output_path: Any,
    plain_C: float = 0.04,
    center_C: float = 0.03,
    center_blend: float = 0.35,
) -> dict:
    """
    Fits plain and prompt-centered LogisticRegression heads on stacked spectral training blocks
    and serializes the weights to safetensors. Zero GPU dependency.
    """
    X_spec_tr = np.hstack(spectral_tr_blocks)

    clf_plain = LogisticRegression(C=plain_C, max_iter=2000, random_state=42)
    clf_plain.fit(X_spec_tr, y_sub)
    W_p = clf_plain.coef_
    b_p = clf_plain.intercept_
    if hasattr(clf_plain, 'classes_'):
        classes = clf_plain.classes_
        if len(classes) < K:
            full_W = np.zeros((K, X_spec_tr.shape[1]), dtype=np.float32)
            full_b = np.zeros(K, dtype=np.float32)
            for idx, c in enumerate(classes):
                full_W[c] = W_p[idx]
                full_b[c] = b_p[idx]
            W_p, b_p = full_W, full_b

    X_spec_tr_c = X_spec_tr.copy()
    for p in np.unique(prompts_sub):
        mask = (prompts_sub == p)
        X_spec_tr_c[mask] -= X_spec_tr_c[mask].mean(axis=0, keepdims=True)
    clf_center = LogisticRegression(C=center_C, max_iter=2000, random_state=42)
    clf_center.fit(X_spec_tr_c, y_sub)

    W_c = clf_center.coef_
    b_c = clf_center.intercept_
    if hasattr(clf_center, 'classes_'):
        classes = clf_center.classes_
        if len(classes) < K:
            full_W = np.zeros((K, X_spec_tr.shape[1]), dtype=np.float32)
            full_b = np.zeros(K, dtype=np.float32)
            for idx, c in enumerate(classes):
                full_W[c] = W_c[idx]
                full_b[c] = b_c[idx]
            W_c, b_c = full_W, full_b

    prm = {
        "W_plain": W_p.astype(np.float32),
        "b_plain": b_p.astype(np.float32),
        "W_center": W_c.astype(np.float32),
        "b_center": b_c.astype(np.float32),
        "center_blend": center_blend,
        "labels": list(LABELS),
    }
    save_spectral_params(prm, output_path)
    print(f"[Spectral Bank Trainer] Saved fitted spectral heads to {output_path}")
    return prm


def predict_spectral_bank(
    spectral_blocks: List[np.ndarray],
    prm: dict,
    D_tfidf: np.ndarray,
    D_neural_sum: Optional[np.ndarray] = None,
    D_neural_weight_total: float = 0.0,
    max_cluster_size: int = 14,
) -> np.ndarray:
    """
    Pure inference evaluation of Spectral Bank using pre-trained weights.
    Zero training data required; executes in <1 ms on CPU.
    """
    if len(spectral_blocks) == 0:
        N = D_tfidf.shape[0] if D_tfidf is not None else 0
        return np.ones((N, K), dtype=np.float32) / float(K)

    X_spec = np.hstack(spectral_blocks)
    N = X_spec.shape[0]

    W_p, b_p = prm["W_plain"], prm["b_plain"]
    W_c, b_c = prm["W_center"], prm["b_center"]
    blend = float(prm.get("center_blend", 0.35))
    labels = prm.get("labels", list(LABELS))

    lp_plain = log_softmax(align_cols(X_spec @ W_p.T + b_p, labels))

    X_spec_c = X_spec.copy()
    D_neural = D_neural_sum / max(1e-6, D_neural_weight_total) if (D_neural_weight_total > 0 and D_neural_sum is not None) else D_tfidf
    D_hybrid = 0.55 * (D_tfidf if D_tfidf is not None else D_neural) + 0.45 * D_neural
    if N >= 2 and D_hybrid is not None and D_hybrid.shape[0] == N:
        g65 = fcluster(linkage(squareform(D_hybrid, checks=False), method='average'), t=0.65, criterion='distance')
        safe_groups = split_oversized_clusters_dist(D_hybrid, g65, max_cluster_size=max_cluster_size)
        for g in np.unique(safe_groups):
            mask = (safe_groups == g)
            if mask.sum() > 1:
                X_spec_c[mask] -= X_spec_c[mask].mean(axis=0, keepdims=True)
            else:
                X_spec_c[mask] = 0.0
    else:
        X_spec_c = np.zeros_like(X_spec)

    lp_center = log_softmax(align_cols(X_spec_c @ W_c.T + b_c, labels))
    return softmax(lp_plain + blend * lp_center, axis=1)


In [ ]:
# === MODULE: src/namethatllm/neural_extractor.py ===
"""
Universal Neural Trunk Loader & Feature Extractor
=================================================
Manages multi-GPU sharding (2xT4 GPUs, max 13.5 GiB), layer truncation (keep_layers),
dynamic bfloat16 row-level retry for 14B fp16 overflow rescue, LayerNorm heads,
and bit-exact CPU fp16 snapshot/restore for shared trunks.
"""

import gc
import os
import re
import json
import time
import shutil
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel, AutoTokenizer
from safetensors.torch import load_file


def infer_adapter_max_layer(ckpt_dir: Path) -> Optional[int]:
    """Returns 1-indexed max transformer layer modified by the LoRA adapter (e.g. 24 or 27)."""
    ckpt_dir = Path(ckpt_dir)
    for rel in (
        "adapter_model.safetensors",
        "adapter/adapter_model.safetensors",
        "adapter_model.bin",
        "adapter/adapter_model.bin",
        "adapter_model.pt",
        "adapter/adapter_model.pt",
    ):
        p = ckpt_dir / rel
        if p.exists():
            try:
                if p.suffix == ".safetensors":
                    sd = load_file(str(p))
                else:
                    sd = torch.load(str(p), map_location="cpu", weights_only=False)
                    if hasattr(sd, "state_dict") and callable(sd.state_dict):
                        sd = sd.state_dict()
                layer_nums = []
                for k in sd.keys():
                    m = re.search(r"layers\.(\d+)\.", str(k))
                    if m:
                        layer_nums.append(int(m.group(1)) + 1)
                if layer_nums:
                    return max(layer_nums)
            except Exception:
                continue
    return None

def in_place_copy(dst: torch.Tensor, src: torch.Tensor):
    """In-place weight copy."""
    dst.data.copy_(src.to(device=dst.device, dtype=dst.dtype))

def _resolve_module_name(k: str, state_dict: dict) -> Optional[str]:
    """Resolves module name across various AutoModel and PEFT naming schemes."""
    raw = k.split(".lora_A")[0]
    candidates = [raw, raw.replace("base_model.model.", "")]
    if "layers." in raw:
        rel = raw[raw.index("layers."):]
        candidates.extend([rel, f"model.{rel}"])
    for c in candidates:
        if c in state_dict and hasattr(state_dict[c], "weight"):
            return c
    return None

def merge_lora(model: nn.Module, adapter_path: Path, sign: float = +1.0) -> bool:
    """
    Atomically merges or unmerges (+1.0 or -1.0) a LoRA adapter into trunk weights.
    Pre-validates all tensor shapes BEFORE mutating any weights so an incompatible
    architecture (e.g. Llama vs Qwen) never partially corrupts the trunk.
    """
    weights_path = adapter_path / "adapter_model.safetensors"
    if not weights_path.exists():
        weights_path = adapter_path / "adapter" / "adapter_model.safetensors"
    if not weights_path.exists():
        return False

    try:
        sd = load_file(str(weights_path))
    except Exception as e:
        print(f"[merge_lora] Could not load {weights_path}: {e}")
        return False

    cfg_p = adapter_path / "adapter_config.json"
    if not cfg_p.exists():
        cfg_p = adapter_path / "adapter" / "adapter_config.json"
    scale = 1.0
    if cfg_p.exists():
        try:
            c = json.load(open(cfg_p))
            scale = float(c.get("lora_alpha", 16)) / float(c.get("r", 16))
        except Exception:
            scale = 1.0

    state_dict = dict(model.named_modules())
    pairs = {}
    for k in sd.keys():
        if "lora_A" in k:
            b_key = k.replace("lora_A", "lora_B")
            if b_key in sd:
                m_name = _resolve_module_name(k, state_dict)
                if m_name is not None:
                    pairs[m_name] = (sd[k], sd[b_key])

    if not pairs:
        return False

    # Pass 1: Pre-validate all shapes before modifying any weight in `model`
    for m_name, (A, B) in pairs.items():
        mod = state_dict.get(m_name)
        if mod is None or not hasattr(mod, "weight"):
            continue
        expected_shape = (B.shape[0], A.shape[1])
        if tuple(mod.weight.shape) != expected_shape:
            print(
                f"[merge_lora] Architecture mismatch on {m_name} in {adapter_path.name}: "
                f"trunk weight {tuple(mod.weight.shape)} != adapter delta {expected_shape}. Aborting merge."
            )
            return False

    # Pass 2: All shapes verified — apply in-place update
    applied = 0
    with torch.no_grad():
        for m_name, (A, B) in pairs.items():
            mod = state_dict.get(m_name)
            if mod is not None and hasattr(mod, "weight"):
                A_d = A.to(mod.weight.device, dtype=torch.float32)
                B_d = B.to(mod.weight.device, dtype=torch.float32)
                delta = (B_d @ A_d) * (scale * sign)
                mod.weight.data.add_(delta.to(mod.weight.dtype))
                applied += 1
    return applied > 0

@torch.no_grad()
def forward_and_pool_batch(model: nn.Module, enc: dict, layers: List[int], norms: nn.ModuleList,
                           dtype: torch.dtype = torch.float16, drop_first_token: bool = True):
    """
    Runs forward pass in `dtype` (float16 on T4).
    CRITICAL: If any target layer overflows fp16 (Inf/NaN), recomputes ONLY the overflowed
    rows in `torch.bfloat16` and pools directly in float32 before restoring `dtype`!
    """
    hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states

    B = enc["input_ids"].shape[0]
    first_norm_dev = norms[0].weight.device

    # Check if any selected layer has Inf/NaN per row
    bad_mask = torch.zeros(B, dtype=torch.bool, device="cpu")
    for l in layers:
        bad_mask |= ~torch.isfinite(hs[l]).all(dim=(1, 2)).cpu()

    n_bad = int(bad_mask.sum().item())
    m = enc["attention_mask"].unsqueeze(-1).float().to(first_norm_dev)
    if drop_first_token:
        m[:, 0, :] = 0.0
    den = m.sum(dim=1).clamp(min=1.0)

    # Pool finite rows first in float32
    pooled_layers = [torch.zeros((B, model.config.hidden_size), dtype=torch.float32, device=first_norm_dev) for _ in layers]
    good_rows = torch.where(~bad_mask)[0].to(first_norm_dev)
    if len(good_rows) > 0:
        for idx_l, l in enumerate(layers):
            h_good = hs[l][good_rows.to(hs[l].device)].to(first_norm_dev).float()
            pooled_layers[idx_l][good_rows] = (h_good * m[good_rows]).sum(dim=1) / den[good_rows]

    # Rescue overflowed rows in bfloat16 (exact exponent range as float32, NEVER overflows!)
    if n_bad > 0:
        bad_rows = torch.where(bad_mask)[0]
        del hs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        model.to(torch.bfloat16)
        enc_bad = {k: v[bad_rows] for k, v in enc.items()}
        hs_bad = model(**enc_bad, output_hidden_states=True, use_cache=False).hidden_states
        bad_rows_dev = bad_rows.to(first_norm_dev)
        for idx_l, l in enumerate(layers):
            h_b = hs_bad[l].to(first_norm_dev).float()
            h_b = torch.nan_to_num(h_b, nan=0.0, posinf=1e4, neginf=-1e4)
            pooled_layers[idx_l][bad_rows_dev] = (h_b * m[bad_rows_dev]).sum(dim=1) / den[bad_rows_dev]
        del hs_bad
        model.to(dtype)

    out = torch.cat([nm(pooled_layers[idx_l]) for idx_l, nm in enumerate(norms)], dim=-1)
    return out.cpu().numpy(), n_bad

def _flatten_state_dict(obj: Any, prefix: str = "", out: Optional[dict] = None) -> dict:
    if out is None:
        out = {}
    if hasattr(obj, "state_dict") and callable(obj.state_dict):
        obj = obj.state_dict()
    if isinstance(obj, dict):
        for k, v in obj.items():
            new_k = f"{prefix}.{k}" if prefix else str(k)
            if isinstance(v, (dict, list, tuple)) or (hasattr(v, "state_dict") and callable(v.state_dict)):
                _flatten_state_dict(v, new_k, out)
            else:
                out[new_k] = v
                if prefix in ("state_dict", "model", "heads", "head") and str(k) not in out:
                    out[str(k)] = v
    elif isinstance(obj, (list, tuple)):
        for idx, item in enumerate(obj):
            new_k = f"{prefix}.{idx}" if prefix else str(idx)
            _flatten_state_dict(item, new_k, out)
    return out

def load_head_file(path_or_dir: Path) -> dict:
    """Unified loader for classification head state dicts (.safetensors, .pt, .bin, .npz)."""
    p = Path(path_or_dir)
    res = {}
    if p.is_file():
        if p.suffix == ".safetensors":
            res = load_file(str(p))
        elif p.suffix in (".pt", ".bin"):
            raw = torch.load(str(p), map_location="cpu", weights_only=False)
            res = _flatten_state_dict(raw)
        elif p.suffix == ".npz":
            npz = np.load(str(p), allow_pickle=True)
            res = {k: npz[k] for k in npz.files}
        if res:
            print(f"[Head File] {p.name} loaded ({len(res)} keys): {list(res.keys())[:15]}")
        return res
    for fname in ("heads.safetensors", "head.safetensors", "heads.pt", "head.pt", "readouts.npz"):
        cand = p / fname
        if cand.exists():
            return load_head_file(cand)
    return {}

def resolve_head_tensors(heads_dict: dict) -> Tuple[Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray]]:
    """Resolves plain and centered head weights and biases matching American/British naming and handles transposed weights."""
    W_p, b_p, W_c, b_c = None, None, None, None
    for wk, bk in [
        ("plain.weight", "plain.bias"),
        ("plain_coef", "plain_intercept"),
        ("head.weight", "head.bias"),
        ("classifier.weight", "classifier.bias"),
        ("fc.weight", "fc.bias"),
        ("W_plain", "b_plain"),
        ("weight", "bias"),
        ("W", "b"),
    ]:
        if wk in heads_dict and bk in heads_dict:
            W_p, b_p = heads_dict[wk], heads_dict[bk]
            break
        elif wk in heads_dict and bk not in heads_dict:
            W_p = heads_dict[wk]
            b_p = np.zeros(K, dtype=np.float32)
            break

    for wk, bk in [
        ("centered.weight", "centered.bias"),
        ("centred.weight", "centred.bias"),
        ("center.weight", "center.bias"),
        ("twin.weight", "twin.bias"),
        ("center_coef", "center_intercept"),
        ("centred_coef", "centred_intercept"),
        ("W_center", "b_center"),
        ("W_twin", "b_twin"),
    ]:
        if wk in heads_dict and bk in heads_dict:
            W_c, b_c = heads_dict[wk], heads_dict[bk]
            break
        elif wk in heads_dict and bk not in heads_dict:
            W_c = heads_dict[wk]
            b_c = np.zeros(K, dtype=np.float32)
            break

    # Suffix / Keyword Matching fallback if W_p or W_c still None
    if W_p is None or W_c is None:
        for k, v in heads_dict.items():
            k_lower = str(k).lower()
            shape = getattr(v, "shape", None)
            if shape is not None and len(shape) == 2 and (shape[0] == K or shape[1] == K):
                # Check for center / twin
                if any(tag in k_lower for tag in ("center", "centred", "twin")):
                    if W_c is None:
                        W_c = v
                        base_prefix = k.rsplit(".", 1)[0] if "." in k else k.rsplit("_", 1)[0]
                        for bk in [f"{base_prefix}.bias", f"{base_prefix}_bias", f"{base_prefix}.intercept", f"{base_prefix}_intercept", "bias", "centered.bias", "b"]:
                            if bk in heads_dict and getattr(heads_dict[bk], "shape", None) == (K,):
                                b_c = heads_dict[bk]
                                break
                        if b_c is None:
                            b_c = np.zeros(K, dtype=np.float32)
                # Check for plain / head / clf / classifier
                elif any(tag in k_lower for tag in ("plain", "head", "clf", "classifier")):
                    if W_p is None:
                        W_p = v
                        base_prefix = k.rsplit(".", 1)[0] if "." in k else k.rsplit("_", 1)[0]
                        for bk in [f"{base_prefix}.bias", f"{base_prefix}_bias", f"{base_prefix}.intercept", f"{base_prefix}_intercept", "bias", "plain.bias", "b"]:
                            if bk in heads_dict and getattr(heads_dict[bk], "shape", None) == (K,):
                                b_p = heads_dict[bk]
                                break
                        if b_p is None:
                            b_p = np.zeros(K, dtype=np.float32)

    if W_p is not None and hasattr(W_p, "cpu"):
        W_p = W_p.cpu().numpy()
    if b_p is not None and hasattr(b_p, "cpu"):
        b_p = b_p.cpu().numpy()
    if W_c is not None and hasattr(W_c, "cpu"):
        W_c = W_c.cpu().numpy()
    if b_c is not None and hasattr(b_c, "cpu"):
        b_c = b_c.cpu().numpy()

    # Transpose Guard: if shape is (D, K) where K == 18, transpose to canonical (18, D)
    if W_p is not None and W_p.ndim == 2:
        if W_p.shape[1] == K and W_p.shape[0] != K:
            W_p = W_p.T
    if W_c is not None and W_c.ndim == 2:
        if W_c.shape[1] == K and W_c.shape[0] != K:
            W_c = W_c.T

    return W_p, b_p, W_c, b_c

def compute_centered_logits(Z_test: np.ndarray, groups: np.ndarray, W_c: np.ndarray, b_c: np.ndarray, src_labels: list) -> Tuple[np.ndarray, np.ndarray]:
    """Computes prompt-topic centered twin logits for valid multi-sample clusters."""
    N = len(Z_test)
    out_lp = np.zeros((N, K), dtype=np.float64)
    if N < 2 or W_c is None:
        return out_lp, np.zeros(N, dtype=bool)
    _, inv_g = np.unique(np.asarray(groups, dtype=str), return_inverse=True)
    safe_g = inv_g + 1
    sz = np.bincount(safe_g)[safe_g]
    valid = (sz >= 2) & (sz <= 18)
    if not valid.any():
        return out_lp, valid
    Zc = Z_test.copy()
    for g in np.unique(safe_g[valid]):
        mask = (safe_g == g)
        Zc[mask] -= Zc[mask].mean(axis=0, keepdims=True)
    raw_c = Zc[valid] @ W_c.T + b_c
    out_lp[valid] = log_softmax(align_cols(raw_c, src_labels))
    return out_lp, valid

class UniversalLoRATrunkLoader:
    def __init__(self, base_dir: Path, keep_layers: int, layers: List[int], max_len: int = 447):
        if base_dir is not None and not (base_dir / "config.json").exists():
            for sub in base_dir.rglob("config.json"):
                base_dir = sub.parent
                break
        self.base_dir = base_dir
        self.keep_layers = keep_layers
        self.layers = layers
        self.max_len = max_len
        self.total_rescued = 0
        self.tok = AutoTokenizer.from_pretrained(str(base_dir), trust_remote_code=True)
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token or "<|endoftext|>"

        cfg = AutoConfig.from_pretrained(str(base_dir), trust_remote_code=True)
        cfg.num_hidden_layers = keep_layers
        if getattr(cfg, "layer_types", None) is not None:
            cfg.layer_types = cfg.layer_types[:keep_layers]
        cfg.use_cache = False

        n_gpus = torch.cuda.device_count()
        if n_gpus >= 2:
            max_memory = {0: "13.5GiB", 1: "13.5GiB"}
            self.model = AutoModel.from_pretrained(
                str(base_dir), config=cfg, device_map="auto", max_memory=max_memory,
                torch_dtype=torch.float16, trust_remote_code=True, low_cpu_mem_usage=True
            )
        elif n_gpus == 1:
            self.model = AutoModel.from_pretrained(
                str(base_dir), config=cfg, device_map="auto",
                torch_dtype=torch.float16, trust_remote_code=True, low_cpu_mem_usage=True
            )
        else:
            self.model = AutoModel.from_pretrained(str(base_dir), config=cfg, torch_dtype=torch.float32, low_cpu_mem_usage=True)

        # Preserve original final trunk norm before optionally disabling
        self.orig_final_norm = getattr(self.model, "norm", None) or getattr(getattr(self.model, "model", None), "norm", None)

        # Default: disable final trunk norm for two-stage models
        if hasattr(self.model, "norm"):
            self.model.norm = nn.Identity()
        elif hasattr(self.model, "model") and hasattr(self.model.model, "norm"):
            self.model.model.norm = nn.Identity()

        self.model.eval()
        self.device_norms = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        # nn.LayerNorm with both weight and bias matching heads.safetensors
        hidden_dim = self.model.config.hidden_size
        self.norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim)
            for _ in layers
        ]).to(self.device_norms).float()

        self.norms.eval()
        self.first_dev = next(self.model.parameters()).device

    def set_final_norm(self, enabled: bool):
        """Restores original trunk norm when enabled=True, disables with nn.Identity() when False."""
        target = self.orig_final_norm if enabled and self.orig_final_norm is not None else nn.Identity()
        if hasattr(self.model, "norm"):
            self.model.norm = target
        elif hasattr(self.model, "model") and hasattr(self.model.model, "norm"):
            self.model.model.norm = target

    def snapshot_weights(self, adapter_path: Path) -> dict:
        """Saves CPU fp16 copies of trunk modules modified by adapter."""
        weights_path = adapter_path / "adapter_model.safetensors"
        if not weights_path.exists():
            weights_path = adapter_path / "adapter" / "adapter_model.safetensors"
        if not weights_path.exists():
            return {}
        sd = load_file(str(weights_path))
        state_dict = dict(self.model.named_modules())
        backup = {}
        for k in sd.keys():
            if "lora_A" in k:
                m_name = _resolve_module_name(k, state_dict)
                if m_name is not None and m_name in state_dict and hasattr(state_dict[m_name], "weight"):
                    backup[m_name] = state_dict[m_name].weight.detach().cpu().half().clone()
        return backup

    def restore_weights(self, backup: dict):
        """Restores base trunk weights bit-for-bit without FP16 subtraction drift."""
        state_dict = dict(self.model.named_modules())
        with torch.no_grad():
            for m_name, orig in backup.items():
                if m_name in state_dict and hasattr(state_dict[m_name], "weight"):
                    state_dict[m_name].weight.data.copy_(orig.to(state_dict[m_name].weight.device))

    def set_active_layers(self, layers: List[int], max_len: Optional[int] = None):
        """Updates active pooled layers and instantiates LayerNorms matching the checkpoint."""
        self.layers = list(layers)
        if max_len is not None:
            self.max_len = max_len
        hidden_dim = self.model.config.hidden_size
        self.norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim)
            for _ in self.layers
        ]).to(self.device_norms).float()
        self.norms.eval()

    def load_norm_weights(self, heads_sd: dict):
        """Loads LayerNorm weights and biases from head state dict after resetting to identity."""
        with torch.no_grad():
            # First reset every ln in self.norms to identity so weights/biases from previous checkpoint never bleed
            for ln in self.norms:
                ln.weight.data.fill_(1.0)
                if hasattr(ln, "bias") and ln.bias is not None:
                    ln.bias.data.zero_()

            # Support all norm key prefixes: norms.{i}, layer_norms.{i}, ln.{i}, norm.{i}
            for i, ln in enumerate(self.norms):
                for pfx in ("norms", "layer_norms", "ln", "norm"):
                    w_k = f"{pfx}.{i}.weight"
                    b_k = f"{pfx}.{i}.bias"
                    if w_k in heads_sd:
                        ln.weight.copy_(heads_sd[w_k].to(ln.weight.device, ln.weight.dtype))
                    elif f"{pfx}.{i}" in heads_sd and hasattr(heads_sd[f"{pfx}.{i}"], "shape") and heads_sd[f"{pfx}.{i}"].shape == ln.weight.shape:
                        ln.weight.copy_(heads_sd[f"{pfx}.{i}"].to(ln.weight.device, ln.weight.dtype))

                    if b_k in heads_sd and hasattr(ln, "bias") and ln.bias is not None:
                        ln.bias.copy_(heads_sd[b_k].to(ln.bias.device, ln.bias.dtype))

            # Check for stacked 2D tensor: norms_weight or norms.weight of shape (len(self.layers), H)
            for stk in ("norms_weight", "norms.weight", "layer_norms.weight", "ln.weight"):
                if stk in heads_sd and hasattr(heads_sd[stk], "ndim") and heads_sd[stk].ndim == 2:
                    stk_t = heads_sd[stk]
                    if stk_t.shape[0] == len(self.norms):
                        for i, ln in enumerate(self.norms):
                            ln.weight.copy_(stk_t[i].to(ln.weight.device, ln.weight.dtype))

    def _extract_batch_ids(self, id_list: List[List[int]], batch_size: int = 8) -> np.ndarray:
        order = np.argsort([len(x) for x in id_list], kind="stable")
        total_dim = len(self.layers) * self.model.config.hidden_size
        Z = np.zeros((len(id_list), total_dim), dtype=np.float32)

        for s in range(0, len(order), batch_size):
            idx = order[s : s + batch_size]
            enc = self.tok.pad({"input_ids": [id_list[i] for i in idx]}, padding=True, return_tensors="pt")
            enc = {k: v.to(self.first_dev) for k, v in enc.items()}
            out_np, n_bad = forward_and_pool_batch(self.model, enc, self.layers, self.norms, torch.float16, drop_first_token=True)
            self.total_rescued += n_bad
            Z[idx] = out_np
        return Z

    @torch.no_grad()
    def extract_exact(self, texts: List[str], batch_size: int = 8, end_id: Optional[int] = None) -> np.ndarray:
        t0 = time.time()
        self.total_rescued = 0
        raw_ids = self.tok(list(texts), add_special_tokens=False, truncation=True, max_length=self.max_len)["input_ids"]
        eff_end = end_id if end_id is not None else getattr(self.tok, "eos_token_id", 151643)
        if eff_end is not None:
            ids = [x + [eff_end] for x in raw_ids]
        else:
            ids = raw_ids
        Z = self._extract_batch_ids(ids, batch_size=batch_size)
        print(f"[Extractor] Extracted {len(texts)} samples in {time.time() - t0:.2f}s (Total BF16 Rescued Rows: {self.total_rescued}).")
        return Z

    def free_memory(self):
        del self.model, self.norms
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def read_checkpoint_meta(ckpt_dir: Path, default_keep: int, default_layers: List[int], default_max_len: int = 447) -> dict:
    """Reads keep, layers, max_len, end_id, and labels dynamically from meta.json or infers from adapter weights."""
    ckpt_dir = Path(ckpt_dir)
    meta_p = ckpt_dir / "meta.json"
    labels = list(LABELS)
    keep = default_keep
    layers = list(default_layers)
    max_len = default_max_len
    end_id = None
    if meta_p.exists():
        try:
            m = json.load(open(meta_p))
            labels = m.get("labels", m.get("classes", labels))
            cfg = m.get("cfg", m)
            keep = int(cfg.get("keep", keep))
            layers = list(cfg.get("layers", layers))
            max_len = int(cfg.get("max_len", max_len))
            for ek in ("end_token_id", "eos_id", "eos_token_id", "end_token"):
                if ek in cfg and isinstance(cfg[ek], int):
                    end_id = int(cfg[ek])
                    break
        except Exception:
            pass
    else:
        # Infer exact layer depth from adapter weights (e.g. l24 vs l27)
        max_l = infer_adapter_max_layer(ckpt_dir)
        if max_l == 24:
            keep = 24
            layers = [2, 4, 6, 8, 12, 16, 20, 24]
        elif max_l == 27:
            keep = 27
            layers = [2, 4, 6, 9, 13, 17, 22, 27]
        elif max_l == 32:
            keep = 32
            layers = [3, 5, 8, 11, 16, 21, 27, 32]
        elif max_l is not None:
            keep = max_l
        else:
            dir_str = str(ckpt_dir).lower().replace("\\", "/")
            if any(x in dir_str for x in ("l24", "q14v5", "ts_q3b14", "ts_q3b8", "q8v5", "ll8v5", "ts_llama8", "l24x")):
                keep = 24
                layers = [2, 4, 6, 8, 12, 16, 20, 24]

    dir_str = str(ckpt_dir).lower().replace("\\", "/")
    is_dhruv = any(x in dir_str for x in ("l24", "q14v5", "ts_q3b14", "ts_q3b8", "q8v5", "ll8v5", "ts_llama8", "l24x"))
    no_final_norm = False if is_dhruv else True
    if is_dhruv:
        max_len = 384
        end_id = None
        labels = sorted(LABELS)

    return {"keep": keep, "layers": layers, "max_len": max_len, "end_id": end_id, "no_final_norm": no_final_norm, "labels": labels}

def extract_deberta_checkpoint(
    ckpt_dir: Path,
    base_deb_dir: Optional[Path],
    texts_test: List[str],
    texts_train: List[str],
    y_train: np.ndarray,
    max_len: int = 512,
    batch_size: int = 8
) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray], list]:
    """
    Dedicated feature extractor for DeBERTa-v3-large sequence classification checkpoints.
    Supports both single-layer encoder (Layer 24 float32 last_hidden_state) and multi-layer heads.
    """
    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    heads_sd = load_head_file(ckpt_dir)
    W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd)

    # Load tokenizer from encoder directory if available, matching exact training setup
    if (ckpt_dir / "encoder").exists():
        tok_dir = str(ckpt_dir / "encoder")
    elif base_deb_dir and (base_deb_dir / "config.json").exists():
        tok_dir = str(base_deb_dir)
    else:
        tok_dir = str(ckpt_dir)
    tok = AutoTokenizer.from_pretrained(tok_dir, use_fast=True)

    meta = read_checkpoint_meta(ckpt_dir, default_keep=24, default_layers=[3, 6, 9, 12, 15, 18, 21, 24], default_max_len=max_len)
    ckpt_labels = meta["labels"]
    layers = meta["layers"]
    max_len = meta["max_len"]

    if (ckpt_dir / "encoder" / "config.json").exists():
        model_src = ckpt_dir / "encoder"
    elif (ckpt_dir / "config.json").exists() and ((ckpt_dir / "model.safetensors").exists() or (ckpt_dir / "pytorch_model.bin").exists()):
        model_src = ckpt_dir
    elif base_deb_dir and (base_deb_dir / "config.json").exists():
        model_src = base_deb_dir
    else:
        model_src = ckpt_dir

    stage_dir = Path("/kaggle/temp/deberta_p2_stage")
    staged = False
    load_src = model_src
    if Path("/kaggle/temp").exists() and os.access("/kaggle/temp", os.W_OK):
        try:
            if stage_dir.exists():
                shutil.rmtree(stage_dir, ignore_errors=True)
            shutil.copytree(model_src, stage_dir)
            load_src = stage_dir
            staged = True
        except Exception:
            load_src = model_src

    # CRITICAL: Always use torch.float32 for DeBERTa-v3 to avoid FP16 overflow!
    try:
        model = AutoModel.from_pretrained(str(load_src), torch_dtype=torch.float32).to(dev)
    finally:
        if staged and stage_dir.exists():
            shutil.rmtree(stage_dir, ignore_errors=True)

    model.eval()

    hidden_dim = model.config.hidden_size
    has_norm_keys = any(k.startswith("norms.") and k.endswith(".weight") for k in heads_sd)
    is_single_last_layer = (W_p is not None and W_p.shape[1] == hidden_dim and not has_norm_keys)

    if not is_single_last_layer:
        if W_p is not None and W_p.shape[1] > 0:
            expected_layers = W_p.shape[1] // hidden_dim
            if len(layers) != expected_layers:
                norm_keys = [k for k in heads_sd if k.startswith("norms.") and k.endswith(".weight")]
                if len(norm_keys) == expected_layers:
                    layers = layers[-expected_layers:] if len(layers) > expected_layers else list(range(expected_layers))
                elif len(layers) > expected_layers:
                    layers = layers[-expected_layers:]

        norms = nn.ModuleList([nn.LayerNorm(hidden_dim) for _ in layers]).to(dev).float()
        with torch.no_grad():
            for i, ln in enumerate(norms):
                w_k = f"norms.{i}.weight"
                b_k = f"norms.{i}.bias"
                if w_k in heads_sd:
                    ln.weight.copy_(heads_sd[w_k].to(dev, ln.weight.dtype))
                if b_k in heads_sd:
                    ln.bias.copy_(heads_sd[b_k].to(dev, ln.bias.dtype))
        norms.eval()
    else:
        norms = None

    def pool_texts(txt_list: List[str], drop_first: bool = False) -> np.ndarray:
        out_dim = hidden_dim if is_single_last_layer else (len(layers) * hidden_dim)
        if len(txt_list) == 0:
            return np.zeros((0, out_dim), dtype=np.float32)
        order = np.argsort([len(t) for t in txt_list], kind="stable")
        all_vecs = np.zeros((len(txt_list), out_dim), dtype=np.float32)
        for s in range(0, len(order), batch_size):
            idx = order[s : s + batch_size]
            chunk = [txt_list[i] for i in idx]
            enc = tok(chunk, padding=True, truncation=True, max_length=max_len, return_tensors="pt")
            enc = {k: v.to(dev) for k, v in enc.items()}
            with torch.no_grad():
                if is_single_last_layer:
                    out_hs = model(**enc).last_hidden_state
                    m = enc["attention_mask"].unsqueeze(-1).float()
                    if drop_first:
                        m[:, 0, :] = 0.0
                    den = m.sum(dim=1).clamp(min=1.0)
                    pooled = (torch.nan_to_num(out_hs.float(), nan=0.0, posinf=1e4, neginf=-1e4) * m).sum(dim=1) / den
                    all_vecs[idx] = pooled.cpu().numpy()
                else:
                    hs = model(**enc, output_hidden_states=True).hidden_states
                    m = enc["attention_mask"].unsqueeze(-1).float()
                    if drop_first:
                        m[:, 0, :] = 0.0
                    den = m.sum(dim=1).clamp(min=1.0)
                    pooled = [((torch.nan_to_num(hs[l].float(), nan=0.0, posinf=1e4, neginf=-1e4) * m).sum(dim=1) / den) for l in layers]
                    out = torch.cat([norms[i](pooled[i]) for i in range(len(layers))], dim=-1)
                    all_vecs[idx] = out.cpu().numpy()
        return all_vecs

    # Probe drop_first_token on train.csv[:180]
    probe_n = min(180, len(texts_train))
    best_drop = False if is_single_last_layer else True
    if probe_n > 0 and W_p is not None:
        best_acc = -1.0
        for cand_drop in ([False, True] if is_single_last_layer else [True, False]):
            z_pr = pool_texts(texts_train[:probe_n], drop_first=cand_drop)
            preds = align_cols(z_pr @ W_p.T + b_p, ckpt_labels).argmax(axis=1)
            acc = float((preds == y_train[:probe_n]).mean())
            if acc > best_acc:
                best_acc = acc
                best_drop = cand_drop
        print(f"[DeBERTa Extractor] Probed drop_first={best_drop} (accuracy = {best_acc:.4f})")

    out_dim = hidden_dim if is_single_last_layer else (len(layers) * hidden_dim)
    Z_test = pool_texts(texts_test, drop_first=best_drop)
    Z_train = pool_texts(texts_train, drop_first=best_drop) if len(texts_train) > 0 else np.zeros((0, out_dim), dtype=np.float32)

    del model
    if norms is not None:
        del norms
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return Z_test, Z_train, W_p, b_p, W_c, b_c, ckpt_labels


In [ ]:
# === MODULE: src/namethatllm/postprocess.py ===
"""
Postprocessing, Fusion & Calibration Engine
===========================================
This module handles all post-ensemble refinement stages:
1. Two-tier convex mixture (85% Main Stack + 15% Complementary Stack)
2. Mass-preserving Specialist Neural Gates (Claude head, Gemini LoRA, GLM LoRA, GPT LoRA)
3. Confidence-aware Within-Family Spectral Gates (dynamic margin tie-breakers)
4. Unified Handwriting & Orthographic Shield (discrete Bayesian tokenization rules + continuous density)
5. Cavity Mean-Field Cluster Uniqueness Repulsion (soft at-most-once constraint on prompt clusters)
6. Multiplicative Sinkhorn-Knopp Column Marginal Balancing (global class prior equalization)
7. Hierarchical Beta-Bernoulli TypoBayes Postprocessor
8. Final Kaggle submission generation (submission.parquet and submission.csv)
"""

import re
from pathlib import Path

import numpy as np
import pandas as pd


# =============================================================================
# 1. WITHIN-FAMILY CONFIDENCE GATES (SPECTRAL / TIE-BREAKER HEURISTICS)
# =============================================================================


def apply_all_family_confidence_gates(
    P_mix: np.ndarray,
    family_gates: dict[str, np.ndarray] | None,
    max_strength: float = 0.20,
    margin_thr: float = 0.45,
) -> np.ndarray:
    """
    Confidence-aware within-family and cross-family specialist reranker.

    Concept:
    --------
    When an ensemble predicts probabilities across 18 classes, sister models within
    the same family (e.g., Gemini 3.7 vs 3.8, or GLM-5.3 vs GLM-5.3-flash) can have
    extremely close probabilities (low margin). This function uses auxiliary gate
    logits (from spectral/TF-IDF classifiers) to break ties in low-margin cases.

    Key Invariant - Mass Conservation:
    ----------------------------------
    The total probability assigned to the family is strictly preserved!
    Only the internal relative distribution among the sisters is adjusted.
    Probabilities for classes outside the family are completely untouched.

    Dynamic Confidence Gating:
    --------------------------
    - Margin = (Top-1 prob within family) - (Top-2 prob within family).
    - If margin >= margin_thr: Model is already confident; gate strength = 0 (no intervention).
    - If margin < margin_thr: Low confidence; gate strength scales up to max_strength.

    Parameters:
    -----------
    P_mix : np.ndarray of shape (N, 18)
        Current blended prediction probabilities (summing to 1.0 per row).
    family_gates : dict mapping group_name -> log_probabilities of shape (N, K_group)
        Pre-computed log-probabilities from the auxiliary family classifiers.
    max_strength : float
        Maximum interpolation weight for the gate (0.0 to 1.0). If <= 0, gates are bypassed.
    margin_thr : float
        Confidence margin threshold above which the gate does not intervene.

    Returns:
    --------
    P_out : np.ndarray of shape (N, 18)
        Calibrated probabilities with sister-class confusions resolved.
    """
    # Early exit: If gates are disabled or weight is zero, return immediately
    if not family_gates or float(max_strength) <= 0.0:
        return P_mix

    P_out = P_mix.astype(np.float64).copy()

    for fam_name, gate_lp in family_gates.items():
        # Identify the column indices belonging to this family/specialist group
        cols = SPECIALIST_GROUPS.get(fam_name, FAMILIES.get(fam_name))
        if cols is None:
            continue
        if gate_lp is None or gate_lp.shape != (len(P_out), len(cols)):
            continue

        # Determine which samples are eligible for this gate:
        # Cross-family triads only apply when top-1 prediction is in the triad.
        # Single-family gates apply across all rows where that family has mass.
        if fam_name in ("frontier_and_fable", "kimi_gpt55_qwen", "glm_kimi_triad"):
            row_mask = np.isin(P_out.argmax(axis=1), cols)
            if not row_mask.any():
                continue
        elif fam_name == "gemini_37_38_pair":
            row_mask = np.isin(P_out.argmax(axis=1), FAMILIES.get("gemini", [5, 6, 7, 8]))
            if not row_mask.any():
                continue
        else:
            row_mask = np.ones(len(P_out), dtype=bool)

        # Hard-sister groups (e.g. Gemini 3.7 vs 3.8, GLM) use wider margin threshold
        eff_margin_thr = 0.65 if fam_name in ("gemini_37_38_pair", "glm", "glm_kimi_triad") else margin_thr
        eff_max_strength = (
            max(float(max_strength), 0.35) if fam_name in ("gemini_37_38_pair", "glm", "glm_kimi_triad") else float(max_strength)
        )

        # Step 1: Extract sub-probabilities and preserve total marginal mass
        sub_p = np.clip(P_out[np.ix_(row_mask, cols)], 1e-12, 1.0)
        tot_mass = sub_p.sum(axis=1, keepdims=True)
        base_rel_p = sub_p / tot_mass  # Relative distribution within the family (sums to 1.0)

        # Step 2: Compute margin between top-1 and top-2 candidates within family
        sp = np.sort(base_rel_p, axis=1)
        margin = sp[:, -1] - sp[:, -2]

        # Step 3: Compute dynamic intervention strength (higher when margin is small)
        strength = np.clip((eff_margin_thr - margin) / max(eff_margin_thr, 1e-6), 0.0, 1.0)[:, None] * eff_max_strength

        # Step 4: Blend base probabilities with gate probabilities in log-space
        base_lp = np.log(np.clip(base_rel_p, 1e-12, 1.0))
        gate_p = softmax(gate_lp[row_mask], axis=1)
        gate_lp_norm = np.log(np.clip(gate_p, 1e-12, 1.0))
        mixed_rel_p = softmax((1.0 - strength) * base_lp + strength * gate_lp_norm, axis=1)

        # Step 5: Restore original total family mass, leaving external classes unaffected
        P_out[np.ix_(row_mask, cols)] = tot_mass * mixed_rel_p

    # Final safety re-normalization to guarantee row sums == 1.0
    return P_out / P_out.sum(axis=1, keepdims=True).clip(min=1e-12)


# =============================================================================
# 2. PROMPT-CLUSTER UNIQUENESS REPULSION (CAVITY MEAN-FIELD)
# =============================================================================


def apply_cluster_uniqueness_repulsion(
    P_mix: np.ndarray,
    groups: np.ndarray | None,
    strength: float = 0.35,
    iters: int = 3,
) -> np.ndarray:
    """
    Enforces soft at-most-once author uniqueness within test prompt clusters.

    Domain Insight:
    ---------------
    In LLM benchmarking, multiple completions are generated by presenting the
    EXACT SAME PROMPT to different LLM models. Hence, when two or more test passages
    share an identical prompt (forming a 'prompt cluster'), competition rules imply
    that each completion came from a DIFFERENT model (at most 1 completion per model).

    Mathematical Formulation (Cavity Mean-Field):
    ---------------------------------------------
    For each text i in cluster g, the 'cavity field' for class k is the probability
    that no other text j in cluster g (j != i) was generated by class k:
        log P(other != k) = sum_{j in g, j != i} log(1 - q_{j, k})

    We update the unnormalized log-posterior:
        L_{i, k} = log(P0_{i, k}) + strength * sum_{j != i} log(1 - q_{j, k})
    where P0 is the original prediction prior, and q is the current cluster belief.
    Softmax is applied over k, iterating 3 times to reach a self-consistent equilibrium.

    Parameters:
    -----------
    P_mix : np.ndarray of shape (N, 18)
        Input prediction probabilities.
    groups : np.ndarray of shape (N,)
        Cluster IDs assigned to each test sample (e.g. from TF-IDF prompt clustering).
    strength : float
        Repulsion strength factor (typically 0.35).
    iters : int
        Number of mean-field coordinate ascent iterations (typically 3).

    Returns:
    --------
    q : np.ndarray of shape (N, 18)
        Repulsed probabilities penalizing multiple identical predictions in one prompt cluster.
    """
    N = len(P_mix)
    # Require at least 18 rows and valid cluster assignments
    if N < 18 or groups is None or len(groups) != N or strength <= 0.0:
        return P_mix

    # Map cluster identifiers to dense integer labels 1, 2, ...
    _, inv_g = np.unique(np.asarray(groups, dtype=str), return_inverse=True)
    safe_g = inv_g + 1
    sz = np.bincount(safe_g)[safe_g]

    # Only operate on multi-sample clusters between 2 and 18 texts (size <= 18 since there are 18 classes)
    valid_g = np.unique(safe_g[(sz >= 2) & (sz <= 18)])
    if len(valid_g) == 0:
        return P_mix

    log_P0 = np.log(np.clip(P_mix.astype(np.float64), 1e-12, 1.0))
    q = P_mix.astype(np.float64).copy()

    # Mean-field relaxation loop
    for _ in range(iters):
        L = log_P0.copy()
        for g in valid_g:
            m = safe_g == g
            q_g = q[m]  # Shape: (cluster_size, 18)

            # lo[j, k] = log(1 - q_{j, k}): log-probability that sample j did NOT pick class k
            lo = np.log(np.clip(1.0 - q_g, 1e-4, 1.0))

            # Cavity sum: sum of lo over all samples in cluster minus sample i's own term
            # Vectorized identity: sum_{j} lo[j] - lo[i] equals sum_{j != i} lo[j]
            cavity_penalty = lo.sum(axis=0, keepdims=True) - lo
            L[m] += float(strength) * cavity_penalty

        # Re-normalize across the 18 classes for next iteration
        q = softmax(L, axis=1)

    return q


# =============================================================================
# 3. TWO-TIER CONVEX ENSEMBLE BLENDING
# =============================================================================


def blend_main_and_complementary(
    LOGIT_MAIN: np.ndarray, COMP_LIST: list[tuple[float, np.ndarray]], main_ratio: float = 0.85, comp_ratio: float = 0.15
) -> tuple[np.ndarray, np.ndarray]:
    """
    Blends the Main Stack and Complementary Stack in probability space.

    Architecture:
    -------------
    - Main Stack (85% weight): High-capacity deep readers (Qwen3-14B Part 2,
      in-house champion LoRA, etc.) providing top semantic representations.
    - Complementary Stack (15% weight): Fast, diverse, non-neural or shallow readers
      (Style SVC 4-view, Lexical 5-view, DeBERTa, 8B patches) providing orthographic
      and stylistic diversity that regularizes neural hallucinations.

    Formula:
    --------
        R_main = softmax(LOGIT_MAIN)
        P_comp = sum(w_i * P_i) / sum(w_i)
        P_mix  = main_ratio * R_main + comp_ratio * P_comp

    Parameters:
    -----------
    LOGIT_MAIN : np.ndarray of shape (N, 18)
        Raw logit sum from the primary deep models.
    COMP_LIST : list of tuples (weight, probability_array)
        List of complementary models and their assigned ensemble weights.
    main_ratio : float
        Convex weight for Main Stack (default 0.85).
    comp_ratio : float
        Convex weight for Complementary Stack (default 0.15).

    Returns:
    --------
    P_mix : np.ndarray of shape (N, 18)
        Blended ensemble probabilities.
    R_main : np.ndarray of shape (N, 18)
        Main stack probabilities alone (used for routing/gating decisions).
    """
    R_main = softmax(LOGIT_MAIN, axis=1)
    if len(COMP_LIST) > 0:
        comp_weight_total = sum(w for w, _ in COMP_LIST)
        P_comp = sum(w * P_c for w, P_c in COMP_LIST) / max(1e-8, comp_weight_total)
        P_mix = main_ratio * R_main + comp_ratio * P_comp
    else:
        P_mix = R_main
    return P_mix, R_main


# =============================================================================
# 4. NEURAL SPECIALIST GATES (CLAUDE HEAD & FAMILY LORAS)
# =============================================================================


def apply_claude_specialist_gate(
    P_mix: np.ndarray,
    R_main: np.ndarray,
    claude_spec_logits: np.ndarray | None,
    claude_classes: list | None = None,
    labels: list = LABELS,
    claude_cols: list = FAMILIES["claude"],
    gate_weight: float = 0.40,
    weight: float | None = None,
) -> np.ndarray:
    """
    Refines passages predicted in the Claude family using a specialist head
    while strictly conserving total Claude family probability mass.
    """
    if claude_spec_logits is None:
        return P_mix
    if weight is not None:
        gate_weight = weight
    if float(gate_weight) <= 0.0:
        return P_mix

    P_out = P_mix.copy()
    P_claude_head = softmax(claude_spec_logits, axis=-1)

    if claude_classes is not None:
        src_c = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in claude_classes]
        if all(labels[c] in src_c for c in claude_cols):
            c_idx = [src_c.index(labels[c]) for c in claude_cols]
            P_claude_head = P_claude_head[:, c_idx]

    fam_mass = P_out[:, claude_cols].sum(axis=1)
    claude_mask = np.isin(R_main.argmax(axis=1), claude_cols) | (fam_mass >= 0.10)
    if claude_mask.any():
        for idx in np.flatnonzero(claude_mask):
            orig_mass = float(P_out[idx, claude_cols].sum())
            if orig_mass > 1e-6:
                gw_eff = float(gate_weight) * min(orig_mass / 0.50, 1.0)
                main_norm = P_out[idx, claude_cols] / orig_mass
                fused_fam = (1.0 - gw_eff) * main_norm + gw_eff * P_claude_head[idx]
                P_out[idx, claude_cols] = fused_fam * orig_mass

    return P_out


def apply_family_specialist_lora_gate(
    P_mix: np.ndarray,
    R_main: np.ndarray,
    spec_logits: np.ndarray | None,
    spec_classes: list | None = None,
    labels: list = LABELS,
    family_cols: list = FAMILIES["gemini"],
    gate_weight: float = 0.40,
) -> np.ndarray:
    """
    Refines intra-family predictions using fine-tuned Qwen3-14B Family-Specialist LoRAs
    while strictly preserving 100% of total marginal family probability mass.
    """
    if spec_logits is None or len(spec_logits) != len(P_mix) or float(gate_weight) <= 0.0:
        return P_mix

    P_out = P_mix.astype(np.float64).copy()
    spec_arr = np.asarray(spec_logits, dtype=np.float64)
    if spec_arr.ndim != 2 or spec_arr.shape[1] != len(family_cols):
        return P_mix

    P_spec = softmax(spec_arr, axis=-1)

    if spec_classes is not None and len(spec_classes) == len(family_cols):
        src_c = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in spec_classes]
        if all(labels[c] in src_c for c in family_cols):
            c_idx = [src_c.index(labels[c]) for c in family_cols]
            P_spec = P_spec[:, c_idx]

    fam_mass = P_out[:, family_cols].sum(axis=1)
    has_signal = np.ptp(spec_arr, axis=1) > 1e-6
    fam_mask = (np.isin(R_main.argmax(axis=1), family_cols) | (fam_mass >= 0.10)) & has_signal

    if fam_mask.any():
        for idx in np.flatnonzero(fam_mask):
            orig_mass = float(P_out[idx, family_cols].sum())
            if orig_mass > 1e-6:
                gw_eff = float(gate_weight) * min(orig_mass / 0.50, 1.0)
                main_norm = P_out[idx, family_cols] / orig_mass
                fused_fam = (1.0 - gw_eff) * main_norm + gw_eff * P_spec[idx]
                P_out[idx, family_cols] = fused_fam * orig_mass

    return P_out


# =============================================================================
# 5. ORTHOGRAPHIC RULES & HANDWRITING SHIELD
# =============================================================================


def build_orthographic_penalty_matrix(
    curly_quote_log_penalty: float = -1.00,  # Rule 1: Claude & GLM never emit curly quotes
    multi_em_dash_log_penalty: float = 0.0,  # Rule 2: Multiple prose em-dashes (rare in Gemini)
    straight_contr_log_penalty: float = 0.0,  # Rule 3: Straight contractions
    curly_contr_log_penalty: float = 0.0,  # Rule 4: Curly contractions
) -> np.ndarray:
    """
    Builds the (4, 18) log-penalty matrix W_pen mapping discrete orthographic
    rule indicators to the 18 canonical model classes.

    Empirical Foundations:
    ----------------------
    - Rule 1 (Curly Quotes): In the competition training corpus, Claude (0 / 1049)
      and GLM (0 / 300) NEVER produce typographic curly quotes (‘, ’, “, ”) in standard prose.
      They consistently output straight quotes (', "). If a passage contains curly quotes,
      the probability of it being Claude or GLM is astronomically low.
      Log penalty of -1.0 reduces their odds by a factor of e^(-1) ≈ 0.368.
    - Rule 2 (Multiple Em-Dashes): Gemini models rarely use two or more prose em-dashes (—).
    - Rule 3 (Straight Contractions): Distinctive signature for gpt-5.6-sol vs sisters.
    - Rule 4 (Curly Contractions): Reinforces rejection of Claude & GLM.
    """
    W_pen = np.zeros((4, K), dtype=np.float64)
    claude_and_glm_idx = FAMILIES["claude"] + FAMILIES["glm"]
    gemini_idx = FAMILIES["gemini"]
    sol_idx = LAB2I["gpt-5.6-sol"]

    # Rule 1: Any curly quote -> penalize Claude (5 models) and GLM (2 models)
    W_pen[0, claude_and_glm_idx] = float(curly_quote_log_penalty)
    # Rule 2: >= 2 prose em-dashes -> penalize Gemini (4 models)
    W_pen[1, gemini_idx] = float(multi_em_dash_log_penalty)
    # Rule 3: >= 2 prose straight contractions (with 0 curly) -> penalize gpt-5.6-sol
    W_pen[2, sol_idx] = float(straight_contr_log_penalty)
    # Rule 4: >= 1 prose curly contraction -> reinforce penalty on Claude & GLM
    W_pen[3, claude_and_glm_idx] = float(curly_contr_log_penalty)

    return W_pen


def _extract_rules_fallback(texts: list[str]) -> np.ndarray:
    """
    Fallback discrete rules extractor using regex when lexical_engine is unavailable.
    Detects curly quotes, em-dashes, and straight/curly prose contractions.
    """
    re_code = re.compile(r"```[\s\S]*?```|`[^`\n]+`")
    re_str = re.compile(r"[A-Za-z]'[A-Za-z]")
    re_cur = re.compile(r"[A-Za-z]\u2019[A-Za-z]")
    rules = np.zeros((len(texts), 4), dtype=np.float32)

    for i, raw_t in enumerate(texts):
        t = str(raw_t)
        prose = re_code.sub(" ", t)  # Strip inline and block code before checking prose
        raw_curly = t.count("\u2018") + t.count("\u2019") + t.count("\u201c") + t.count("\u201d")
        rules[i, 0] = 1.0 if raw_curly > 0 else 0.0
        rules[i, 1] = 1.0 if prose.count("\u2014") >= 2 else 0.0
        rules[i, 2] = 1.0 if (len(re_str.findall(prose)) >= 2 and (t.count("\u2018") + t.count("\u2019")) == 0) else 0.0
        rules[i, 3] = 1.0 if len(re_cur.findall(prose)) >= 1 else 0.0

    return rules


def apply_unified_handwriting_shield(
    P_mix: np.ndarray,
    test_texts: list[str],
    log_p_handwriting: np.ndarray | None = None,
    rule_flags: np.ndarray | None = None,
    curly_quote_multiplier: float = 0.36787944117144233,  # exp(-1.0)
    multi_em_dash_log_penalty: float = 0.0,
    straight_contr_log_penalty: float = 0.0,
    curly_contr_log_penalty: float = 0.0,
    continuous_hw_weight: float = 0.0,
) -> np.ndarray:
    """
    Single unified point of application for ALL handwriting and orthographic rules:
    1. Computes vectorized discrete log-penalties via matrix product: rule_flags @ W_pen
    2. Adds continuous 26-feature handwriting Gaussian log-likelihoods
    3. Re-normalizes rows to the probability simplex via softmax.
    """
    if rule_flags is None:
        if "extract_handwriting_and_orthographic_features" in globals():
            _, rule_flags = extract_handwriting_and_orthographic_features(test_texts)
        else:
            rule_flags = _extract_rules_fallback(test_texts)

    curly_log_pen = float(np.log(max(curly_quote_multiplier, 1e-12)))
    W_pen = build_orthographic_penalty_matrix(
        curly_quote_log_penalty=curly_log_pen,
        multi_em_dash_log_penalty=multi_em_dash_log_penalty,
        straight_contr_log_penalty=straight_contr_log_penalty,
        curly_contr_log_penalty=curly_contr_log_penalty,
    )

    log_P = np.log(np.clip(P_mix.astype(np.float64), 1e-12, 1.0))

    # 1. Discrete Bayesian rule subtraction (vectorized matrix multiply)
    log_P += rule_flags.astype(np.float64) @ W_pen

    # 2. Continuous 26-feature handwriting log-likelihood contribution
    if log_p_handwriting is not None and continuous_hw_weight > 0.0:
        log_P += float(continuous_hw_weight) * log_p_handwriting.astype(np.float64)

    return softmax(log_P, axis=1)


def apply_curly_quote_shield(P_mix: np.ndarray, test_texts: list[str], multiplier: float = 0.36787944117144233) -> np.ndarray:
    """Backward-compatible wrapper invoking the unified handwriting shield."""
    return apply_unified_handwriting_shield(
        P_mix=P_mix,
        test_texts=test_texts,
        log_p_handwriting=None,
        rule_flags=None,
        curly_quote_multiplier=multiplier,
        multi_em_dash_log_penalty=0.0,
        straight_contr_log_penalty=0.0,
        curly_contr_log_penalty=0.0,
        continuous_hw_weight=0.0,
    )


# =============================================================================
# 6. SINKHORN-KNOPP COLUMN MASS BALANCING
# =============================================================================


def sinkhorn_balancing(
    P: np.ndarray,
    iters: int = 100,
    sinkhorn_strength: float = 0.65,
    target_mix: float = 0.35,
    train_prior: np.ndarray | None = None,
    allow_pure: bool = False,
) -> np.ndarray:
    """
    Soft Multiplicative Sinkhorn-Knopp Column Marginal Balancing.
    =============================================================
    Equalizes global class marginals using Soft Sinkhorn shrinkage.
    Pure Sinkhorn (1.0 / 1.0) is disallowed by default as it destroys test performance
    by hard-forcing uniform marginals on batches with natural class variance.

    Parameters:
    -----------
    P : np.ndarray of shape (N, 18)
        Probability matrix to balance.
    iters : int
        Number of alternating row/column scaling steps (default 100).
    sinkhorn_strength : float
        Soft shrinkage interpolation between original P and balanced Q (default 0.65).
        Capped at 0.75 unless allow_pure=True.
    target_mix : float
        Convex blend between predicted marginals and prior (default 0.35).
        Capped at 0.50 unless allow_pure=True.
    train_prior : np.ndarray of shape (18,), optional
        Empirical training class frequencies (used when target_mix < 1.0).
    allow_pure : bool, optional
        Whether to allow 100% hard pure Sinkhorn (default False).

    Returns:
    --------
    Balanced probability matrix Q of shape (N, 18).
    """
    if len(P) < 18 or sinkhorn_strength <= 0.0:
        return P

    P_work = np.clip(P.astype(np.float64), 1e-12, 1.0)
    K_cols = P_work.shape[1]

    # Target column marginal
    if train_prior is not None:
        p_train = np.clip(train_prior.astype(np.float64), 1e-6, None)
        p_train /= p_train.sum()
    else:
        p_train = np.full(K_cols, 1.0 / K_cols, dtype=np.float64)

    eff_mix = target_mix if allow_pure else min(target_mix, 0.50)
    if eff_mix < 1.0:
        pred_marg = np.clip(P_work.mean(axis=0), 1e-6, None)
        pred_marg /= pred_marg.sum()
        rho = float(np.clip(eff_mix, 0.0, 1.0))
        target_col = (1.0 - rho) * pred_marg + rho * p_train
        target_col = np.clip(target_col, 1e-6, None)
        target_col /= target_col.sum()
    else:
        target_col = p_train

    # Alternating row and column projections
    Q = P_work.copy()
    for _ in range(iters):
        # Row normalization: sum across classes = 1.0
        Q /= Q.sum(axis=1, keepdims=True).clip(min=1e-12)
        # Column scaling: sum across samples for class k = N * target_col[k]
        Q *= (len(Q) * target_col[None, :]) / Q.sum(axis=0, keepdims=True).clip(min=1e-12)

    # Final row normalization
    Q /= Q.sum(axis=1, keepdims=True).clip(min=1e-12)

    if allow_pure and sinkhorn_strength >= 1.0:
        return Q

    # Soft logarithmic mixture between original P and balanced Q (pure Sinkhorn disallowed)
    eff_strength = float(np.clip(sinkhorn_strength, 0.0, 0.75 if not allow_pure else 1.0))
    if eff_strength <= 0.0:
        return P_work

    log_P = (1.0 - eff_strength) * np.log(np.clip(P_work, 1e-12, None)) + eff_strength * np.log(np.clip(Q, 1e-12, None))
    return softmax(log_P, axis=1)


def apply_prior_and_sinkhorn(
    logits: np.ndarray,
    temperature: float = 1.5,
    iters: int = 80,
    sinkhorn_strength: float = 0.85,
    target_mix: float = 0.15,
) -> np.ndarray:
    """
    Applies per-sample standardization, iterative column-wise logit prior centering (20 steps),
    and Sinkhorn marginal probability balancing to ensure unbiased class priors.
    """
    if len(logits) < 18 or sinkhorn_strength <= 0.0:
        return softmax(logits, axis=1)

    row_mean = logits.mean(axis=1, keepdims=True)
    row_std = np.clip(logits.std(axis=1, keepdims=True), 1e-4, None)
    norm_logits = (logits - row_mean) / row_std

    adj_logits = norm_logits / max(temperature, 1e-3)
    num_classes = logits.shape[1]
    for _ in range(20):
        probs = softmax(adj_logits, axis=1)
        col_mean = np.clip(probs.mean(axis=0, keepdims=True), 1e-6, None)
        adj_logits = adj_logits - 0.5 * np.log(col_mean * num_classes)

    return sinkhorn_balancing(
        softmax(adj_logits, axis=1),
        iters=iters,
        sinkhorn_strength=sinkhorn_strength,
        target_mix=target_mix,
    )


# =============================================================================
# 7. TYPOBAYES POSTPROCESSOR (HIERARCHICAL BETA-BERNOULLI SHRINKAGE)
# =============================================================================


def apply_typobayes_postprocess(
    P_or_logP: np.ndarray,
    train_texts: list[str],
    y_train: np.ndarray,
    test_texts: list[str],
    labels: list = LABELS,
    kappa: float = 24.0,
    evidence_scale: float = 1.2,
    evidence_cap: float = 0.30,
) -> np.ndarray:
    """
    Applies Hierarchical Beta-Bernoulli 37-Marker TypoBayes postprocessing.

    Concept:
    --------
    Extracts 37 orthographic, punctuation, and structural markdown markers
    (e.g., ' — ', '`', '###', 'However,', '\n- ') across training texts and models.
    Uses Beta-Bernoulli empirical Bayes with prior strength kappa to calculate
    smoothed class-conditional emission rates, deriving log-likelihood ratio evidence.
    Applies conservative confidence gating so high-confidence predictions are not overridden.
    """
    if len(train_texts) == 0 or len(test_texts) == 0 or len(y_train) == 0:
        return P_or_logP

    y_tr = np.array([LAB2I[y] if isinstance(y, str) else int(y) for y in y_train], dtype=int)
    marks = [
        " \u2014 ",
        "\u2014",
        "\u2013",
        "\u2019",
        "\u2018",
        "\u201c",
        "\u201d",
        "\u00ab",
        "\u00bb",
        '"',
        ";",
        ":",
        "(",
        ")",
        "[",
        "]",
        "**",
        "`",
        "\u2026",
        "\u2022",
        "\n",
        "\n\n",
        "e.g.",
        "i.e.",
        "However,",
        "Overall,",
        "In conclusion",
        "Therefore,",
        "Moreover,",
        "\t",
        " - ",
        " = ",
        "\n- ",
        "\n1.",
        "###",
        "```",
        "> ",
    ]
    K_cls = len(labels)

    def _extract_marker_row(t: str) -> list[float]:
        # Mutually exclusive decoupling for overlapping substrings:
        # 1. ' — ' vs unspaced '—'
        has_spaced_em = " \u2014 " in t
        t_no_spaced = t.replace(" \u2014 ", "")
        has_unspaced_em = "\u2014" in t_no_spaced

        # 2. '\n\n', '\n- ', '\n1.' vs standalone '\n'
        has_double_nl = "\n\n" in t
        has_bullet = "\n- " in t
        has_num_list = "\n1." in t
        t_no_compound_nl = t.replace("\n\n", "").replace("\n- ", "").replace("\n1.", "")
        has_single_nl = "\n" in t_no_compound_nl

        row = []
        for m in marks:
            if m == " \u2014 ":
                row.append(float(has_spaced_em))
            elif m == "\u2014":
                row.append(float(has_unspaced_em))
            elif m == "\n\n":
                row.append(float(has_double_nl))
            elif m == "\n- ":
                row.append(float(has_bullet))
            elif m == "\n1.":
                row.append(float(has_num_list))
            elif m == "\n":
                row.append(float(has_single_nl))
            else:
                row.append(float(m in t))
        return row

    Xtr = np.array([_extract_marker_row(t) for t in train_texts], dtype=np.float32)
    Xte = np.array([_extract_marker_row(t) for t in test_texts], dtype=np.float32)
    freq = np.bincount(y_tr, minlength=K_cls).astype(np.float64)
    hits = np.stack([Xtr[y_tr == k].sum(axis=0) for k in range(K_cls)], axis=0).astype(np.float64)

    # Hierarchical Beta-Bernoulli smoothing: class rates shrink toward global rate
    global_rate = (Xtr.sum(axis=0) + 2.0) / (len(Xtr) + 4.0)
    rate = (hits + kappa * global_rate[None, :]) / (freq[:, None] + kappa)
    rate = np.clip(rate, 0.01, 0.99)
    global_rate = np.clip(global_rate, 0.01, 0.99)

    log_rate = np.log(rate)
    log_not_rate = np.log1p(-rate)
    base_log_rate = np.log(global_rate)
    base_log_not_rate = np.log1p(-global_rate)

    # Weight markers by discriminative power across classes
    signal = np.sqrt(np.mean((rate - global_rate[None, :]) ** 2, axis=0))
    med_signal = float(np.median(signal[signal > 0])) if np.any(signal > 0) else 1.0
    fw = signal / max(med_signal, 1e-6)
    fw = np.clip(fw, 0.25, 2.0)

    # Matrix form: weighted Bernoulli log-likelihood ratio for each class
    evidence = (Xte * fw[None, :]) @ log_rate.T + ((1.0 - Xte) * fw[None, :]) @ log_not_rate.T
    evidence -= (((Xte * fw[None, :]) @ base_log_rate) + (((1.0 - Xte) * fw[None, :]) @ base_log_not_rate))[:, None]
    evidence -= evidence.mean(axis=1, keepdims=True)
    evidence = np.clip(evidence_scale * evidence / max(float(fw.sum()), 1e-6), -evidence_cap, evidence_cap)

    # Convert inputs to probabilities and log-probabilities
    if np.all(P_or_logP <= 0.0) and not np.all(P_or_logP == 0.0):
        raw_logP = P_or_logP.copy()
        probs = softmax(raw_logP, axis=1)
    else:
        probs = P_or_logP.copy()
        raw_logP = np.log(np.clip(probs, 1e-12, 1.0))

    # Gating: only apply evidence when model is uncertain, text is long, and markers are present
    sorted_p = np.sort(probs, axis=1)
    margin = sorted_p[:, -1] - sorted_p[:, -2]
    confidence_gate = np.clip((0.60 - margin) / 0.60, 0.0, 1.0)
    word_count = np.array([len(re.findall(r"\b\w+\b", str(t))) for t in test_texts], dtype=np.float32)
    length_gate = np.clip(np.sqrt(np.maximum(word_count, 1.0) / 80.0), 0.40, 1.0)
    active_markers = Xte.sum(axis=1)
    coverage_gate = np.clip(np.sqrt((active_markers + 1.0) / 4.0), 0.45, 1.0)
    gate = confidence_gate * length_gate * coverage_gate

    safe_logits = raw_logP + gate[:, None] * evidence
    return softmax(safe_logits, axis=1)


# =============================================================================
# 7.5. UNIFIED END-TO-END INFERENCE POSTPROCESSING PIPELINE
# =============================================================================


def run_full_inference_pipeline(
    P_mix: np.ndarray,
    texts: list[str],
    post_cfg: dict,
    R_main: np.ndarray | None = None,
    claude_spec_logits: np.ndarray | None = None,
    claude_classes: list | None = None,
    log_p_handwriting: np.ndarray | None = None,
    rule_flags: np.ndarray | None = None,
    train_texts: list[str] | None = None,
    y_train: np.ndarray | None = None,
    train_prior: np.ndarray | None = None,
) -> np.ndarray:
    """
    Unified end-to-end postprocessing pipeline executing all downstream stages:
    1. Claude Specialist Gate (Vectorized, mass-conserving)
    2. Continuous Handwriting & Orthographic Shield (Bayesian log-penalty + continuous density)
    3. Sinkhorn Mass Balancing (Iterative row/column equalization)
    4. TypoBayes 37-Marker Hierarchical Beta-Bernoulli Postprocessing

    Guarantees 100% mathematical parity between test inference and unseen benchmark evaluation.
    """
    if len(P_mix) == 0:
        return P_mix

    P_curr = P_mix.copy()
    R_ref = R_main if R_main is not None else P_curr

    # 1. Claude Specialist Gate
    gate_weight = float(post_cfg.get("claude_specialist_weight", 0.15))
    if claude_spec_logits is not None:
        P_curr = apply_claude_specialist_gate(
            P_mix=P_curr,
            R_main=R_ref,
            claude_spec_logits=claude_spec_logits,
            claude_classes=claude_classes,
            labels=LABELS,
            claude_cols=FAMILIES["claude"],
            gate_weight=gate_weight,
        )

    # 2. Unified Handwriting & Orthographic Shield
    default_curly_multiplier = float(np.exp(-1.0))
    multiplier = float(post_cfg.get("curly_quote_multiplier", default_curly_multiplier))
    multi_em_dash_log_pen = float(post_cfg.get("multi_em_dash_log_penalty", -0.45))
    straight_contr_log_pen = float(post_cfg.get("straight_contr_log_penalty", -0.60))
    curly_contr_log_pen = float(post_cfg.get("curly_contr_log_penalty", -0.40))
    continuous_hw_weight = float(post_cfg.get("continuous_hw_weight", 0.04))

    P_curr = apply_unified_handwriting_shield(
        P_mix=P_curr,
        test_texts=texts,
        log_p_handwriting=log_p_handwriting,
        rule_flags=rule_flags,
        curly_quote_multiplier=multiplier,
        multi_em_dash_log_penalty=multi_em_dash_log_pen,
        straight_contr_log_penalty=straight_contr_log_pen,
        curly_contr_log_penalty=curly_contr_log_pen,
        continuous_hw_weight=continuous_hw_weight,
    )

    # 3. Sinkhorn Mass Balancing
    sinkhorn_iters = int(post_cfg.get("sinkhorn_iterations", 100))
    sinkhorn_strength = float(post_cfg.get("sinkhorn_strength", 0.65))
    sinkhorn_target_mix = float(post_cfg.get("sinkhorn_target_mix", 0.35))

    P_curr = sinkhorn_balancing(
        P_curr,
        iters=sinkhorn_iters,
        sinkhorn_strength=sinkhorn_strength,
        target_mix=sinkhorn_target_mix,
        train_prior=train_prior,
    )

    # 4. TypoBayes 37-Marker Postprocessing
    if post_cfg.get("enable_typobayes", True) and train_texts and y_train is not None and len(train_texts) > 0:
        P_curr = apply_typobayes_postprocess(
            P_curr,
            train_texts=train_texts,
            y_train=y_train,
            test_texts=texts,
            labels=LABELS,
            kappa=float(post_cfg.get("typobayes_kappa", 24.0)),
            evidence_scale=float(post_cfg.get("typobayes_evidence_scale", 1.2)),
            evidence_cap=float(post_cfg.get("typobayes_evidence_cap", 0.30)),
        )

    return P_curr


# =============================================================================
# 8. FINAL SUBMISSION EXPORT
# =============================================================================


def generate_submission(P_final: np.ndarray, test_ids: pd.Series, working_dir: Path) -> tuple[pd.DataFrame, Path, Path]:
    """
    Generates submission.parquet and submission.csv with exact Kaggle competition schema:
    - id: test sample identifier string
    - model: predicted model class name string (argmax over calibrated probabilities)
    """
    preds = [LABELS[idx] for idx in P_final.argmax(axis=1)]
    df_sub = pd.DataFrame({"id": test_ids, "model": preds})
    sub_parquet_path = working_dir / "submission.parquet"
    sub_csv_path = working_dir / "submission.csv"

    df_sub.to_parquet(sub_parquet_path, index=False)
    df_sub.to_csv(sub_csv_path, index=False)
    return df_sub, sub_parquet_path, sub_csv_path


In [ ]:
# === MODULE: src/namethatllm/ablation.py ===
"""
Zero-Leakage Unseen Holdout Benchmark & Forward Stepwise Ablation Engine
========================================================================
Constructs a strictly unseen evaluation dataset from mounted parquet/CSV corpora
by subtracting all passages present in `train.csv` and `pool_v4.parquet`
(the training sets of the pre-trained Stage-1/Stage-2 checkpoints), quarantines
those holdout passages from `lexical_engine.py` and `trainer.py`, and evaluates:
1. Standalone (Solo) Unseen Accuracy of each model.
2. Forward Stepwise Cumulative Accuracy (1 model -> +2nd model -> ... -> Full Ensemble).
"""

import os
import gc
import hashlib
import warnings
from itertools import combinations
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, Set
import numpy as np
import pandas as pd
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import GroupKFold, KFold


def _ablation_sinkhorn(P: np.ndarray, iters: int = 200, eps: float = 1e-12, **kwargs) -> np.ndarray:
    """Fast vectorized Sinkhorn mass balancing for balanced multiclass prediction."""
    if len(P) < 18:
        return P
    P = np.clip(P.astype(np.float64), eps, 1.0)
    target_col = 1.0 / P.shape[1]
    for _ in range(iters):
        P /= P.sum(axis=1, keepdims=True).clip(min=eps)
        P /= (P.sum(axis=0, keepdims=True) / (target_col * P.shape[0])).clip(min=eps)
    return P / P.sum(axis=1, keepdims=True).clip(min=eps)

def _fast_sinkhorn_bacc(lp_mat: np.ndarray, y_true: np.ndarray, iters: int = 40) -> float:
    """Computes balanced accuracy after fast Sinkhorn mass equalization."""
    P = softmax(lp_mat, axis=1)
    if len(P) >= 18:
        P = _ablation_sinkhorn(P, iters=iters)
    return float(balanced_accuracy_score(y_true, P.argmax(axis=1)))

def _text_hash(text: str) -> str:
    """Normalized SHA1 hash of whitespace-collapsed text to guarantee zero exact/near-exact leakage."""
    norm = " ".join(str(text).strip().split())
    return hashlib.sha1(norm.encode("utf-8", errors="ignore")).hexdigest()

def build_unseen_holdout_dataset(
    input_dir: Path,
    train_path: Optional[Path] = None,
    samples_per_class: int = 100,
    seed: int = 42,
) -> Tuple[pd.DataFrame, Set[str]]:
    """
    Builds a balanced 18-class unseen holdout dataset (`df_unseen`) by:
    1. Hashing all texts in `train.csv` and `pool_v4.parquet` (and `pool_v1.parquet`) as `contaminated_hashes`.
    2. Scanning `nlm_matched_clean.parquet`, `nlm_pool_v5.parquet`, and `openrouter-harvest`
       for passages whose text hash is strictly NOT in `contaminated_hashes`.
    3. Sampling up to `samples_per_class` passages per class (e.g., 18 * 100 = 1,800 unseen passages)
       and returning `(df_unseen, holdout_hashes)` so `lexical_engine.py` and `trainer.py`
       exclude `holdout_hashes` from training!
    """
    input_dir = Path(input_dir) if input_dir else Path(".")

    # 0. Check for verified clean benchmark 1800 rows first
    bench_patterns = [
        "clean_benchmark_1800_rows.parquet",
        "*clean_benchmark*.parquet",
    ]
    for pat in bench_patterns:
        for bench_p in input_dir.rglob(pat):
            try:
                df_bench = pd.read_parquet(bench_p)
                if "text" in df_bench.columns and "model" in df_bench.columns:
                    df_bench = df_bench.dropna(subset=["text", "model"]).copy()
                    df_bench = df_bench[df_bench["model"].isin(LABELS)]
                    if len(df_bench) >= 180:
                        needs_clustering = False
                        if "prompt_id" not in df_bench.columns and "prompt_canon" in df_bench.columns:
                            df_bench["prompt_id"] = df_bench["prompt_canon"]

                        if "prompt_id" not in df_bench.columns:
                            needs_clustering = True
                        else:
                            p_col = df_bench["prompt_id"]
                            is_int = pd.api.types.is_integer_dtype(p_col) or (
                                p_col.dropna().apply(lambda x: isinstance(x, (int, np.integer))).all()
                                if len(p_col) > 0 else False
                            )
                            n_unique = p_col.nunique()
                            if (not is_int) or (n_unique > 0.5 * len(df_bench)):
                                needs_clustering = True

                        if needs_clustering:
                            try:
                                tfidf = TfidfVectorizer(stop_words="english", max_features=10000, sublinear_tf=True)
                                X_tfidf = tfidf.fit_transform(df_bench["text"].fillna("").tolist())
                                D = 1.0 - (X_tfidf @ X_tfidf.T).toarray().clip(0.0, 1.0)
                                np.fill_diagonal(D, 0.0)
                                cond_d = squareform(D, checks=False)
                                clusters = fcluster(linkage(cond_d, method="average"), t=0.85, criterion="distance")
                            except Exception:
                                clusters = np.arange(1, len(df_bench) + 1, dtype=int)
                            df_bench["cluster_id"] = clusters
                            df_bench["prompt_id"] = clusters
                        else:
                            if "cluster_id" not in df_bench.columns:
                                df_bench["cluster_id"] = df_bench["prompt_id"]

                        df_bench["_hash"] = [_text_hash(t) for t in df_bench["text"]]
                        holdout_hashes = set(df_bench["_hash"])
                        print(f"[Unseen Benchmark] Loaded verified clean holdout benchmark: {len(df_bench)} rows across {len(set(df_bench['model']))} classes from {bench_p}")
                        return df_bench, holdout_hashes
            except Exception as e:
                print(f"[Unseen Benchmark Warning] Failed to load {bench_p}: {e}")

    contaminated_hashes: Set[str] = set()

    # 1. Mark all train.csv rows as contaminated (seen by all Stage-2 LoRAs)
    if train_path and Path(train_path).exists():
        try:
            df_tr = pd.read_csv(train_path, usecols=["text"])
            contaminated_hashes.update(_text_hash(t) for t in df_tr["text"].fillna(""))
        except Exception:
            try:
                df_tr = pd.read_csv(train_path)
                if "text" in df_tr.columns:
                    contaminated_hashes.update(_text_hash(t) for t in df_tr["text"].fillna(""))
            except Exception:
                pass

    # 2. Mark all training pools as contaminated:
    # - pool_v4.parquet, pool_v1.parquet (seen by Part-2 LoRAs, DeBERTa, inhouse)
    # - nlm_pool_v5.parquet, nlm_matched_clean.parquet, nlm_pool_v2.parquet (seen by v5 and ts LoRAs)
    for pat in [
        "pool_v4.parquet",
        "pool_v1.parquet",
        "nlm_pool_v5.parquet",
        "nlm_matched_clean.parquet",
        "nlm_pool_v2.parquet",
    ]:
        for p in input_dir.rglob(pat):
            try:
                df_p = pd.read_parquet(p, columns=["text"])
                contaminated_hashes.update(_text_hash(t) for t in df_p["text"].fillna(""))
            except Exception:
                pass

    # 3. Collect strictly unseen candidate passages strictly from disjoint corpora (e.g. openrouter-harvest)
    candidate_dfs = []
    seen_paths = set()
    for pat in [
        "*openrouter*.parquet",
        "*openrouter*.csv",
        "*harvest*.parquet",
        "*harvest*.csv",
    ]:
        for p in input_dir.rglob(pat):
            if p in seen_paths:
                continue
            seen_paths.add(p)
            try:
                if p.suffix == ".parquet":
                    df_c = pd.read_parquet(p)
                elif p.suffix == ".csv":
                    df_c = pd.read_csv(p)
                else:
                    continue

                if "text" in df_c.columns and "model" in df_c.columns:
                    keep_cols = [c for c in ["id", "prompt_id", "text", "model"] if c in df_c.columns]
                    df_c = df_c[keep_cols].dropna(subset=["text", "model"]).copy()
                    df_c = df_c[df_c["model"].isin(LABELS)]
                    df_c["_hash"] = [_text_hash(t) for t in df_c["text"]]
                    df_c = df_c[~df_c["_hash"].isin(contaminated_hashes)]
                    if len(df_c) > 0:
                        candidate_dfs.append(df_c)
            except Exception:
                pass

    # Fallback to other non-training parquet/csv files if no openrouter files are discovered
    if not candidate_dfs:
        for pat in ["*.parquet", "*.csv"]:
            for p in input_dir.rglob(pat):
                p_name = p.name.lower()
                if p in seen_paths or any(x in p_name for x in ["train", "test", "submission", "pool", "matched"]):
                    continue
                seen_paths.add(p)
                try:
                    df_c = pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_csv(p)
                    if "text" in df_c.columns and "model" in df_c.columns:
                        keep_cols = [c for c in ["id", "prompt_id", "text", "model"] if c in df_c.columns]
                        df_c = df_c[keep_cols].dropna(subset=["text", "model"]).copy()
                        df_c = df_c[df_c["model"].isin(LABELS)]
                        df_c["_hash"] = [_text_hash(t) for t in df_c["text"]]
                        df_c = df_c[~df_c["_hash"].isin(contaminated_hashes)]
                        if len(df_c) > 0:
                            candidate_dfs.append(df_c)
                except Exception:
                    pass

    if not candidate_dfs:
        return pd.DataFrame(columns=["id", "prompt_id", "text", "model", "_hash"]), set()

    df_pool_unseen = pd.concat(candidate_dfs, ignore_index=True).drop_duplicates(subset=["_hash"])

    # Stratified balanced sampling across all 18 classes
    rng = np.random.RandomState(seed)
    sampled_parts = []

    # Prompt-grouped sampling: if prompt_id exists with multi-model prompts, sample by prompt_id first
    if "prompt_id" in df_pool_unseen.columns:
        prompt_counts = df_pool_unseen.groupby("prompt_id")["model"].nunique()
        multi_prompts = prompt_counts[prompt_counts > 1].index.tolist()
        rng.shuffle(multi_prompts)

        class_counts = {m: 0 for m in LABELS}
        chosen_indices = []
        for pid in multi_prompts:
            sub_p = df_pool_unseen[df_pool_unseen["prompt_id"] == pid]
            sub_m_counts = sub_p["model"].value_counts().to_dict()
            if all(class_counts.get(m, 0) + c <= samples_per_class for m, c in sub_m_counts.items()):
                chosen_indices.extend(sub_p.index.tolist())
                for m, c in sub_m_counts.items():
                    class_counts[m] = class_counts.get(m, 0) + c

        if chosen_indices:
            sampled_parts.append(df_pool_unseen.loc[chosen_indices])
            rem_pool = df_pool_unseen.drop(index=chosen_indices)
        else:
            rem_pool = df_pool_unseen
    else:
        rem_pool = df_pool_unseen
        class_counts = {m: 0 for m in LABELS}

    # Fill remaining class quotas up to samples_per_class
    for m in LABELS:
        curr_cnt = class_counts.get(m, 0)
        needed = samples_per_class - curr_cnt
        if needed > 0:
            sub = rem_pool[rem_pool["model"] == m]
            if len(sub) > 0:
                n_take = min(needed, len(sub))
                idx = rng.choice(sub.index.values, size=n_take, replace=False)
                sampled_parts.append(sub.loc[idx])

    if not sampled_parts:
        return pd.DataFrame(columns=["id", "prompt_id", "text", "model", "_hash"]), set()

    df_unseen = pd.concat(sampled_parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    if "id" not in df_unseen.columns:
        df_unseen["id"] = [f"unseen_{i}" for i in range(len(df_unseen))]

    # Compute valid stopword TF-IDF average-linkage clusters (t=0.85) so prompt_id is positive integer 1..G
    if len(df_unseen) >= 2:
        try:
            tfidf = TfidfVectorizer(stop_words="english", max_features=10000, sublinear_tf=True)
            X_tfidf = tfidf.fit_transform(df_unseen["text"].fillna("").tolist())
            D = 1.0 - (X_tfidf @ X_tfidf.T).toarray().clip(0.0, 1.0)
            np.fill_diagonal(D, 0.0)
            cond_d = squareform(D, checks=False)
            clusters = fcluster(linkage(cond_d, method="average"), t=0.85, criterion="distance")
        except Exception:
            clusters = np.arange(1, len(df_unseen) + 1, dtype=int)
    else:
        clusters = np.ones(len(df_unseen), dtype=int)

    df_unseen["cluster_id"] = clusters
    df_unseen["prompt_id"] = clusters

    holdout_hashes = set(df_unseen["_hash"].tolist())
    print(
        f"[Unseen Benchmark] Built strictly unseen holdout set: {len(df_unseen)} rows "
        f"across {df_unseen['model'].nunique()} classes (0% overlap with train.csv & pool_v4.parquet)."
    )
    return df_unseen, holdout_hashes

class UnseenStepwiseAuditor:
    """
    Evaluates every model on the strictly unseen holdout dataset (`df_unseen`),
    reporting:
    1. Table 1 — Standalone (Solo) Unseen Balanced Accuracy (sorted descending).
    2. Table 2 — Greedy Forward Stepwise Selection (optimal combinations, stopping when lift < 0.0).
    3. Table 3 — Exhaustive All-2^M Subset Search & Leave-One-Out (LOO) Attribution.
    """
    def __init__(self, y_unseen: np.ndarray, groups_unseen: np.ndarray, N_test: int):
        self.y_unseen = np.asarray(y_unseen, dtype=int) if len(y_unseen) > 0 else np.array([], dtype=int)
        if groups_unseen is not None and len(groups_unseen) > 0:
            try:
                self.groups_unseen = np.asarray(groups_unseen, dtype=int)
            except (ValueError, TypeError):
                codes, _ = pd.factorize(groups_unseen)
                self.groups_unseen = (codes + 1).astype(int)
        else:
            self.groups_unseen = np.array([], dtype=int)
        self.N_test = int(N_test)
        self.models: List[Dict[str, Any]] = []

    def add_model(
        self,
        name: str,
        tier: str,
        weight: float,
        unseen_log_probs: np.ndarray,
        test_log_probs: np.ndarray,
    ):
        if len(self.y_unseen) > 0 and len(unseen_log_probs) != len(self.y_unseen):
            return
        u_lp = log_softmax(np.asarray(unseen_log_probs, dtype=np.float64), axis=1) if len(self.y_unseen) > 0 and len(unseen_log_probs) == len(self.y_unseen) else np.zeros((0, K), dtype=np.float64)
        t_lp = log_softmax(np.asarray(test_log_probs, dtype=np.float64), axis=1) if len(test_log_probs) == self.N_test else np.zeros((self.N_test, K), dtype=np.float64)
        solo_raw = float(balanced_accuracy_score(self.y_unseen, u_lp.argmax(axis=1))) if len(self.y_unseen) > 0 else 0.0
        solo_sink = _fast_sinkhorn_bacc(u_lp, self.y_unseen, iters=200) if len(self.y_unseen) >= 18 else solo_raw
        self.models.append({
            "name": name,
            "tier": tier,
            "weight": float(weight),
            "unseen_lp": u_lp,
            "test_lp": t_lp,
            "solo_unseen_bacc": solo_raw,
            "solo_unseen_sinkhorn_bacc": solo_sink,
        })

    def optimize_weights_and_blend(
        self,
        working_dir: Path,
        extra_test_log_probs: Optional[List[Tuple[str, float, np.ndarray]]] = None,
        optimize_on_unseen: bool = True,
        lock_prior_weights: bool = False,
        extra_unseen_log_probs: Optional[List[Tuple[str, float, np.ndarray]]] = None,
        **kwargs,
    ) -> Tuple[np.ndarray, Dict[str, float], pd.DataFrame]:
        if lock_prior_weights:
            optimize_on_unseen = False
        """
        1. Runs Greedy Forward Stepwise Selection + Coordinate Line Search on `df_unseen`
           to determine each model's optimal individual weight `w_star` (>= 0.0).
        2. Any model that fails to improve unseen balanced accuracy gets `w_star = 0.0` (PRUNED).
        3. Combines all active models + extra_test_log_probs (wpca_dp, spectral_bank) directly
           in flat log-probability space:
               LOGIT_FLAT = sum_m (w_star[m] * test_lp[m])
        """
        if len(self.models) == 0:
            self.P_mix_unseen = np.zeros((0, K), dtype=np.float64)
            return np.ones((self.N_test, K), dtype=np.float64) / float(K), {}, pd.DataFrame()

        extra_u_lp_base = np.zeros((len(self.y_unseen), K), dtype=np.float64)
        if extra_unseen_log_probs and len(self.y_unseen) > 0:
            for ex_name, w_ex, lp_ex in extra_unseen_log_probs:
                if len(lp_ex) == len(self.y_unseen) and w_ex > 0.0:
                    extra_u_lp_base += float(w_ex) * log_softmax(lp_ex, axis=1)

        if not optimize_on_unseen or len(self.y_unseen) < 36:
            weights_map = {m["name"]: m["weight"] for m in self.models}
            logit_flat = np.zeros((self.N_test, K), dtype=np.float64)
            logit_flat_u = extra_u_lp_base.copy() if len(self.y_unseen) > 0 else np.zeros((0, K), dtype=np.float64)
            for m in self.models:
                logit_flat += m["weight"] * m["test_lp"]
                if len(m["unseen_lp"]) == len(self.y_unseen):
                    logit_flat_u += m["weight"] * m["unseen_lp"]
            if extra_test_log_probs:
                for ex_name, w_ex, lp_ex in extra_test_log_probs:
                    if len(lp_ex) == self.N_test and w_ex > 0.0:
                        logit_flat += float(w_ex) * log_softmax(lp_ex, axis=1)
                        weights_map[ex_name] = float(w_ex)
            P_mix = softmax(logit_flat, axis=1)
            self.P_mix_unseen = softmax(logit_flat_u, axis=1) if len(self.y_unseen) > 0 else np.zeros((0, K), dtype=np.float64)
            df_summary = pd.DataFrame([
                {"model": m["name"], "tier": m["tier"], "prior_weight": m.get("prior_weight", m["weight"]), "learned_weight": m["weight"]}
                for m in self.models
            ])
            return P_mix, weights_map, df_summary

        def _composite_obj(lp_mat: np.ndarray, y_target: Optional[np.ndarray] = None) -> Tuple[float, float]:
            y_use = self.y_unseen if y_target is None else y_target
            P = softmax(lp_mat, axis=1)
            P_s = _ablation_sinkhorn(P, iters=60) if len(P) >= 18 else P
            bacc = float(balanced_accuracy_score(y_use, P_s.argmax(axis=1)))
            nll = float(-np.mean(np.log(np.clip(P_s[np.arange(len(y_use)), y_use], 1e-9, 1.0))))
            return bacc - 0.015 * nll, bacc

        M = len(self.models)
        order = sorted(range(M), key=lambda i: self.models[i]["solo_unseen_sinkhorn_bacc"], reverse=True)

        active_weights = np.zeros(M, dtype=np.float64)
        first_idx = order[0]
        active_weights[first_idx] = self.models[first_idx]["weight"]
        cum_u_lp = extra_u_lp_base + active_weights[first_idx] * self.models[first_idx]["unseen_lp"]
        best_obj, best_bacc = _composite_obj(cum_u_lp)

        remaining = list(order[1:])
        multipliers = [0.10, 0.20, 0.35, 0.50, 0.75, 1.00, 1.25]

        # At each step, test ALL remaining models across all weight multipliers
        # and add the (model, weight) pair that maximizes the composite objective
        while remaining:
            step_best_idx = None
            step_best_w = 0.0
            step_best_obj = best_obj
            step_best_bacc = best_bacc

            for idx in remaining:
                prior_w = self.models[idx]["weight"]
                u_lp = self.models[idx]["unseen_lp"]
                for mult in multipliers:
                    cand_w = prior_w * mult
                    trial_lp = cum_u_lp + cand_w * u_lp
                    obj_val, bacc_val = _composite_obj(trial_lp)
                    if obj_val > step_best_obj + 1e-6:
                        step_best_obj = obj_val
                        step_best_bacc = bacc_val
                        step_best_idx = idx
                        step_best_w = cand_w

            if step_best_idx is not None and step_best_w > 0.0:
                active_weights[step_best_idx] = step_best_w
                cum_u_lp = cum_u_lp + step_best_w * self.models[step_best_idx]["unseen_lp"]
                best_obj = step_best_obj
                best_bacc = step_best_bacc
                remaining.remove(step_best_idx)
            else:
                break

        weights_map: Dict[str, float] = {}
        logit_flat = np.zeros((self.N_test, K), dtype=np.float64)

        # Safeguard: Apply minimum weight floors for models with validated solo accuracy (>= 0.38 for main, >= 0.38 or valid specialist for comp)
        # to prevent holdout set leakage (e.g. from pool training data) from zeroing out genuine ensemble models:
        for idx, m in enumerate(self.models):
            solo_acc = m.get("solo_unseen_sinkhorn_bacc", m.get("solo_unseen_bacc", 0.0))
            prior_w = m.get("prior_weight", m["weight"])
            tier = str(m.get("tier", "main")).lower()
            if "main" in tier and solo_acc >= 0.38:
                min_floor = 0.75 * prior_w
                active_weights[idx] = max(active_weights[idx], min_floor)
            elif "comp" in tier and (solo_acc >= 0.38 or (solo_acc >= 0.28 and "spec" in m["name"].lower())):
                min_floor = 0.50 * prior_w
                active_weights[idx] = max(active_weights[idx], min_floor)

        logit_flat_u = extra_u_lp_base.copy() if len(self.y_unseen) > 0 else np.zeros((0, K), dtype=np.float64)
        for idx, m in enumerate(self.models):
            m["prior_weight"] = m.get("prior_weight", m["weight"])
            m["learned_weight"] = float(active_weights[idx])
            m["weight"] = float(active_weights[idx])
            weights_map[m["name"]] = float(active_weights[idx])
            if active_weights[idx] > 0.0:
                logit_flat += active_weights[idx] * m["test_lp"]
                if len(m["unseen_lp"]) == len(self.y_unseen):
                    logit_flat_u += active_weights[idx] * m["unseen_lp"]

        if extra_test_log_probs:
            for ex_name, w_ex, lp_ex in extra_test_log_probs:
                if len(lp_ex) == self.N_test and w_ex > 0.0:
                    logit_flat += float(w_ex) * log_softmax(lp_ex, axis=1)
                    weights_map[ex_name] = float(w_ex)

        in_sample_bacc = _fast_sinkhorn_bacc(logit_flat_u, self.y_unseen, iters=200) if len(self.y_unseen) >= 18 else best_bacc

        # 5-fold GroupKFold OOF Unseen BAcc calculation
        oof_bacc = in_sample_bacc
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=UserWarning)
                groups = self.groups_unseen if len(self.groups_unseen) == len(self.y_unseen) else np.arange(len(self.y_unseen))
                n_groups = len(np.unique(groups))
                if n_groups >= 2:
                    n_splits = min(5, n_groups)
                    gkf = GroupKFold(n_splits=n_splits)
                    splits = list(gkf.split(self.y_unseen, self.y_unseen, groups=groups))
                else:
                    kf = KFold(n_splits=5, shuffle=True, random_state=42)
                    splits = list(kf.split(self.y_unseen))

                oof_lp = np.zeros((len(self.y_unseen), K), dtype=np.float64)
                for tr_idx, val_idx in splits:
                    y_tr = self.y_unseen[tr_idx]
                    ex_tr = extra_u_lp_base[tr_idx]
                    ex_val = extra_u_lp_base[val_idx]

                    fold_order = sorted(
                        range(M),
                        key=lambda i: _fast_sinkhorn_bacc(self.models[i]["unseen_lp"][tr_idx], y_tr, iters=40) if len(tr_idx) >= 18 else 0.0,
                        reverse=True,
                    )
                    fold_w = np.zeros(M, dtype=np.float64)
                    f_first = fold_order[0]
                    fold_w[f_first] = self.models[f_first]["weight"]
                    f_cum_lp = ex_tr + fold_w[f_first] * self.models[f_first]["unseen_lp"][tr_idx]
                    f_obj, _ = _composite_obj(f_cum_lp, y_target=y_tr)

                    f_rem = list(fold_order[1:])
                    while f_rem:
                        s_best_idx = None
                        s_best_w = 0.0
                        s_best_obj = f_obj
                        for idx in f_rem:
                            prior_w = self.models[idx]["weight"]
                            u_lp_tr = self.models[idx]["unseen_lp"][tr_idx]
                            for mult in multipliers:
                                cand_w = prior_w * mult
                                t_lp = f_cum_lp + cand_w * u_lp_tr
                                obj_val, _ = _composite_obj(t_lp, y_target=y_tr)
                                if obj_val > s_best_obj + 1e-6:
                                    s_best_obj = obj_val
                                    s_best_idx = idx
                                    s_best_w = cand_w
                        if s_best_idx is not None and s_best_w > 0.0:
                            fold_w[s_best_idx] = s_best_w
                            f_cum_lp = f_cum_lp + s_best_w * self.models[s_best_idx]["unseen_lp"][tr_idx]
                            f_obj = s_best_obj
                            f_rem.remove(s_best_idx)
                        else:
                            break

                    for idx, m in enumerate(self.models):
                        solo_acc_tr = _fast_sinkhorn_bacc(m["unseen_lp"][tr_idx], y_tr, iters=40) if len(tr_idx) >= 18 else 0.0
                        prior_w = m.get("prior_weight", m["weight"])
                        tier = str(m.get("tier", "main")).lower()
                        if "main" in tier and solo_acc_tr >= 0.38:
                            fold_w[idx] = max(fold_w[idx], 0.75 * prior_w)
                        elif "comp" in tier and (solo_acc_tr >= 0.38 or (solo_acc_tr >= 0.28 and "spec" in m["name"].lower())):
                            fold_w[idx] = max(fold_w[idx], 0.50 * prior_w)

                    fold_val_lp = ex_val.copy()
                    for idx, m in enumerate(self.models):
                        if fold_w[idx] > 0.0:
                            fold_val_lp += fold_w[idx] * m["unseen_lp"][val_idx]
                    oof_lp[val_idx] = fold_val_lp

                oof_bacc = _fast_sinkhorn_bacc(oof_lp, self.y_unseen, iters=200)
        except Exception as e:
            print(f"[Ablation Warning] OOF GroupKFold computation failed ({e}), using in-sample BAcc fallback.")
            oof_bacc = in_sample_bacc

        print(f"[Ablation] In-Sample Unseen BAcc: {in_sample_bacc:.4f} | 5-Fold GroupKFold OOF Unseen BAcc: {oof_bacc:.4f}")

        P_mix = softmax(logit_flat, axis=1)
        self.P_mix_unseen = softmax(logit_flat_u, axis=1) if len(self.y_unseen) > 0 else np.zeros((0, K), dtype=np.float64)
        df_summary = pd.DataFrame([
            {"model": m["name"], "tier": m["tier"], "prior_weight": m.get("prior_weight", m["weight"]), "learned_weight": m["weight"]}
            for m in self.models
        ])
        return P_mix, weights_map, df_summary

    def evaluate_marginal_lift(
        self,
        candidate_unseen_lp: np.ndarray,
        weight: float = 1.0,
    ) -> float:
        """
        Multi-scale marginal lift search: tests adding `candidate_unseen_lp` across
        multipliers [0.05, 0.10, 0.15, 0.25, 0.35, 0.50, 0.75, 1.00] * weight
        and returns the maximum Delta BAcc achieved at any positive scale.
        """
        if len(self.y_unseen) == 0 or len(candidate_unseen_lp) != len(self.y_unseen) or len(self.models) == 0:
            return 0.0

        cum_unseen_lp = np.zeros((len(self.y_unseen), K), dtype=np.float64)
        for m in self.models:
            w_m = m["weight"] if m["weight"] > 0 else m.get("prior_weight", 0.15)
            cum_unseen_lp += w_m * m["unseen_lp"]

        cur_bacc = _fast_sinkhorn_bacc(cum_unseen_lp, self.y_unseen, iters=40)
        cand_lp = log_softmax(np.asarray(candidate_unseen_lp, dtype=np.float64), axis=1)

        best_lift = -1.0
        full_weight_lift = 0.0
        for mult in (0.05, 0.10, 0.15, 0.25, 0.35, 0.50, 0.75, 1.00):
            cand_w = float(weight) * mult
            new_unseen_lp = cum_unseen_lp + cand_w * cand_lp
            new_bacc = _fast_sinkhorn_bacc(new_unseen_lp, self.y_unseen, iters=40)
            lift = new_bacc - cur_bacc
            if mult == 1.00:
                full_weight_lift = lift
            if lift > best_lift:
                best_lift = lift

        if best_lift <= 0.0:
            return float(full_weight_lift)
        return float(best_lift)

    def print_and_save_report(self, working_dir: Path, final_test_preds: Optional[np.ndarray] = None) -> pd.DataFrame:
        if not self.models or len(self.y_unseen) == 0:
            print("[Ablation] No models or unseen holdout samples available; skipping report.")
            return pd.DataFrame()

        # Part 1: Standalone (Solo) Accuracy, sorted descending
        print("\n" + "=" * 105)
        print(f" PART 1: STANDALONE (SOLO) ACCURACY ON STRICTLY UNSEEN HOLDOUT ({len(self.y_unseen)} Disjoint Passages)")
        print("=" * 105)
        print(f"{'Model / Checkpoint':<28} | {'Tier':<6} | {'Unseen Raw BAcc':<16} | {'Unseen+Sinkhorn BAcc':<20} | {'Agrees w/ Final Test'}")
        print("-" * 105)
        sorted_solo = sorted(self.models, key=lambda m: m["solo_unseen_sinkhorn_bacc"], reverse=True)
        for m in sorted_solo:
            agree_str = "N/A"
            if final_test_preds is not None and len(final_test_preds) == self.N_test:
                pct = 100.0 * float((m["test_lp"].argmax(axis=1) == final_test_preds).mean())
                agree_str = f"{pct:5.1f}%"
            print(
                f"{m['name']:<28} | {m['tier']:<6} | "
                f"{m['solo_unseen_bacc']:<16.4f} | {m['solo_unseen_sinkhorn_bacc']:<20.4f} | {agree_str}"
            )

        # Part 2: Greedy Forward Stepwise Selection (starts with strongest solo model, tests all candidates at each step)
        print("\n" + "=" * 105)
        print(" PART 2: GREEDY FORWARD STEPWISE SELECTION ON STRICTLY UNSEEN HOLDOUT (Optimal Combinations)")
        print("=" * 105)
        print(f"{'Step':<5} | {'Added Model':<28} | {'Wt':<5} | {'Cumul Raw':<10} | {'Cumul+Sink':<11} | {'Marginal Lift':<13} | {'Test Rows Flipped'}")
        print("-" * 105)

        M = len(self.models)
        remaining = list(range(M))
        cum_unseen_lp = np.zeros((len(self.y_unseen), K), dtype=np.float64)
        cum_test_lp = np.zeros((self.N_test, K), dtype=np.float64)
        prev_bacc = 0.0
        prev_test_preds = None
        records = []

        step_idx = 1
        while remaining:
            best_cand_idx = None
            best_cand_bacc = -1.0
            best_cand_raw = 0.0

            if step_idx == 1:
                # Pick model with highest solo unseen sinkhorn bacc
                for idx in remaining:
                    cand = self.models[idx]
                    cand_w = cand["weight"] if cand["weight"] > 0 else cand.get("prior_weight", 0.15)
                    cand_lp = cand_w * cand["unseen_lp"]
                    cand_bacc = _fast_sinkhorn_bacc(cand_lp, self.y_unseen, iters=40)
                    if cand_bacc > best_cand_bacc:
                        best_cand_bacc = cand_bacc
                        best_cand_idx = idx
                        best_cand_raw = float(balanced_accuracy_score(self.y_unseen, cand_lp.argmax(axis=1)))
            else:
                # Test all remaining candidate models
                for idx in remaining:
                    cand = self.models[idx]
                    cand_w = cand["weight"] if cand["weight"] > 0 else cand.get("prior_weight", 0.15)
                    cand_lp = cum_unseen_lp + cand_w * cand["unseen_lp"]
                    cand_bacc = _fast_sinkhorn_bacc(cand_lp, self.y_unseen, iters=40)
                    if cand_bacc > best_cand_bacc:
                        best_cand_bacc = cand_bacc
                        best_cand_idx = idx
                        best_cand_raw = float(balanced_accuracy_score(self.y_unseen, cand_lp.argmax(axis=1)))

            if best_cand_idx is None:
                break

            lift = best_cand_bacc - prev_bacc if step_idx > 1 else 0.0
            if step_idx > 1 and lift < 0.0:
                print(f"[Greedy Forward Selection] Best remaining candidate '{self.models[best_cand_idx]['name']}' gives negative lift ({lift:+.4f}); stopping forward selection.")
                break

            m = self.models[best_cand_idx]
            remaining.remove(best_cand_idx)

            w = m["weight"] if m["weight"] > 0 else m.get("prior_weight", 0.15)
            cum_unseen_lp = cum_unseen_lp + w * m["unseen_lp"]
            cum_test_lp = cum_test_lp + w * m["test_lp"]

            lift_str = f"{lift:+.4f}" if step_idx > 1 else "BASELINE"
            cur_test_preds = _ablation_sinkhorn(softmax(cum_test_lp, axis=1), iters=40).argmax(axis=1) if self.N_test > 0 else np.array([])
            if prev_test_preds is None:
                flips = self.N_test
                flip_str = f"{flips} (Init)"
            else:
                flips = int((cur_test_preds != prev_test_preds).sum()) if len(cur_test_preds) == len(prev_test_preds) else 0
                flip_pct = 100.0 * flips / max(1, self.N_test)
                flip_str = f"{flips:4d} ({flip_pct:4.1f}%)"

            print(
                f"{step_idx:<5} | + {m['name']:<26} | {w:<5.2f} | "
                f"{best_cand_raw:<10.4f} | {best_cand_bacc:<11.4f} | {lift_str:<13} | {flip_str}"
            )
            prev_bacc = best_cand_bacc
            prev_test_preds = cur_test_preds
            records.append({
                "step": step_idx,
                "model": m["name"],
                "added_model": m["name"],
                "tier": m["tier"],
                "weight": w,
                "solo_unseen_bacc": m["solo_unseen_bacc"],
                "solo_unseen_sinkhorn_bacc": m["solo_unseen_sinkhorn_bacc"],
                "cumul_unseen_raw_bacc": best_cand_raw,
                "cumul_unseen_sinkhorn_bacc": best_cand_bacc,
                "marginal_lift": lift,
                "test_rows_flipped": flips,
            })
            step_idx += 1

        print("=" * 105 + "\n")

        # Part 3: Exhaustive All 2^M Subset Search & LOO Attribution (for M <= 12)
        if 2 <= M <= 12:
            print("=" * 105)
            print(f" PART 3: GLOBALLY OPTIMAL SUBSET & LEAVE-ONE-OUT (LOO) ATTRIBUTION (All {2**M - 1} Non-Empty Subsets)")
            print("=" * 105)

            best_subset = None
            best_subset_bacc = -1.0

            for r in range(1, M + 1):
                for sub in combinations(range(M), r):
                    sub_unseen_lp = sum((self.models[i]["weight"] if self.models[i]["weight"] > 0 else self.models[i].get("prior_weight", 0.15)) * self.models[i]["unseen_lp"] for i in sub)
                    bacc = _fast_sinkhorn_bacc(sub_unseen_lp, self.y_unseen, iters=30)
                    if bacc > best_subset_bacc:
                        best_subset_bacc = bacc
                        best_subset = sub

            opt_names = [self.models[i]["name"] for i in best_subset]
            print(f"Globally Optimal Ensemble BAcc: {best_subset_bacc:.4f} (Uses {len(best_subset)} / {M} models)")
            print(f"Optimal Set: {', '.join(opt_names)}")
            print("-" * 105)
            print(f"{'Model / Checkpoint':<28} | {'In Optimal?':<12} | {'Weight':<7} | {'Delta_LOO (BAcc Drop If Excluded)'}")
            print("-" * 105)

            opt_set = set(best_subset)
            for i, m in enumerate(self.models):
                in_opt = i in opt_set
                in_str = "YES" if in_opt else "NO (Pruned)"
                if in_opt:
                    if len(best_subset) == 1:
                        delta_loo = best_subset_bacc
                    else:
                        loo_sub = [j for j in best_subset if j != i]
                        loo_lp = sum((self.models[j]["weight"] if self.models[j]["weight"] > 0 else self.models[j].get("prior_weight", 0.15)) * self.models[j]["unseen_lp"] for j in loo_sub)
                        loo_bacc = _fast_sinkhorn_bacc(loo_lp, self.y_unseen, iters=30)
                        delta_loo = best_subset_bacc - loo_bacc
                    loo_str = f"-{delta_loo:.4f}" if delta_loo >= 0 else f"+{abs(delta_loo):.4f}"
                else:
                    with_m_sub = list(best_subset) + [i]
                    with_m_lp = sum((self.models[j]["weight"] if self.models[j]["weight"] > 0 else self.models[j].get("prior_weight", 0.15)) * self.models[j]["unseen_lp"] for j in with_m_sub)
                    with_m_bacc = _fast_sinkhorn_bacc(with_m_lp, self.y_unseen, iters=30)
                    delta_add = with_m_bacc - best_subset_bacc
                    loo_str = f"{delta_add:+.4f} (Degrades if added)"

                print(f"{m['name']:<28} | {in_str:<12} | {m['weight']:<7.2f} | {loo_str}")

                for r in records:
                    if r["model"] == m["name"]:
                        r["in_optimal_subset"] = in_opt
                        r["delta_loo"] = delta_loo if in_opt else 0.0

            print("=" * 105 + "\n")

        df_rep = pd.DataFrame(records)
        working_dir = Path(working_dir)
        working_dir.mkdir(parents=True, exist_ok=True)
        df_rep.to_csv(working_dir / "unseen_stepwise_ablation.csv", index=False)
        return df_rep


In [ ]:
# === MODULE: src/namethatllm/pipeline.py ===
"""
Top-Level Pipeline Orchestrator
===============================
Coordinates the complete end-to-end execution flow:
- Module 1: Dual-Output Lexical Engine (subprocess isolated or in-process)
- Module 2: Neural Trunk Extraction Loop (All 4 trunks and 6 checkpoints)
- Module 3: wPCA DP Posterior (ExactAuthorPosterior) + 384-Dim Spectral Bank
- Module 4: Two-Tier Simplex Blend + Vectorized Claude Gate + Sinkhorn Balancing
"""

import os
import gc
import sys
import json
import time
import subprocess
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import balanced_accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer


def verify_checkpoint_gate(
    Z_tr: np.ndarray,
    W_p: Optional[np.ndarray],
    b_p: Optional[np.ndarray],
    ckpt_labels: list,
    y_sub: np.ndarray,
    name: str,
    min_bacc: float = 0.50
) -> bool:
    """
    Verifies that a checkpoint's plain classification head achieves >= min_bacc
    on the prompt-heldout validation partition (train_sub) before adding its logits.
    Guarantees that corrupt or misaligned heads cannot degrade Main or Comp stack.
    """
    if W_p is None or b_p is None or len(Z_tr) == 0:
        print(f"[Gate] {name}: No plain head or empty Z_tr; skipping head logits.")
        return False
    if W_p.shape[1] != Z_tr.shape[1] or W_p.shape[0] != K:
        print(f"[Gate] {name}: Shape mismatch W_p={W_p.shape} vs Z_tr={Z_tr.shape}; skipping head logits.")
        return False
    logits_tr = align_cols(Z_tr @ W_p.T + b_p, ckpt_labels)
    preds_tr = logits_tr.argmax(axis=1)
    acc = float((preds_tr == y_sub).mean())
    passed = acc >= min_bacc
    print(f"[Gate] {name}: train_sub accuracy = {acc:.4f} (threshold={min_bacc:.2f}) -> {'PASSED' if passed else 'REJECTED'}")
    return passed

def auto_align_and_verify_head(
    Z_tr: np.ndarray,
    W_p: Optional[np.ndarray],
    b_p: Optional[np.ndarray],
    W_c: Optional[np.ndarray],
    b_c: Optional[np.ndarray],
    ckpt_labels: list,
    y_sub: np.ndarray,
    prompts_sub: np.ndarray,
    name: str,
    min_bacc: float = 0.50,
) -> Tuple[bool, Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray], Optional[np.ndarray], list, float]:
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import GroupKFold
    from sklearn.metrics import balanced_accuracy_score

    if len(Z_tr) == 0:
        return False, W_p, b_p, W_c, b_c, ckpt_labels, 0.0

    if W_p is not None and b_p is not None and W_p.shape == (K, Z_tr.shape[1]):
        best_order = ckpt_labels
        best_acc = -1.0
        for _, cand_order in [("canonical", ckpt_labels), ("alphabetical", sorted(LABELS))]:
            try:
                logits_tr = align_cols(Z_tr @ W_p.T + b_p, cand_order)
                acc = float(balanced_accuracy_score(y_sub, logits_tr.argmax(axis=1)))
                if acc > best_acc:
                    best_acc = acc
                    best_order = cand_order
            except Exception:
                continue
        if best_acc >= min_bacc:
            print(f"[Gate] {name}: Pre-trained head PASSED with BAcc = {best_acc:.4f} (threshold={min_bacc:.2f})")
            return True, W_p, b_p, W_c, b_c, best_order, best_acc
        print(f"[Gate] {name}: Pre-trained head BAcc = {best_acc:.4f} < {min_bacc:.2f}; probing 5-fold OOF LogReg readout...")

    if len(y_sub) >= 90 and len(np.unique(prompts_sub)) >= 5:
        scale = float(np.sqrt(Z_tr.shape[1])) / 8.0
        Z_scaled = Z_tr / scale
        gkf = GroupKFold(n_splits=5)
        oof_preds = np.zeros(len(y_sub), dtype=int)
        for tr_i, val_i in gkf.split(Z_scaled, y_sub, groups=prompts_sub):
            clf = LogisticRegression(C=0.20, max_iter=1000, random_state=42)
            clf.fit(Z_scaled[tr_i], y_sub[tr_i])
            oof_preds[val_i] = clf.predict(Z_scaled[val_i])
        oof_bacc = float(balanced_accuracy_score(y_sub, oof_preds))
        if oof_bacc >= min_bacc:
            clf_full = LogisticRegression(C=0.20, max_iter=1000, random_state=42)
            clf_full.fit(Z_scaled, y_sub)
            W_p_new = (clf_full.coef_ / scale).astype(np.float32)
            b_p_new = clf_full.intercept_.astype(np.float32)

            Z_c = Z_scaled.copy()
            for p in np.unique(prompts_sub):
                m = (prompts_sub == p)
                if m.sum() > 1:
                    Z_c[m] -= Z_c[m].mean(axis=0, keepdims=True)
            clf_c = LogisticRegression(C=0.12, max_iter=1000, random_state=42)
            clf_c.fit(Z_c, y_sub)
            W_c_new = (clf_c.coef_ / scale).astype(np.float32)
            b_c_new = clf_c.intercept_.astype(np.float32)
            fit_labels = [LABELS[c] for c in clf_full.classes_]
            print(f"[Gate] {name}: 5-Fold OOF Readout PASSED with BAcc = {oof_bacc:.4f}!")
            return True, W_p_new, b_p_new, W_c_new, b_c_new, fit_labels, oof_bacc

    print(f"[Gate] {name}: REJECTED (failed both pre-trained and 5-fold OOF gates).")
    return False, None, None, None, None, ckpt_labels, 0.0

def _combine_plain_and_twin_lp(
    lp_plain: np.ndarray,
    lp_twin: Optional[np.ndarray],
    w_plain: float,
    w_twin: float,
) -> np.ndarray:
    """
    Combines plain and prompt-centered twin log-probabilities proportionally:
    On rows where lp_twin is active (non-zero), blends with weights (w_plain, w_twin)
    and normalizes via log_softmax. On singleton rows where lp_twin is zero,
    scales by w_plain / (w_plain + w_twin) so downstream multiplication by (w_plain + w_twin)
    yields the exact w_plain * lp_plain contribution.
    """
    lp_p = log_softmax(lp_plain, axis=1)
    if lp_twin is None or w_twin <= 0.0 or len(lp_twin) != len(lp_plain):
        return lp_p
    w_tot = float(w_plain) + float(w_twin)
    if w_tot <= 1e-8:
        return lp_p
    has_twin = np.ptp(lp_twin, axis=1) > 1e-6
    out = np.empty_like(lp_p, dtype=np.float64)
    # Singletons: scale so w_tot * out == w_plain * lp_p
    out[~has_twin] = (float(w_plain) / w_tot) * lp_p[~has_twin]
    # Multi-sample clusters: exact weighted log-probability sum
    if has_twin.any():
        lp_t = log_softmax(lp_twin[has_twin], axis=1)
        out[has_twin] = (float(w_plain) * lp_p[has_twin] + float(w_twin) * lp_t) / w_tot
    return out.astype(np.float32)

def run_submission_pipeline(cfg: Optional[Dict[str, Any]] = None) -> Tuple[pd.DataFrame, Path]:
    """
    Executes the complete production or smoke-test submission pipeline.
    Returns (submission_dataframe, submission_parquet_path).
    """
    if cfg is None:
        cfg = {}

    start_time = time.time()
    is_smoke = os.environ.get("NTL_SMOKE") == "1"

    # 1. Environment & Paths
    if "working_dir" in cfg:
        working_dir = Path(cfg["working_dir"])
    else:
        working_dir = Path("/kaggle/working")
        if not working_dir.exists():
            working_dir = Path("local_data")
    working_dir.mkdir(parents=True, exist_ok=True)

    input_dir = Path(cfg.get("input_dir", "/kaggle/input"))
    if not input_dir.exists():
        input_dir = Path(".")

    if "train_path" in cfg and "test_path" in cfg:
        train_path = Path(cfg["train_path"])
        test_path = Path(cfg["test_path"])
        pool_file = Path(cfg["pool_file"]) if "pool_file" in cfg else None
        all_pool_files = [pool_file] if pool_file else resolve_all_pool_files(input_dir)
        if pool_file is None and all_pool_files:
            pool_file = all_pool_files[0]
    else:
        train_path, test_path, pool_file = resolve_dataset_paths(input_dir)
        all_pool_files = resolve_all_pool_files(input_dir)

    enable_ablation_probe = bool(cfg.get("enable_ablation_probe", True)) if cfg else True
    probe_gate_min_bacc = float(cfg.get("probe_gate_min_bacc", 0.40)) if cfg else 0.40

    df_test_raw = pd.read_csv(test_path) if test_path and Path(test_path).exists() else pd.DataFrame({"id": [], "text": []})
    test_ids = df_test_raw["id"] if "id" in df_test_raw.columns else pd.Series(range(len(df_test_raw)))

    # -------------------------------------------------------------------------
    # Decoupled Execution Mode Architecture:
    # "benchmark_only": Evaluates exclusively on clean 1,800 unseen samples (for fast experiments without burning GPU quota).
    # "submission_only": Evaluates exclusively on competition test.csv (for official Kaggle leaderboard scoring).
    # "dual": Evaluates both datasets cleanly and independently.
    # -------------------------------------------------------------------------
    execution_mode = cfg.get("execution_mode") if cfg else None
    if execution_mode is None:
        if bool(cfg.get("benchmark_mode", False)) if cfg else False:
            execution_mode = "benchmark_only"
        elif len(df_test_raw) <= 10 and enable_ablation_probe:
            execution_mode = "benchmark_only"
        elif enable_ablation_probe:
            execution_mode = "dual"
        else:
            execution_mode = "submission_only"

    run_benchmark = bool(cfg.get("run_benchmark", execution_mode in ("benchmark_only", "dual"))) if cfg else (execution_mode in ("benchmark_only", "dual"))
    run_submission = bool(cfg.get("run_submission", execution_mode in ("submission_only", "dual"))) if cfg else (execution_mode in ("submission_only", "dual"))

    # When competition test.csv has <= 10 dummy rows on Kaggle, auto-disable submission inference to safeguard GPU quota
    if len(df_test_raw) <= 10 and run_benchmark:
        run_submission = False

    test_texts = df_test_raw["text"].fillna("").astype(str).tolist() if (run_submission and "text" in df_test_raw.columns) else []
    N_test = len(test_texts)

    # Build Zero-Leakage Unseen Holdout Benchmark (N_unseen = 1800 rows default)
    df_unseen = pd.DataFrame()
    unseen_texts: List[str] = []
    N_unseen = 0
    y_unseen = np.array([], dtype=int)
    groups_unseen = np.array([], dtype=int)

    if run_benchmark and not is_smoke:
        df_unseen, holdout_hashes = build_unseen_holdout_dataset(input_dir, train_path, samples_per_class=int(cfg.get("unseen_samples_per_class", 100)))
        if len(df_unseen) > 0:
            unseen_parquet_path = working_dir / "unseen_holdout.parquet"
            df_unseen.to_parquet(unseen_parquet_path, index=False)
            unseen_texts = df_unseen["text"].fillna("").astype(str).tolist()
            N_unseen = len(unseen_texts)
            y_unseen = np.array([LAB2I[m] for m in df_unseen["model"].values], dtype=int) if "model" in df_unseen.columns else np.array([], dtype=int)
            if "cluster_id" in df_unseen.columns:
                groups_unseen = df_unseen["cluster_id"].values
            elif "prompt_id" in df_unseen.columns:
                groups_unseen = df_unseen["prompt_id"].values
            elif "prompt_canon" in df_unseen.columns:
                groups_unseen = (pd.factorize(df_unseen["prompt_canon"])[0] + 1).astype(int)
            elif "prompt_text" in df_unseen.columns:
                groups_unseen = (pd.factorize(df_unseen["prompt_text"])[0] + 1).astype(int)
            else:
                groups_unseen = np.ones(N_unseen, dtype=int)
            print(f"[Pipeline] Loaded clean benchmark dataset: {N_unseen} samples ({len(set(y_unseen))} classes).")
    else:
        if not run_benchmark:
            print("[Pipeline] Frozen Production Mode (run_benchmark=False): Skipping unseen holdout extraction to save GPU time.")

    print("=" * 70)
    print(f"[Pipeline] EXECUTION MODE: {execution_mode.upper()}")
    print(f"  - run_benchmark  : {run_benchmark} ({N_unseen} unseen samples)")
    print(f"  - run_submission : {run_submission} ({N_test} test rows)")
    print("=" * 70)

    auditor = UnseenStepwiseAuditor(y_unseen=y_unseen, groups_unseen=groups_unseen, N_test=N_test)

    # Discover fine-tuned checkpoints early for pre-flight validation
    ckpts = find_checkpoints(input_dir)
    print(f"[Pipeline] Discovered checkpoints ({len(ckpts)}): {list(ckpts.keys())}")

    if not is_smoke:
        verified_bases = validate_production_assets(
            input_dir=input_dir,
            train_path=train_path,
            test_path=test_path,
            pool_files=all_pool_files or ([pool_file] if pool_file else []),
            ckpts=ckpts,
            cfg=cfg,
        )
    else:
        verified_bases = {}

    # 2. MODULE 1: Subprocess Dual-Output Lexical Engine
    run_lex_script = working_dir / "run_lexical.py"
    if run_lex_script.exists() and not is_smoke:
        print("[Pipeline] Module 1: Launching subprocess run_lexical.py...")
        subprocess.run([sys.executable, str(run_lex_script)], check=True)

        def _load_optional_npy(p: Path, default=None) -> Optional[np.ndarray]:
            return np.load(p) if p.exists() else default

        log_p_svc_main = np.load(working_dir / "log_p_svc_main.npy")
        p_lex_comp = np.load(working_dir / "p_lex_comp.npy")
        D_tfidf = np.load(working_dir / "D_tfidf.npy")
        log_p_handwriting = _load_optional_npy(working_dir / "log_p_handwriting.npy")
        rule_flags_test = _load_optional_npy(working_dir / "rule_flags_test.npy")
        unseen_log_p_svc_main = _load_optional_npy(working_dir / "unseen_log_p_svc_main.npy", default=np.zeros((0, K), dtype=np.float32))
        unseen_p_lex_comp = _load_optional_npy(working_dir / "unseen_p_lex_comp.npy", default=np.zeros((0, K), dtype=np.float32))
        D_tfidf_u = _load_optional_npy(working_dir / "D_tfidf_unseen.npy")
        log_p_handwriting_unseen = _load_optional_npy(working_dir / "log_p_handwriting_unseen.npy")
        rule_flags_unseen = _load_optional_npy(working_dir / "rule_flags_unseen.npy")

        if (run_benchmark and unseen_log_p_svc_main.shape[0] != N_unseen) or (run_submission and log_p_svc_main.shape[0] != N_test):
            print(f"[Pipeline] Subprocess lexical cache shape mismatch (test: {log_p_svc_main.shape[0]} vs {N_test}, unseen: {unseen_log_p_svc_main.shape[0]} vs {N_unseen}). Falling back to in-process run_lexical_engine...")
            tok_paths = list(input_dir.rglob("vocab.json"))
            qwen_tok_dir = str(tok_paths[0].parent) if tok_paths else ""
            lex_out = run_lexical_engine(train_path, test_path, all_pool_files or pool_file, qwen_tok_dir, working_dir)
            log_p_svc_main = lex_out["log_p_svc_main"]
            p_lex_comp = lex_out["p_lex_comp"]
            D_tfidf = lex_out["D_tfidf"]
            log_p_handwriting = lex_out.get("log_p_handwriting")
            rule_flags_test = lex_out.get("rule_flags_test")
            unseen_log_p_svc_main = lex_out.get("unseen_log_p_svc_main", np.zeros((0, K), dtype=np.float32))
            unseen_p_lex_comp = lex_out.get("unseen_p_lex_comp", np.zeros((0, K), dtype=np.float32))
            D_tfidf_u = lex_out.get("D_tfidf_unseen")
            log_p_handwriting_unseen = lex_out.get("log_p_handwriting_unseen")
            rule_flags_unseen = lex_out.get("rule_flags_unseen")
    else:
        print("[Pipeline] Module 1: Executing run_lexical_engine directly...")
        tok_paths = list(input_dir.rglob("vocab.json"))
        qwen_tok_dir = str(tok_paths[0].parent) if tok_paths else ""
        lex_out = run_lexical_engine(train_path, test_path, all_pool_files or pool_file, qwen_tok_dir, working_dir)
        log_p_svc_main = lex_out["log_p_svc_main"]
        p_lex_comp = lex_out["p_lex_comp"]
        D_tfidf = lex_out["D_tfidf"]
        log_p_handwriting = lex_out.get("log_p_handwriting")
        rule_flags_test = lex_out.get("rule_flags_test")
        unseen_log_p_svc_main = lex_out.get("unseen_log_p_svc_main", np.zeros((0, K), dtype=np.float32))
        unseen_p_lex_comp = lex_out.get("unseen_p_lex_comp", np.zeros((0, K), dtype=np.float32))
        D_tfidf_u = lex_out.get("D_tfidf_unseen")
        log_p_handwriting_unseen = lex_out.get("log_p_handwriting_unseen")
        rule_flags_unseen = lex_out.get("rule_flags_unseen")

    print(f"[Pipeline] Module 1 complete: log_p_svc_main shape = {log_p_svc_main.shape}, unseen_log_p_svc_main shape = {unseen_log_p_svc_main.shape}")

    # Read configuration hierarchies from configs/production_ensemble.json
    main_cfg = cfg.get("main_stack", {})
    comp_cfg = cfg.get("complementary_stack", {})
    fusion_cfg = cfg.get("fusion", {})
    post_cfg = cfg.get("postprocess", {})
    model_weights = cfg.get("model_weights", {})

    svc_weight = model_weights.get("svc_4view", {}).get("plain_weight", main_cfg.get("linear_svc_weight", 1.00))
    main_members = main_cfg.get("members", {})
    comp_streams = comp_cfg.get("stream_weights", {})
    comp_members = comp_cfg.get("lora_members", {})
    lex_comp_weight = model_weights.get("lexical_5v", {}).get("plain_weight", comp_streams.get("lexical", 0.20))
    se_floors = comp_cfg.get("se_floors", {
        "qwen3_14b_p2": 0.10, "qwen25_14b_p2": 0.10, "deberta_p2": 0.10,
        "qwen3_14b_p1": 0.35, "instyle": 0.35, "qwen3_8b_claude": 0.35
    })
    prior_scales = comp_cfg.get("prior_scales", PRIOR_SCALES)
    dp_weights = comp_cfg.get("dp_weights", {
        "q3_14b": 0.38, "q25_14b": 0.24, "deberta": 0.20, "instyle": 0.10, "part1_14b": 0.08
    })
    twin_res = comp_cfg.get("twin_resolutions", {
        "t85": 0.50, "t72": 0.30, "t60": 0.20
    })
    w85 = float(twin_res.get("t85", 0.50))
    w72 = float(twin_res.get("t72", 0.30))
    w60 = float(twin_res.get("t60", 0.20))
    max_cl = post_cfg.get("max_cluster_size", 14)

    LOGIT_MAIN = log_p_svc_main * svc_weight if (run_submission and len(log_p_svc_main) == N_test) else np.zeros((N_test, K), dtype=np.float32)
    COMP_LIST: List[Tuple[float, np.ndarray]] = [
        (lex_comp_weight, p_lex_comp if (run_submission and len(p_lex_comp) == N_test) else np.zeros((N_test, K), dtype=np.float32))
    ]
    u_lp_svc = unseen_log_p_svc_main if (run_benchmark and N_unseen > 0 and len(unseen_log_p_svc_main) == N_unseen) else np.zeros((N_unseen, K), dtype=np.float32)
    te_lp_svc = log_p_svc_main if (run_submission and N_test > 0 and len(log_p_svc_main) == N_test) else np.zeros((N_test, K), dtype=np.float32)
    auditor.add_model(
        name="LinearSVC (4-View)",
        tier="Main",
        weight=svc_weight,
        unseen_log_probs=u_lp_svc,
        test_log_probs=te_lp_svc,
    )
    u_p_lex = unseen_p_lex_comp if (run_benchmark and N_unseen > 0 and len(unseen_p_lex_comp) == N_unseen) else (np.ones((N_unseen, K), dtype=np.float32) / float(K))
    te_p_lex = p_lex_comp if (run_submission and N_test > 0 and len(p_lex_comp) == N_test) else (np.ones((N_test, K), dtype=np.float32) / float(K))
    auditor.add_model(
        name="Lexical Comp (5-View)",
        tier="Comp",
        weight=lex_comp_weight,
        unseen_log_probs=np.log(np.clip(u_p_lex, 1e-9, 1.0)),
        test_log_probs=np.log(np.clip(te_p_lex, 1e-9, 1.0)),
    )
    claude_spec_logits = None
    claude_spec_logits_u = None
    claude_classes = None
    features_unseen_dict: Dict[str, np.ndarray] = {}

    def compute_multires_clusters(
        D_mat: np.ndarray,
        thresholds: Tuple[float, ...] = (0.85, 0.72, 0.65, 0.60),
        max_cluster_sz: int = 14
    ) -> Dict[float, np.ndarray]:
        n_samples = D_mat.shape[0] if (D_mat is not None and D_mat.ndim == 2) else 0
        if n_samples < 2:
            return {t: np.ones(max(n_samples, 1), dtype=int) for t in thresholds}
        cond_d = squareform(D_mat, checks=False)
        Z_link = linkage(cond_d, method="average")
        return {
            t: split_oversized_clusters_dist(D_mat, fcluster(Z_link, t=t, criterion="distance"), max_cluster_size=max_cluster_sz)
            for t in thresholds
        }

    class HybridDistanceTracker:
        """Symmetric multi-resolution distance and topic cluster tracker for test and unseen benchmark."""
        def __init__(self, D_init: Optional[np.ndarray], texts: Optional[List[str]] = None, max_cluster_sz: int = 14):
            self.max_cluster_size = max_cluster_sz
            if D_init is not None and D_init.ndim == 2 and D_init.shape[0] >= 2:
                self.D_tfidf = D_init.copy()
            elif texts is not None and len(texts) >= 2:
                Xt = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=100000, dtype=np.float32).fit_transform(texts)
                d = np.clip(1.0 - (Xt @ Xt.T).toarray(), 0.0, None)
                np.fill_diagonal(d, 0.0)
                self.D_tfidf = d
            elif texts is not None and len(texts) == 1:
                self.D_tfidf = np.zeros((1, 1), dtype=np.float32)
            else:
                self.D_tfidf = np.zeros((0, 0), dtype=np.float32)

            self.N = self.D_tfidf.shape[0]
            self.D_neural_sum = np.zeros((self.N, self.N), dtype=np.float64) if self.N > 0 else np.zeros((0, 0), dtype=np.float64)
            self.D_neural_weight_total = 0.0
            self.D_hybrid = self.D_tfidf.copy()
            self.groups = compute_multires_clusters(self.D_hybrid, max_cluster_sz=max_cluster_sz)

        def update(self, Z_mat: np.ndarray, ckpt_scale: float) -> None:
            if self.N < 2 or Z_mat is None or len(Z_mat) != self.N:
                return
            Z_c = Z_mat - Z_mat.mean(axis=0, keepdims=True)
            Z_norm = Z_c / np.linalg.norm(Z_c, axis=1, keepdims=True).clip(min=1e-6)
            D_cos = np.clip(1.0 - (Z_norm @ Z_norm.T), 0.0, None)
            np.fill_diagonal(D_cos, 0.0)
            w_d = float(ckpt_scale)
            self.D_neural_sum += w_d * D_cos
            self.D_neural_weight_total += w_d
            D_neural_avg = self.D_neural_sum / max(1e-6, self.D_neural_weight_total)
            self.D_hybrid = 0.55 * self.D_tfidf + 0.45 * D_neural_avg
            self.groups = compute_multires_clusters(self.D_hybrid, max_cluster_sz=self.max_cluster_size)

    def compute_multires_centered_blend(
        Z: np.ndarray,
        groups_map: Dict[float, np.ndarray],
        W_c: Optional[np.ndarray],
        b_c: Optional[np.ndarray],
        labels: list,
        weights: Tuple[float, float, float] = (0.50, 0.30, 0.20),
    ) -> np.ndarray:
        if W_c is None or b_c is None or len(Z) == 0:
            return np.zeros((len(Z), K), dtype=np.float32)
        w85, w72, w60 = weights
        lp85, _ = compute_centered_logits(Z, groups_map.get(0.85, np.ones(len(Z), dtype=int)), W_c, b_c, labels)
        lp72, _ = compute_centered_logits(Z, groups_map.get(0.72, np.ones(len(Z), dtype=int)), W_c, b_c, labels)
        lp60, _ = compute_centered_logits(Z, groups_map.get(0.60, np.ones(len(Z), dtype=int)), W_c, b_c, labels)
        return (w85 * lp85 + w72 * lp72 + w60 * lp60).astype(np.float32)

    tracker_test = HybridDistanceTracker(D_tfidf, texts=test_texts, max_cluster_sz=max_cl) if (run_submission and N_test > 0) else None
    tracker_unseen = HybridDistanceTracker(D_tfidf_u if ('D_tfidf_u' in locals() and D_tfidf_u is not None) else None, texts=unseen_texts, max_cluster_sz=max_cl) if (run_benchmark and N_unseen > 0) else None

    # Backward-compatible references (using active tracker)
    active_tracker = tracker_test if tracker_test is not None else tracker_unseen
    D_neural_sum = active_tracker.D_neural_sum if active_tracker else np.zeros((0, 0), dtype=np.float32)
    D_neural_weight_total = active_tracker.D_neural_weight_total if active_tracker else 0.0
    D_hybrid = active_tracker.D_hybrid if active_tracker else np.zeros((0, 0), dtype=np.float32)
    groups_085 = active_tracker.groups[0.85] if active_tracker else np.zeros(0, dtype=int)
    groups_072 = active_tracker.groups[0.72] if active_tracker else np.zeros(0, dtype=int)
    groups_065 = active_tracker.groups[0.65] if active_tracker else np.zeros(0, dtype=int)
    groups_060 = active_tracker.groups[0.60] if active_tracker else np.zeros(0, dtype=int)

    def update_hybrid_groups(Z_te_mat: Optional[np.ndarray], ckpt_scale: float, Z_u_mat: Optional[np.ndarray] = None):
        nonlocal D_hybrid, groups_085, groups_072, groups_065, groups_060, D_neural_weight_total
        if tracker_test is not None and Z_te_mat is not None and len(Z_te_mat) > 0:
            tracker_test.update(Z_te_mat, ckpt_scale)
            D_hybrid = tracker_test.D_hybrid
            D_neural_weight_total = tracker_test.D_neural_weight_total
            groups_085 = tracker_test.groups[0.85]
            groups_072 = tracker_test.groups[0.72]
            groups_065 = tracker_test.groups[0.65]
            groups_060 = tracker_test.groups[0.60]
        if tracker_unseen is not None and Z_u_mat is not None and len(Z_u_mat) > 0:
            tracker_unseen.update(Z_u_mat, ckpt_scale)
            if tracker_test is None:
                D_hybrid = tracker_unseen.D_hybrid
                D_neural_weight_total = tracker_unseen.D_neural_weight_total
                groups_085 = tracker_unseen.groups[0.85]
                groups_072 = tracker_unseen.groups[0.72]
                groups_065 = tracker_unseen.groups[0.65]
                groups_060 = tracker_unseen.groups[0.60]

    has_neural = len(ckpts) > 0 and not is_smoke
    spectral_blocks_tr: List[np.ndarray] = []
    spectral_blocks_te: List[np.ndarray] = []
    spectral_blocks_u: List[np.ndarray] = []
    dp_posteriors: List[Tuple[float, np.ndarray]] = []
    dp_posteriors_u: List[Tuple[float, np.ndarray]] = []
    comp_lora_posteriors: List[Tuple[float, np.ndarray]] = []
    P_spectral = None
    P_spectral_u = None

    df_train = pd.read_csv(train_path) if train_path and Path(train_path).exists() else None
    extract_train_wpca = comp_cfg.get("extract_train_wpca", False)
    train_sub_texts: List[str] = []
    y_sub: np.ndarray = np.array([], dtype=int)
    prompts_sub: np.ndarray = np.array([])
    N_tr_sub: int = 0

    if has_neural:
        import torch

        if extract_train_wpca and df_train is not None:
            rng = np.random.RandomState(42)
            u_p = df_train["prompt_id"].unique()
            sel_prompts = rng.choice(u_p, size=min(90, len(u_p)), replace=False)
            sub_idx = df_train[df_train["prompt_id"].isin(sel_prompts)].index.values
            train_sub = df_train.loc[sub_idx].reset_index(drop=True)
            train_sub_texts = train_sub["text"].fillna("").astype(str).tolist()
            y_sub = np.array([LAB2I[m] for m in train_sub["model"]])
            prompts_sub = train_sub["prompt_id"].values
            N_tr_sub = len(train_sub)
            print(f"[Pipeline] Selected {N_tr_sub} training rows ({len(sel_prompts)} prompts) for wPCA")

        # Discover pre-trained wPCA bundle
        wpca_bundle = None
        wpca_candidates = [
            working_dir / "wpca_bundle.safetensors",
            Path("local_data/statistical_heads/wpca_bundle.safetensors"),
            Path("/kaggle/input/ntl-statistical-heads/wpca_bundle.safetensors"),
        ]
        if Path("/kaggle/input").exists():
            for p in Path("/kaggle/input").rglob("wpca_bundle.safetensors"):
                if p not in wpca_candidates:
                    wpca_candidates.append(p)
        for cand in wpca_candidates:
            if cand.exists():
                try:
                    wpca_bundle = load_wpca_bundle(cand)
                    print(f"[Pipeline] Loaded pre-trained wPCA bundle from {cand}")
                    break
                except Exception as e:
                    print(f"[Pipeline] Note: Could not load candidate wPCA bundle {cand}: {e}")

        # Discover pre-trained Spectral Bank params
        spectral_params = None
        spectral_candidates = [
            working_dir / "spectral_heads.safetensors",
            Path("local_data/statistical_heads/spectral_heads.safetensors"),
            Path("/kaggle/input/ntl-statistical-heads/spectral_heads.safetensors"),
        ]
        if Path("/kaggle/input").exists():
            for p in Path("/kaggle/input").rglob("spectral_heads.safetensors"):
                if p not in spectral_candidates:
                    spectral_candidates.append(p)
        for cand in spectral_candidates:
            if cand.exists():
                try:
                    spectral_params = load_spectral_params(cand)
                    print(f"[Pipeline] Loaded pre-trained Spectral Bank params from {cand}")
                    break
                except Exception as e:
                    print(f"[Pipeline] Note: Could not load candidate Spectral Bank params {cand}: {e}")

        wpca_bundle_export: Dict[str, dict] = {}

        def _extract_trunk(trunk_obj, end_id=None):
            if run_submission and N_test > 0 and run_benchmark and N_unseen > 0:
                Z_all = trunk_obj.extract_exact(test_texts + unseen_texts, batch_size=16, end_id=end_id)
                Z_te = Z_all[:N_test]
                Z_u = Z_all[N_test : N_test + N_unseen]
            elif run_submission and N_test > 0:
                Z_te = trunk_obj.extract_exact(test_texts, batch_size=16, end_id=end_id)
                Z_u = np.zeros((0, Z_te.shape[1]), dtype=np.float32)
            elif run_benchmark and N_unseen > 0:
                Z_u = trunk_obj.extract_exact(unseen_texts, batch_size=16, end_id=end_id)
                Z_te = np.zeros((0, Z_u.shape[1]), dtype=np.float32)
            else:
                Z_te = np.zeros((0, 4096), dtype=np.float32)
                Z_u = np.zeros((0, 4096), dtype=np.float32)
            dim = Z_u.shape[1] if len(Z_u) > 0 else (Z_te.shape[1] if len(Z_te) > 0 else 4096)
            Z_tr = np.zeros((0, dim), dtype=np.float32)
            return Z_te, Z_u, Z_tr

        def apply_wpca_and_spectral(
            model_key: str,
            Z_te_mat: np.ndarray,
            Z_tr_mat: np.ndarray,
            Z_u_mat: Optional[np.ndarray],
            dp_w: float,
            prior_scale: float,
            d_target: int = 256,
            se_floor: float = 0.10,
        ) -> None:
            res = None
            if wpca_bundle is not None and model_key in wpca_bundle:
                res = wpca_bundle[model_key]
            elif len(Z_tr_mat) > 0:
                res = fit_shared_gaussian_lowram(Z_tr_mat, y_sub, prompts_sub, d_target=d_target, se_floor=se_floor)
                wpca_bundle_export[model_key] = res

            if res is not None:
                if run_submission and len(Z_te_mat) > 0 and tracker_test is not None:
                    U_te = transform_lowram(Z_te_mat, res)
                    lp_dp = scores_exact_dp(U_te, tracker_test.groups[0.65], res, tracker_test.D_hybrid)
                    dp_posteriors.append((dp_w, softmax(lp_dp, axis=1)))
                    S_te = project_spectral_block(Z_te_mat, res, prior_scale)
                    spectral_blocks_te.append(S_te)
                    del U_te

                if len(Z_tr_mat) > 0:
                    S_tr = project_spectral_block(Z_tr_mat, res, prior_scale)
                    spectral_blocks_tr.append(S_tr)

                if run_benchmark and Z_u_mat is not None and len(Z_u_mat) > 0 and tracker_unseen is not None:
                    U_u = transform_lowram(Z_u_mat, res)
                    lp_dp_u = scores_exact_dp(U_u, tracker_unseen.groups[0.65], res, tracker_unseen.D_hybrid)
                    dp_posteriors_u.append((dp_w, softmax(lp_dp_u, axis=1)))
                    S_u = project_spectral_block(Z_u_mat, res, prior_scale)
                    spectral_blocks_u.append(S_u)
                    del U_u
                del res

        # ==============================================================================
        # TRUNK 1: Qwen3-14B (40 layers, hidden=5120)
        # ==============================================================================
        base_q3_14b = verified_bases.get("qwen3_14b") or find_base_model(input_dir, "qwen3", 5120, 40)
        if base_q3_14b and ("qwen3_14b_p2" in ckpts or "qwen3_14b_p1" in ckpts or "qwen3_14b_ts" in ckpts or "qwen3_14b_v5" in ckpts or "qwen3_14b_inhouse" in ckpts):
            print(f"[Pipeline] Loading Trunk 1: Qwen3-14B from {base_q3_14b}...")
            trunk1 = UniversalLoRATrunkLoader(base_q3_14b, keep_layers=27, layers=[2, 4, 6, 9, 13, 17, 22, 27], max_len=447)

            # Checkpoint 1A: Part 2 (lora_qwen3_14b)
            if "qwen3_14b_p2" in ckpts:
                p2_dir = ckpts["qwen3_14b_p2"]
                meta_p2 = read_checkpoint_meta(p2_dir, default_keep=27, default_layers=[2, 4, 6, 9, 13, 17, 22, 27], default_max_len=447)
                trunk1.set_active_layers(meta_p2["layers"], meta_p2["max_len"])
                backup_t1 = trunk1.snapshot_weights(p2_dir)
                merged = merge_lora(trunk1.model, p2_dir, sign=+1.0)
                if not merged and not is_smoke:
                    raise RuntimeError(f"Failed to merge required LoRA adapter at {p2_dir}")
                heads_sd = load_head_file(p2_dir)
                trunk1.load_norm_weights(heads_sd)
                W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd)
                if W_p is None and not is_smoke:
                    raise RuntimeError(f"Required classification head W_p is None for {p2_dir}")
                p2_labels = meta_p2["labels"]

                Z_te, Z_unseen, Z_tr = _extract_trunk(trunk1, end_id=meta_p2.get("end_id"))
                if N_unseen > 0:
                    features_unseen_dict["qwen3_14b_p2"] = Z_unseen.astype(np.float16)
                update_hybrid_groups(Z_te, prior_scales.get("qwen3_14b_p2", 1.00), Z_unseen if N_unseen > 0 else None)

                # Main Stack contributions
                w_main_p = model_weights.get("qwen3_14b_p2", {}).get("plain_weight", main_members.get("lora_qwen3_14b", {}).get("plain_weight", 0.42))
                w_main_c = model_weights.get("qwen3_14b_p2", {}).get("twin_weight", main_members.get("lora_qwen3_14b", {}).get("twin_weight", 0.12))
                lp_u_p = np.zeros((0, K), dtype=np.float32)
                lp_u_c = np.zeros((0, K), dtype=np.float32)
                lp_c_blend = np.zeros((N_test, K), dtype=np.float32)
                lp_p = np.zeros((N_test, K), dtype=np.float32)
                lp_c_blend = np.zeros((N_test, K), dtype=np.float32)
                if W_p is not None:
                    if run_submission and len(Z_te) > 0:
                        lp_p = log_softmax(align_cols(Z_te @ W_p.T + b_p, p2_labels))
                        LOGIT_MAIN += lp_p * w_main_p
                        w_comp_p = comp_members.get("lora_qwen3_14b", {}).get("plain_weight", 0.30)
                        comp_lora_posteriors.append((w_comp_p, softmax(lp_p, axis=1)))
                    if run_benchmark and len(Z_unseen) > 0:
                        lp_u_p = log_softmax(align_cols(Z_unseen @ W_p.T + b_p, p2_labels))

                if W_c is not None:
                    if run_submission and len(Z_te) > 0 and tracker_test is not None:
                        lp_c85, valid_c85 = compute_centered_logits(Z_te, tracker_test.groups[0.85], W_c, b_c, p2_labels)
                        LOGIT_MAIN[valid_c85] += lp_c85[valid_c85] * w_main_c
                        lp_c_blend = compute_multires_centered_blend(Z_te, tracker_test.groups, W_c, b_c, p2_labels, weights=(w85, w72, w60))
                        w_comp_c = comp_members.get("lora_qwen3_14b", {}).get("twin_weight", 0.10)
                        comp_lora_posteriors.append((w_comp_c, softmax(lp_c_blend, axis=1)))
                    if run_benchmark and len(Z_unseen) > 0 and tracker_unseen is not None:
                        lp_u_c = compute_multires_centered_blend(Z_unseen, tracker_unseen.groups, W_c, b_c, p2_labels, weights=(w85, w72, w60))

                if W_p is not None:
                    w_tot_14b = w_main_p + (w_main_c if W_c is not None else 0.0)
                    lp_model_te = _combine_plain_and_twin_lp(lp_p, lp_c_blend if W_c is not None else None, w_main_p, w_main_c) if (run_submission and len(lp_p) == N_test) else np.zeros((0, K), dtype=np.float32)
                    lp_model_u = _combine_plain_and_twin_lp(lp_u_p, lp_u_c if W_c is not None else None, w_main_p, w_main_c) if (run_benchmark and len(lp_u_p) == N_unseen) else np.zeros((0, K), dtype=np.float32)
                    auditor.add_model("Qwen3-14B Part-2", tier="Main", weight=w_tot_14b, unseen_log_probs=lp_model_u, test_log_probs=lp_model_te)

                # wPCA + DP Posterior + Spectral Bank (Symmetric test and unseen)
                apply_wpca_and_spectral(
                    model_key="qwen3_14b_p2",
                    Z_te_mat=Z_te,
                    Z_tr_mat=Z_tr,
                    Z_u_mat=Z_unseen if N_unseen > 0 else None,
                    dp_w=dp_weights.get("q3_14b", 0.38),
                    prior_scale=prior_scales.get("qwen3_14b_p2", 1.00),
                    d_target=256,
                    se_floor=se_floors.get("qwen3_14b_p2", 0.10)
                )

                trunk1.restore_weights(backup_t1)
                del backup_t1, Z_tr, Z_te, heads_sd
                if "Z_unseen" in locals():
                    del Z_unseen
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            # Checkpoint 1B / Optional Two-Stage LoRA24 & Part-1
            for opt_14b_key in ("qwen3_14b_inhouse", "qwen3_14b_ts", "qwen3_14b_v5", "qwen3_14b_p1"):
                if opt_14b_key in ckpts:
                    primary_ts14_dir = ckpts[opt_14b_key]
                    prefix_14b = "q14v5" if "v5" in opt_14b_key else "ts_q3b14" if "ts" in opt_14b_key else opt_14b_key
                    found_14b_folds = find_fold_adapters(input_dir, prefix_14b)

                    def _is_fold_14b(p: Path) -> bool:
                        nm = p.name.lower()
                        return any(f"_f{i}" in nm or f"-f{i}" in nm or f"_fold{i}" in nm for i in range(10))

                    folds_only_14b = sorted([p for p in found_14b_folds if _is_fold_14b(p)], key=lambda x: str(x).lower())
                    full_only_14b = [p for p in found_14b_folds if not _is_fold_14b(p)]
                    if not full_only_14b:
                        full_only_14b = [primary_ts14_dir]
                    fold_dirs_14b = folds_only_14b + full_only_14b
                    ts14_dir = fold_dirs_14b[-1]

                    def_layers_14b = [2, 4, 6, 9, 13, 17, 21, 24] if opt_14b_key == "qwen3_14b_p1" else [2, 4, 6, 8, 12, 16, 20, 24]
                    meta_ts14 = read_checkpoint_meta(ts14_dir, default_keep=24, default_layers=def_layers_14b, default_max_len=447)
                    trunk1.set_active_layers(meta_ts14["layers"], meta_ts14["max_len"])
                    trunk1.set_final_norm(not meta_ts14["no_final_norm"])
                    backup_t1_ts = trunk1.snapshot_weights(ts14_dir)
                    merged = merge_lora(trunk1.model, ts14_dir, sign=+1.0)
                    if not merged:
                        print(f"[Pipeline] Optional checkpoint {opt_14b_key} failed to merge LoRA; skipping.")
                        trunk1.restore_weights(backup_t1_ts)
                        trunk1.set_final_norm(False)
                    else:
                        heads_sd = load_head_file(ts14_dir)
                        trunk1.load_norm_weights(heads_sd)
                        W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd)
                        ts14_labels = meta_ts14["labels"]
                        p_comp_def = 0.12 if opt_14b_key == "qwen3_14b_p1" else 0.22
                        c_comp_def = 0.04 if opt_14b_key == "qwen3_14b_p1" else 0.08
                        comp_alias = "lora14b_v1" if opt_14b_key == "qwen3_14b_p1" else opt_14b_key
                        w_comp_p = model_weights.get(opt_14b_key, {}).get("plain_weight", comp_members.get(comp_alias, {}).get("plain_weight", p_comp_def))
                        w_comp_c = model_weights.get(opt_14b_key, {}).get("twin_weight", comp_members.get(comp_alias, {}).get("twin_weight", c_comp_def))
                        w_comp_tot = w_comp_p + w_comp_c

                        # Single pass for test + unseen benchmark (Pure Inference)
                        Z_te, Z_unseen, Z_tr = _extract_trunk(trunk1, end_id=meta_ts14.get("end_id"))
                        if N_unseen > 0:
                            features_unseen_dict[opt_14b_key] = Z_unseen.astype(np.float16)

                        # Auto-align label order directly on Z_unseen
                        cand_orders = [ts14_labels]
                        if sorted(LABELS) not in cand_orders:
                            cand_orders.append(sorted(LABELS))
                        if list(LABELS) not in cand_orders:
                            cand_orders.append(list(LABELS))

                        best_align_order = ts14_labels
                        best_solo_bacc = -1.0
                        best_lp_u = None
                        if W_p is not None and N_unseen > 0:
                            for cand_o in cand_orders:
                                try:
                                    trial_lp_p = log_softmax(align_cols(Z_unseen @ W_p.T + b_p, cand_o))
                                    if W_c is not None:
                                        trial_lp_c = compute_multires_centered_blend(Z_unseen, tracker_unseen.groups, W_c, b_c, cand_o, weights=(w85, w72, w60))
                                        trial_lp = _combine_plain_and_twin_lp(trial_lp_p, trial_lp_c, w_comp_p, w_comp_c)
                                    else:
                                        trial_lp = trial_lp_p
                                    trial_acc = float(balanced_accuracy_score(y_unseen, trial_lp.argmax(axis=1)))
                                    if trial_acc > best_solo_bacc:
                                        best_solo_bacc = trial_acc
                                        best_align_order = cand_o
                                        best_lp_u = trial_lp
                                except Exception:
                                    continue

                        solo_bacc = best_solo_bacc if best_solo_bacc >= 0 else 0.0
                        lp_u = best_lp_u if best_lp_u is not None else (log_softmax(align_cols(Z_unseen @ W_p.T + b_p, ts14_labels)) if (W_p is not None and N_unseen > 0) else np.zeros((0, K), dtype=np.float32))
                        ts14_labels = best_align_order
                        align_name = "canonical" if best_align_order == list(LABELS) else "alphabetical" if best_align_order == sorted(LABELS) else "custom"
                        print(f"[Pipeline] {opt_14b_key}: solo unseen BAcc = {solo_bacc:.4f} (alignment: {align_name})")

                        gate_passed = True
                        if len(auditor.models) > 0 and N_unseen > 0:
                            lift = auditor.evaluate_marginal_lift(candidate_unseen_lp=lp_u, weight=w_comp_tot)
                            if lift < -0.002:
                                print(f"[Pipeline] {opt_14b_key}: marginal lift {lift:+.4f} < -0.002 -> skipping blend.")
                                gate_passed = False

                        if gate_passed:
                            pr_sc = prior_scales.get(opt_14b_key, 0.50)
                            update_hybrid_groups(Z_te, pr_sc, Z_unseen if N_unseen > 0 else None)
                            lp_p = None
                            lp_c_blend = np.zeros((N_test, K), dtype=np.float32)
                            if run_submission and len(Z_te) > 0:
                                lp_p = log_softmax(align_cols(Z_te @ W_p.T + b_p, ts14_labels)) if W_p is not None else None
                                if W_c is not None and tracker_test is not None:
                                    lp_c_blend = compute_multires_centered_blend(Z_te, tracker_test.groups, W_c, b_c, ts14_labels, weights=(w85, w72, w60))
                                if W_p is not None and lp_p is not None:
                                    comp_lora_posteriors.append((w_comp_p, softmax(lp_p, axis=1)))
                                if W_c is not None:
                                    comp_lora_posteriors.append((w_comp_c, softmax(lp_c_blend, axis=1)))

                            def_dp_14b = 0.08 if opt_14b_key == "qwen3_14b_p1" else 0.10
                            def_se_14b = 0.35 if opt_14b_key == "qwen3_14b_p1" else 0.25
                            dp_w_14b = dp_weights.get(opt_14b_key, dp_weights.get("part1_14b" if opt_14b_key == "qwen3_14b_p1" else "q3_14b_ts", def_dp_14b))
                            apply_wpca_and_spectral(
                                model_key=opt_14b_key,
                                Z_te_mat=Z_te,
                                Z_tr_mat=Z_tr,
                                Z_u_mat=Z_unseen if (N_unseen > 0 and 'Z_unseen' in locals() and Z_unseen is not None and len(Z_unseen) == N_unseen) else None,
                                dp_w=dp_w_14b,
                                prior_scale=pr_sc,
                                d_target=256,
                                se_floor=se_floors.get(opt_14b_key, def_se_14b)
                            )

                            if W_p is not None:
                                w_tot_ts14 = w_comp_p + (w_comp_c if W_c is not None else 0.0)
                                lp_m_te = _combine_plain_and_twin_lp(lp_p, lp_c_blend if W_c is not None else None, w_comp_p, w_comp_c) if (run_submission and lp_p is not None and len(lp_p) == N_test) else np.zeros((0, K), dtype=np.float32)
                                lp_m_u = lp_u if (run_benchmark and N_unseen > 0 and 'lp_u' in locals() and len(lp_u) == N_unseen) else np.zeros((0, K), dtype=np.float32)
                                aud_name_14b = "Qwen3-14B Part-1" if opt_14b_key == "qwen3_14b_p1" else opt_14b_key
                                auditor.add_model(name=aud_name_14b, tier="Comp", weight=w_tot_ts14, unseen_log_probs=lp_m_u, test_log_probs=lp_m_te)
                        else:
                            print(f"[Pipeline] Optional checkpoint {opt_14b_key} rejected by gate; excluded from logits and spectral bank.")

                        if backup_t1_ts is not None:
                            trunk1.restore_weights(backup_t1_ts)
                            del backup_t1_ts
                        trunk1.set_final_norm(False)
                        del Z_tr, Z_te, heads_sd
                        if "Z_unseen" in locals():
                            del Z_unseen
                        gc.collect()
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()

            trunk1.free_memory()
            del trunk1
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # ==============================================================================
        # TRUNK 2: Qwen2.5-14B (48 layers, hidden=5120)
        # ==============================================================================
        base_q25_14b = verified_bases.get("qwen25_14b") or find_base_model(input_dir, "qwen2", 5120, 48)
        if base_q25_14b and any(k in ckpts for k in ("qwen25_14b_p2", "qwen25_14b_inhouse")):
            trunk2 = None
            for q25_key in ("qwen25_14b_p2", "qwen25_14b_inhouse"):
                if q25_key not in ckpts:
                    continue
                p2_dir = ckpts[q25_key]
                meta_q25 = read_checkpoint_meta(p2_dir, default_keep=32, default_layers=[3, 5, 8, 11, 16, 21, 27, 32], default_max_len=447)
                if trunk2 is None:
                    print(f"[Pipeline] Loading Trunk 2: Qwen2.5-14B ({q25_key}) from {base_q25_14b}...")
                    trunk2 = UniversalLoRATrunkLoader(base_q25_14b, keep_layers=meta_q25["keep"], layers=meta_q25["layers"], max_len=meta_q25["max_len"])
                else:
                    trunk2.set_active_layers(meta_q25["layers"], meta_q25["max_len"])
                trunk2.set_final_norm(not meta_q25["no_final_norm"])
                backup_t2 = trunk2.snapshot_weights(p2_dir)
                merged = merge_lora(trunk2.model, p2_dir, sign=+1.0)
                if not merged and not is_smoke:
                    raise RuntimeError(f"Failed to merge required LoRA adapter at {p2_dir}")
                heads_sd = load_head_file(p2_dir)
                trunk2.load_norm_weights(heads_sd)
                W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd)
                if W_p is None and not is_smoke:
                    raise RuntimeError(f"Required classification head W_p is None for {p2_dir}")
                p2_labels = meta_q25["labels"]

                Z_te, Z_unseen, Z_tr = _extract_trunk(trunk2, end_id=meta_q25.get("end_id"))
                if N_unseen > 0:
                    features_unseen_dict[q25_key] = Z_unseen.astype(np.float16)
                update_hybrid_groups(Z_te, prior_scales.get(q25_key, 0.85), Z_unseen if N_unseen > 0 else None)

                w_main_p = model_weights.get(q25_key, {}).get("plain_weight", main_members.get("lora_qwen25_14b", {}).get("plain_weight", 0.24))
                w_main_c = model_weights.get(q25_key, {}).get("twin_weight", main_members.get("lora_qwen25_14b", {}).get("twin_weight", 0.10))
                lp_u_p = np.zeros((0, K), dtype=np.float32)
                lp_u_c = np.zeros((0, K), dtype=np.float32)
                lp_c_blend = np.zeros((N_test, K), dtype=np.float32)
                lp_p = np.zeros((N_test, K), dtype=np.float32)
                lp_c_blend = np.zeros((N_test, K), dtype=np.float32)
                if W_p is not None:
                    if run_submission and len(Z_te) > 0:
                        lp_p = log_softmax(align_cols(Z_te @ W_p.T + b_p, p2_labels))
                        LOGIT_MAIN += lp_p * w_main_p
                        w_comp_p = comp_members.get("lora_qwen25_14b", {}).get("plain_weight", 0.20)
                        comp_lora_posteriors.append((w_comp_p, softmax(lp_p, axis=1)))
                    if run_benchmark and len(Z_unseen) > 0:
                        lp_u_p = log_softmax(align_cols(Z_unseen @ W_p.T + b_p, p2_labels))

                if W_c is not None:
                    if run_submission and len(Z_te) > 0 and tracker_test is not None:
                        lp_c85, valid_c85 = compute_centered_logits(Z_te, tracker_test.groups[0.85], W_c, b_c, p2_labels)
                        LOGIT_MAIN[valid_c85] += lp_c85[valid_c85] * w_main_c
                        lp_c_blend = compute_multires_centered_blend(Z_te, tracker_test.groups, W_c, b_c, p2_labels, weights=(w85, w72, w60))
                        w_comp_c = comp_members.get("lora_qwen25_14b", {}).get("twin_weight", 0.08)
                        comp_lora_posteriors.append((w_comp_c, softmax(lp_c_blend, axis=1)))
                    if run_benchmark and len(Z_unseen) > 0 and tracker_unseen is not None:
                        lp_u_c = compute_multires_centered_blend(Z_unseen, tracker_unseen.groups, W_c, b_c, p2_labels, weights=(w85, w72, w60))

                if W_p is not None:
                    w_tot_q25 = w_main_p + (w_main_c if W_c is not None else 0.0)
                    lp_model_te = _combine_plain_and_twin_lp(lp_p, lp_c_blend if W_c is not None else None, w_main_p, w_main_c) if (run_submission and len(lp_p) == N_test) else np.zeros((0, K), dtype=np.float32)
                    lp_model_u = _combine_plain_and_twin_lp(lp_u_p, lp_u_c if W_c is not None else None, w_main_p, w_main_c) if (run_benchmark and len(lp_u_p) == N_unseen) else np.zeros((0, K), dtype=np.float32)
                    aud_name = "Qwen2.5-14B Part-2" if q25_key == "qwen25_14b_p2" else "Qwen2.5-14B In-House"
                    auditor.add_model(aud_name, tier="Main", weight=w_tot_q25, unseen_log_probs=lp_model_u, test_log_probs=lp_model_te)

                apply_wpca_and_spectral(
                    model_key=q25_key,
                    Z_te_mat=Z_te,
                    Z_tr_mat=Z_tr,
                    Z_u_mat=Z_unseen if N_unseen > 0 else None,
                    dp_w=dp_weights.get("q25_14b", 0.24),
                    prior_scale=prior_scales.get(q25_key, 0.85),
                    d_target=256,
                    se_floor=se_floors.get(q25_key, 0.10)
                )

                trunk2.restore_weights(backup_t2)
                trunk2.set_final_norm(False)
                del backup_t2, Z_tr, Z_te, heads_sd
                if "Z_unseen" in locals():
                    del Z_unseen
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            if trunk2 is not None:
                trunk2.free_memory()
                del trunk2
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        # ==============================================================================
        # TRUNK 3: Qwen3-8B (36 layers, hidden=4096)
        # ==============================================================================
        base_q3_8b = verified_bases.get("qwen3_8b") or find_base_model(input_dir, "qwen3", 4096, 36)
        if base_q3_8b and ("instyle" in ckpts or "qwen3_8b_claude" in ckpts or "qwen3_8b_ts" in ckpts or "qwen3_8b_v5" in ckpts or "qwen3_8b_inhouse" in ckpts):
            print(f"[Pipeline] Loading Trunk 3: Qwen3-8B from {base_q3_8b}...")
            trunk3 = UniversalLoRATrunkLoader(base_q3_8b, keep_layers=24, layers=[2, 4, 6, 8, 12, 16, 20, 24], max_len=447)

            # In-Style LoRA
            if "instyle" in ckpts:
                instyle_dir = ckpts["instyle"]
                meta_ins = read_checkpoint_meta(instyle_dir, default_keep=24, default_layers=[2, 4, 6, 8, 12, 16, 20, 24], default_max_len=447)
                trunk3.set_active_layers(meta_ins["layers"], meta_ins["max_len"])
                backup_t3 = trunk3.snapshot_weights(instyle_dir)
                merged = merge_lora(trunk3.model, instyle_dir, sign=+1.0)
                if not merged:
                    print(f"[Pipeline] Optional checkpoint instyle failed to merge LoRA; skipping.")
                    trunk3.restore_weights(backup_t3)
                else:
                    heads_sd = load_head_file(instyle_dir)
                    trunk3.load_norm_weights(heads_sd)
                    W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd)
                    if W_p is None:
                        print(f"[Pipeline] In-Style checkpoint has no plain classification head W_p; computing density and spectral features.")

                    w_comp_p = model_weights.get("instyle", {}).get("plain_weight", comp_members.get("instyle", {}).get("plain_weight", 0.16))
                    w_comp_c = model_weights.get("instyle", {}).get("twin_weight", comp_members.get("instyle", {}).get("twin_weight", 0.06))
                    w_comp_tot = w_comp_p + w_comp_c
                    # Single pass for test + unseen benchmark (Pure Inference)
                    Z_te, Z_unseen, Z_tr = _extract_trunk(trunk3, end_id=meta_ins.get("end_id"))
                    if N_unseen > 0:
                        features_unseen_dict["instyle"] = Z_unseen.astype(np.float16)

                    # Auto-align label order directly on Z_unseen
                    cand_orders = [meta_ins["labels"]]
                    if sorted(LABELS) not in cand_orders:
                        cand_orders.append(sorted(LABELS))
                    if list(LABELS) not in cand_orders:
                        cand_orders.append(list(LABELS))

                    best_align_order = meta_ins["labels"]
                    best_solo_bacc = -1.0
                    best_lp_u = None
                    if W_p is not None and N_unseen > 0:
                        for cand_o in cand_orders:
                            try:
                                trial_lp_p = log_softmax(align_cols(Z_unseen @ W_p.T + b_p, cand_o))
                                if W_c is not None:
                                    trial_lp_c = compute_multires_centered_blend(Z_unseen, tracker_unseen.groups, W_c, b_c, cand_o, weights=(w85, w72, w60))
                                    trial_lp = _combine_plain_and_twin_lp(trial_lp_p, trial_lp_c, w_comp_p, w_comp_c)
                                else:
                                    trial_lp = trial_lp_p
                                trial_acc = float(balanced_accuracy_score(y_unseen, trial_lp.argmax(axis=1)))
                                if trial_acc > best_solo_bacc:
                                    best_solo_bacc = trial_acc
                                    best_align_order = cand_o
                                    best_lp_u = trial_lp
                            except Exception:
                                continue

                    solo_bacc = best_solo_bacc if best_solo_bacc >= 0 else 0.0
                    lp_u = best_lp_u if best_lp_u is not None else (log_softmax(align_cols(Z_unseen @ W_p.T + b_p, meta_ins["labels"])) if (W_p is not None and N_unseen > 0) else np.zeros((0, K), dtype=np.float32))
                    ins_labels = best_align_order
                    align_name = "canonical" if best_align_order == list(LABELS) else "alphabetical" if best_align_order == sorted(LABELS) else "custom"
                    print(f"[Pipeline] instyle: solo unseen BAcc = {solo_bacc:.4f} (alignment: {align_name})")

                    gate_passed = True
                    if len(auditor.models) > 0 and N_unseen > 0:
                        lift = auditor.evaluate_marginal_lift(candidate_unseen_lp=lp_u, weight=w_comp_tot)
                        if lift < -0.002:
                            print(f"[Pipeline] instyle: marginal lift {lift:+.4f} < -0.002 -> skipping blend.")
                            gate_passed = False

                    if gate_passed:
                        update_hybrid_groups(Z_te, prior_scales.get("instyle", 0.45), Z_unseen if N_unseen > 0 else None)

                        lp_p = None
                        lp_c_blend = np.zeros((N_test, K), dtype=np.float32)
                        lp_u_p = np.zeros((0, K), dtype=np.float32)
                        lp_u_c = np.zeros((0, K), dtype=np.float32)

                        if run_submission and len(Z_te) > 0:
                            lp_p = log_softmax(align_cols(Z_te @ W_p.T + b_p, ins_labels)) if W_p is not None else None
                            if W_p is not None and lp_p is not None:
                                comp_lora_posteriors.append((w_comp_p, softmax(lp_p, axis=1)))
                            if W_c is not None and tracker_test is not None:
                                lp_c_blend = compute_multires_centered_blend(Z_te, tracker_test.groups, W_c, b_c, ins_labels, weights=(w85, w72, w60))
                                comp_lora_posteriors.append((w_comp_c, softmax(lp_c_blend, axis=1)))

                        if run_benchmark and len(Z_unseen) > 0:
                            if W_p is not None:
                                lp_u_p = log_softmax(align_cols(Z_unseen @ W_p.T + b_p, ins_labels))
                            if W_c is not None and tracker_unseen is not None:
                                lp_u_c = compute_multires_centered_blend(Z_unseen, tracker_unseen.groups, W_c, b_c, ins_labels, weights=(w85, w72, w60))

                        apply_wpca_and_spectral(
                            model_key="instyle",
                            Z_te_mat=Z_te,
                            Z_tr_mat=Z_tr,
                            Z_u_mat=Z_unseen if N_unseen > 0 else None,
                            dp_w=dp_weights.get("instyle", 0.10),
                            prior_scale=prior_scales.get("instyle", 0.45),
                            d_target=256,
                            se_floor=se_floors.get("instyle", 0.35)
                        )

                        if W_p is not None:
                            w_tot_ins = w_comp_p + (w_comp_c if W_c is not None else 0.0)
                            lp_m_te = _combine_plain_and_twin_lp(lp_p, lp_c_blend if W_c is not None else None, w_comp_p, w_comp_c) if (run_submission and lp_p is not None and len(lp_p) == N_test) else np.zeros((0, K), dtype=np.float32)
                            lp_m_u = _combine_plain_and_twin_lp(lp_u_p, lp_u_c if W_c is not None else None, w_comp_p, w_comp_c) if (run_benchmark and len(lp_u_p) == N_unseen) else np.zeros((0, K), dtype=np.float32)
                            auditor.add_model("Qwen3-8B In-Style", tier="Comp", weight=w_tot_ins, unseen_log_probs=lp_m_u, test_log_probs=lp_m_te)

                    trunk3.restore_weights(backup_t3)
                    del backup_t3, heads_sd, Z_te, Z_tr
                    if "Z_tr" in locals():
                        del Z_tr
                    if "Z_te" in locals():
                        del Z_te
                    if "Z_unseen" in locals():
                        del Z_unseen
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()

            # Optional Checkpoints on Trunk 3: Two-Stage LoRA24 (qwen3_8b_ts, qwen3_8b_v5, qwen3_8b_inhouse)
            for opt_8b_key in ("qwen3_8b_ts", "qwen3_8b_v5", "qwen3_8b_inhouse"):
                if opt_8b_key in ckpts:
                    primary_ts8_dir = ckpts[opt_8b_key]
                    prefix_8b = "q8v5" if "v5" in opt_8b_key else "ts_q3b8" if "ts" in opt_8b_key else opt_8b_key
                    found_8b_folds = find_fold_adapters(input_dir, prefix_8b)

                    def _is_fold_8b(p: Path) -> bool:
                        nm = p.name.lower()
                        return any(f"_f{i}" in nm or f"-f{i}" in nm or f"_fold{i}" in nm for i in range(10))

                    folds_only_8b = sorted([p for p in found_8b_folds if _is_fold_8b(p)], key=lambda x: str(x).lower())
                    full_only_8b = [p for p in found_8b_folds if not _is_fold_8b(p)]
                    if not full_only_8b:
                        full_only_8b = [primary_ts8_dir]
                    fold_dirs_8b = folds_only_8b + full_only_8b
                    ts8_dir = fold_dirs_8b[-1]

                    meta_ts8 = read_checkpoint_meta(ts8_dir, default_keep=24, default_layers=[2, 4, 6, 8, 12, 16, 20, 24], default_max_len=447)
                    trunk3.set_active_layers(meta_ts8["layers"], meta_ts8["max_len"])
                    trunk3.set_final_norm(not meta_ts8["no_final_norm"])
                    backup_t3_ts = trunk3.snapshot_weights(ts8_dir)
                    merged = merge_lora(trunk3.model, ts8_dir, sign=+1.0)
                    if not merged:
                        print(f"[Pipeline] Optional checkpoint {opt_8b_key} failed to merge LoRA; skipping.")
                        trunk3.restore_weights(backup_t3_ts)
                        trunk3.set_final_norm(False)
                    else:
                        heads_sd = load_head_file(ts8_dir)
                        trunk3.load_norm_weights(heads_sd)
                        W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd)
                        ts8_labels = meta_ts8["labels"]
                        w_comp_p = model_weights.get(opt_8b_key, {}).get("plain_weight", comp_members.get(opt_8b_key, {}).get("plain_weight", 0.16))
                        w_comp_c = model_weights.get(opt_8b_key, {}).get("twin_weight", comp_members.get(opt_8b_key, {}).get("twin_weight", 0.06))
                        w_comp_tot = w_comp_p + w_comp_c

                        # Single pass extraction (with multi-fold averaging if multiple folds discovered)
                        num_8b_folds = len(fold_dirs_8b)
                        if num_8b_folds > 1:
                            trunk3.restore_weights(backup_t3_ts)
                            del backup_t3_ts

                            P_p_acc = np.zeros((N_test, K), dtype=np.float64)
                            P_c_acc = np.zeros((N_test, K), dtype=np.float64)
                            P_u_p_acc = np.zeros((N_unseen, K), dtype=np.float64)
                            P_u_c_acc = np.zeros((N_unseen, K), dtype=np.float64)
                            has_any_c = False
                            Z_te = None
                            Z_tr = np.zeros((0, 4096), dtype=np.float32)
                            Z_unseen = None
                            print(f"[Pipeline] {opt_8b_key}: extracting & averaging across {num_8b_folds} fold adapters...")

                            for f_idx, f_dir in enumerate(fold_dirs_8b):
                                is_last = (f_idx == num_8b_folds - 1)
                                f_meta = read_checkpoint_meta(f_dir, default_keep=24, default_layers=[2, 4, 6, 8, 12, 16, 20, 24], default_max_len=447)
                                trunk3.set_active_layers(f_meta["layers"], f_meta["max_len"])
                                trunk3.set_final_norm(not f_meta["no_final_norm"])
                                f_backup = trunk3.snapshot_weights(f_dir)
                                f_merged = merge_lora(trunk3.model, f_dir, sign=+1.0)
                                if not f_merged:
                                    trunk3.restore_weights(f_backup)
                                    continue
                                f_heads_sd = load_head_file(f_dir)
                                trunk3.load_norm_weights(f_heads_sd)
                                f_W_p, f_b_p, f_W_c, f_b_c = resolve_head_tensors(f_heads_sd)
                                f_labels = f_meta["labels"]

                                f_Z_te, f_Z_u, _ = _extract_trunk(trunk3, end_id=f_meta.get("end_id"))

                                if is_last:
                                    Z_te = f_Z_te
                                    Z_unseen = f_Z_u
                                    dim_f = f_Z_u.shape[1] if len(f_Z_u) > 0 else (f_Z_te.shape[1] if len(f_Z_te) > 0 else 4096)
                                    Z_tr = np.zeros((0, dim_f), dtype=np.float32)
                                    if N_unseen > 0:
                                        features_unseen_dict[opt_8b_key] = Z_unseen.astype(np.float16)

                                if f_idx == 0 or is_last:
                                    update_hybrid_groups(f_Z_te, prior_scales.get(opt_8b_key, 0.40), f_Z_u if N_unseen > 0 else None)

                                if run_submission and len(f_Z_te) > 0:
                                    if f_W_p is not None:
                                        p_fold = softmax(align_cols(f_Z_te @ f_W_p.T + f_b_p, f_labels), axis=1)
                                        P_p_acc += p_fold / num_8b_folds
                                    if f_W_c is not None and tracker_test is not None:
                                        has_any_c = True
                                        c_blend_f = compute_multires_centered_blend(f_Z_te, tracker_test.groups, f_W_c, f_b_c, f_labels, weights=(w85, w72, w60))
                                        P_c_acc += softmax(c_blend_f, axis=1) / num_8b_folds

                                if run_benchmark and len(f_Z_u) > 0:
                                    if f_W_p is not None:
                                        p_u_fold = softmax(align_cols(f_Z_u @ f_W_p.T + f_b_p, f_labels), axis=1)
                                        P_u_p_acc += p_u_fold / num_8b_folds
                                    if f_W_c is not None and tracker_unseen is not None:
                                        has_any_c = True
                                        c_u_blend_f = compute_multires_centered_blend(f_Z_u, tracker_unseen.groups, f_W_c, f_b_c, f_labels, weights=(w85, w72, w60))
                                        P_u_c_acc += softmax(c_u_blend_f, axis=1) / num_8b_folds

                                trunk3.restore_weights(f_backup)
                                del f_backup, f_heads_sd, f_Z_te
                                if "f_Z_u" in locals():
                                    del f_Z_u
                                gc.collect()
                                if torch.cuda.is_available():
                                    torch.cuda.empty_cache()

                            trunk3.set_final_norm(False)
                            lp_p = np.log(np.clip(P_p_acc, 1e-9, 1.0)) if (run_submission and len(P_p_acc) == N_test) else np.zeros((0, K), dtype=np.float32)
                            lp_c_blend = np.log(np.clip(P_c_acc, 1e-9, 1.0)) if (run_submission and has_any_c and len(P_c_acc) == N_test) else np.zeros((N_test, K), dtype=np.float32)
                            lp_u = np.zeros((0, K), dtype=np.float32)
                            if run_benchmark and len(P_u_p_acc) == N_unseen:
                                lp_u_p = np.log(np.clip(P_u_p_acc, 1e-9, 1.0))
                                lp_u_c = np.log(np.clip(P_u_c_acc, 1e-9, 1.0)) if has_any_c else None
                                lp_u = _combine_plain_and_twin_lp(lp_u_p, lp_u_c, w_comp_p, w_comp_c)
                            gate_passed = True
                            backup_t3_ts = None
                        else:
                            Z_te, Z_unseen, Z_tr = _extract_trunk(trunk3, end_id=meta_ts8.get("end_id"))
                            if N_unseen > 0:
                                features_unseen_dict[opt_8b_key] = Z_unseen.astype(np.float16)

                            # Auto-align label order directly on Z_unseen
                            cand_orders = [ts8_labels]
                            if sorted(LABELS) not in cand_orders:
                                cand_orders.append(sorted(LABELS))
                            if list(LABELS) not in cand_orders:
                                cand_orders.append(list(LABELS))

                            best_align_order = ts8_labels
                            best_solo_bacc = -1.0
                            best_lp_u = None
                            if W_p is not None and N_unseen > 0:
                                for cand_o in cand_orders:
                                    try:
                                        trial_lp_p = log_softmax(align_cols(Z_unseen @ W_p.T + b_p, cand_o))
                                        if W_c is not None:
                                            trial_lp_c = compute_multires_centered_blend(Z_unseen, tracker_unseen.groups, W_c, b_c, cand_o, weights=(w85, w72, w60))
                                            trial_lp = _combine_plain_and_twin_lp(trial_lp_p, trial_lp_c, w_comp_p, w_comp_c)
                                        else:
                                            trial_lp = trial_lp_p
                                        trial_acc = float(balanced_accuracy_score(y_unseen, trial_lp.argmax(axis=1)))
                                        if trial_acc > best_solo_bacc:
                                            best_solo_bacc = trial_acc
                                            best_align_order = cand_o
                                            best_lp_u = trial_lp
                                    except Exception:
                                        continue

                            solo_bacc = best_solo_bacc if best_solo_bacc >= 0 else 0.0
                            lp_u = best_lp_u if best_lp_u is not None else (log_softmax(align_cols(Z_unseen @ W_p.T + b_p, ts8_labels)) if (W_p is not None and N_unseen > 0) else np.zeros((0, K), dtype=np.float32))
                            ts8_labels = best_align_order
                            align_name = "canonical" if best_align_order == list(LABELS) else "alphabetical" if best_align_order == sorted(LABELS) else "custom"
                            print(f"[Pipeline] {opt_8b_key}: solo unseen BAcc = {solo_bacc:.4f} (alignment: {align_name})")

                            gate_passed = True
                            if len(auditor.models) > 0 and N_unseen > 0:
                                lift = auditor.evaluate_marginal_lift(candidate_unseen_lp=lp_u, weight=w_comp_tot)
                                if lift < -0.002:
                                    print(f"[Pipeline] {opt_8b_key}: marginal lift {lift:+.4f} < -0.002 -> skipping blend.")
                                    gate_passed = False
                            if gate_passed:
                                def_pr = 0.35 if opt_8b_key == "qwen3_8b_inhouse" else 0.40
                                pr_sc = prior_scales.get(opt_8b_key, def_pr)
                                update_hybrid_groups(Z_te, pr_sc, Z_unseen if ('Z_unseen' in locals() and N_unseen > 0) else None)
                                lp_p = log_softmax(align_cols(Z_te @ W_p.T + b_p, ts8_labels)) if W_p is not None else None
                                if W_c is not None:
                                    lp_c_blend = compute_multires_centered_blend(Z_te, tracker_test.groups, W_c, b_c, ts8_labels, weights=(w85, w72, w60))
                                else:
                                    lp_c_blend = np.zeros((N_test, K), dtype=np.float32)
                                if N_unseen > 0 and 'Z_unseen' in locals() and Z_unseen is not None and len(Z_unseen) == N_unseen:
                                    lp_u_p = log_softmax(align_cols(Z_unseen @ W_p.T + b_p, ts8_labels)) if W_p is not None else None
                                    if W_c is not None:
                                        lp_u_c = compute_multires_centered_blend(Z_unseen, tracker_unseen.groups, W_c, b_c, ts8_labels, weights=(w85, w72, w60))
                                        lp_u = _combine_plain_and_twin_lp(lp_u_p, lp_u_c, w_comp_p, w_comp_c)
                                    else:
                                        lp_u = lp_u_p if lp_u_p is not None else np.zeros((0, K), dtype=np.float32)

                        if gate_passed:
                            def_pr = 0.35 if opt_8b_key == "qwen3_8b_inhouse" else 0.40
                            def_se = 0.40 if opt_8b_key == "qwen3_8b_inhouse" else 0.35
                            pr_sc = prior_scales.get(opt_8b_key, def_pr)
                            se_fl = se_floors.get(opt_8b_key, def_se)

                            if num_8b_folds > 1:
                                comp_lora_posteriors.append((w_comp_p, P_p_acc))
                                if has_any_c:
                                    comp_lora_posteriors.append((w_comp_c, P_c_acc))
                            else:
                                if W_p is not None and lp_p is not None:
                                    comp_lora_posteriors.append((w_comp_p, softmax(lp_p, axis=1)))
                                if W_c is not None:
                                    comp_lora_posteriors.append((w_comp_c, softmax(lp_c_blend, axis=1)))

                            dp_w_8b = dp_weights.get(opt_8b_key, dp_weights.get("q3_8b_ts", 0.10))
                            apply_wpca_and_spectral(
                                model_key=opt_8b_key,
                                Z_te_mat=Z_te,
                                Z_tr_mat=Z_tr,
                                Z_u_mat=Z_unseen if (N_unseen > 0 and 'Z_unseen' in locals() and Z_unseen is not None and len(Z_unseen) == N_unseen) else None,
                                dp_w=dp_w_8b,
                                prior_scale=pr_sc,
                                d_target=256,
                                se_floor=se_fl
                            )

                            if lp_p is not None:
                                w_tot_ts8 = w_comp_p + (w_comp_c if (has_any_c if num_8b_folds > 1 else W_c is not None) else 0.0)
                                lp_m_te = _combine_plain_and_twin_lp(lp_p, lp_c_blend if (has_any_c if num_8b_folds > 1 else W_c is not None) else None, w_comp_p, w_comp_c) if (run_submission and len(lp_p) == N_test) else np.zeros((0, K), dtype=np.float32)
                                lp_m_u = lp_u if (run_benchmark and N_unseen > 0 and 'lp_u' in locals() and len(lp_u) == N_unseen) else np.zeros((0, K), dtype=np.float32)
                                auditor.add_model(name=opt_8b_key, tier="Comp", weight=w_tot_ts8, unseen_log_probs=lp_m_u, test_log_probs=lp_m_te)
                        else:
                            print(f"[Pipeline] Optional checkpoint {opt_8b_key} rejected by gate; excluded from logits and spectral bank.")

                        if backup_t3_ts is not None:
                            trunk3.restore_weights(backup_t3_ts)
                            del backup_t3_ts
                        trunk3.set_final_norm(False)
                        del Z_tr, Z_te, heads_sd
                        if "Z_unseen" in locals():
                            del Z_unseen
                        gc.collect()
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()

            # Claude Head LoRA
            if "qwen3_8b_claude" in ckpts:
                claude_dir = ckpts["qwen3_8b_claude"]
                meta_cl = read_checkpoint_meta(claude_dir, default_keep=24, default_layers=[2, 4, 6, 8, 12, 16, 20, 24], default_max_len=447)
                trunk3.set_active_layers(meta_cl["layers"], meta_cl["max_len"])
                merged = merge_lora(trunk3.model, claude_dir, sign=+1.0)
                if not merged and not is_smoke:
                    raise RuntimeError(f"Failed to merge required LoRA adapter at {claude_dir}")

                heads_sd_cl = load_head_file(claude_dir)
                if heads_sd_cl:
                    trunk3.load_norm_weights(heads_sd_cl)

                W_p, b_p, W_c, b_c = None, None, None, None
                W_spec, b_spec = None, None
                ro_labels = None
                readouts_file = claude_dir / "readouts.npz"
                if readouts_file.exists():
                    with np.load(str(readouts_file), allow_pickle=True) as rnpz:
                        if "classes" in rnpz:
                            ro_labels = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in rnpz["classes"]]
                        elif "labels" in rnpz:
                            ro_labels = [x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x) for x in rnpz["labels"]]
                        for wk, bk in [("plain_coef", "plain_intercept"), ("plain.weight", "plain.bias")]:
                            if wk in rnpz and bk in rnpz and rnpz[wk].shape[0] == 18:
                                W_p, b_p = np.array(rnpz[wk]), np.array(rnpz[bk])
                                break
                        for wk, bk in [("center_coef", "center_intercept"), ("center.weight", "center.bias")]:
                            if wk in rnpz and bk in rnpz and rnpz[wk].shape[0] == 18:
                                W_c, b_c = np.array(rnpz[wk]), np.array(rnpz[bk])
                                break
                        if "claude_coef" in rnpz and "claude_intercept" in rnpz:
                            if rnpz["claude_coef"].shape[0] == 5:
                                W_spec, b_spec = np.array(rnpz["claude_coef"]), np.array(rnpz["claude_intercept"])
                                if "claude_classes" in rnpz:
                                    claude_classes = list(rnpz["claude_classes"])
                if W_p is None and heads_sd_cl:
                    cand_W, cand_b, cand_Wc, cand_bc = resolve_head_tensors(heads_sd_cl)
                    if cand_W is not None and cand_W.shape[0] == 18:
                        W_p, b_p = cand_W, cand_b
                        if cand_Wc is not None and cand_Wc.shape[0] == 18:
                            W_c, b_c = cand_Wc, cand_bc

                # Ensure ro_labels is a valid 18-class label list (defaults to sorted(LABELS) alphabetical order)
                if ro_labels is None or len(ro_labels) != 18 or set(ro_labels) != set(LABELS):
                    cand_meta = meta_cl.get("labels", [])
                    if len(cand_meta) == 18 and set(cand_meta) == set(LABELS):
                        ro_labels = list(cand_meta)
                    else:
                        ro_labels = sorted(LABELS)

                if W_spec is None:
                    spec_file = claude_dir / "claude_head.safetensors"
                    if spec_file.exists():
                        ch_sd = load_head_file(spec_file)
                        for wk, bk in [("weight", "bias"), ("plain.weight", "plain.bias")]:
                            if wk in ch_sd and ch_sd[wk].shape[0] == 5:
                                W_spec = ch_sd[wk].cpu().numpy()
                                b_spec = ch_sd[bk].cpu().numpy() if bk in ch_sd else np.zeros(5, dtype=np.float32)
                                break
                if W_spec is None and not is_smoke:
                    raise RuntimeError(f"Required specialist head W_spec is None for {claude_dir}")
                if claude_classes is None:
                    claude_classes = [LABELS[i] for i in FAMILIES["claude"]]

                Z_te, Z_unseen, Z_tr = _extract_trunk(trunk3, end_id=meta_cl.get("end_id"))
                if N_unseen > 0:
                    features_unseen_dict["qwen3_8b_claude"] = Z_unseen.astype(np.float16)
                update_hybrid_groups(Z_te, prior_scales.get("qwen3_8b_claude", 0.30), Z_unseen if N_unseen > 0 else None)

                if W_spec is not None and W_spec.shape[0] == 5:
                    claude_spec_logits = (Z_te @ W_spec.T + b_spec) if (run_submission and len(Z_te) > 0) else None
                    claude_spec_logits_u = (Z_unseen @ W_spec.T + b_spec) if (run_benchmark and len(Z_unseen) > 0) else None

                if W_p is not None and W_p.shape[0] == 18:
                    w_comp_p = model_weights.get("qwen3_8b_claude", {}).get("plain_weight", comp_members.get("ntl_claude_head", {}).get("plain_weight", 0.10))
                    w_comp_c = model_weights.get("qwen3_8b_claude", {}).get("twin_weight", comp_members.get("ntl_claude_head", {}).get("twin_weight", 0.04))
                    lp_p = np.zeros((N_test, K), dtype=np.float32)
                    lp_c_blend = np.zeros((N_test, K), dtype=np.float32)
                    lp_u_p = np.zeros((0, K), dtype=np.float32)
                    lp_u_c = np.zeros((0, K), dtype=np.float32)

                    if run_submission and len(Z_te) > 0:
                        lp_p = log_softmax(align_cols(Z_te @ W_p.T + b_p, ro_labels))
                        comp_lora_posteriors.append((w_comp_p, softmax(lp_p, axis=1)))
                        if W_c is not None and W_c.shape[0] == 18 and tracker_test is not None:
                            lp_c_blend = compute_multires_centered_blend(Z_te, tracker_test.groups, W_c, b_c, ro_labels, weights=(w85, w72, w60))
                            comp_lora_posteriors.append((w_comp_c, softmax(lp_c_blend, axis=1)))

                    if run_benchmark and len(Z_unseen) > 0:
                        lp_u_p = log_softmax(align_cols(Z_unseen @ W_p.T + b_p, ro_labels))
                        if W_c is not None and W_c.shape[0] == 18 and tracker_unseen is not None:
                            lp_u_c = compute_multires_centered_blend(Z_unseen, tracker_unseen.groups, W_c, b_c, ro_labels, weights=(w85, w72, w60))

                    w_tot_cl = w_comp_p + (w_comp_c if (W_c is not None and W_c.shape[0] == 18) else 0.0)
                    twin_c_te = lp_c_blend if (W_c is not None and W_c.shape[0] == 18) else None
                    lp_m_te = _combine_plain_and_twin_lp(lp_p, twin_c_te, w_comp_p, w_comp_c) if (run_submission and len(lp_p) == N_test) else np.zeros((0, K), dtype=np.float32)
                    twin_c_u = lp_u_c if (W_c is not None and W_c.shape[0] == 18) else None
                    lp_m_u = _combine_plain_and_twin_lp(lp_u_p, twin_c_u, w_comp_p, w_comp_c) if (run_benchmark and len(lp_u_p) == N_unseen) else np.zeros((0, K), dtype=np.float32)
                    auditor.add_model("Qwen3-8B Claude Head", tier="Comp", weight=w_tot_cl, unseen_log_probs=lp_m_u, test_log_probs=lp_m_te)

                apply_wpca_and_spectral(
                    model_key="qwen3_8b_claude",
                    Z_te_mat=Z_te,
                    Z_tr_mat=Z_tr,
                    Z_u_mat=Z_unseen if N_unseen > 0 else None,
                    dp_w=dp_weights.get("claude_8b", 0.04),
                    prior_scale=prior_scales.get("qwen3_8b_claude", 0.30),
                    d_target=256,
                    se_floor=se_floors.get("qwen3_8b_claude", 0.35)
                )

                merge_lora(trunk3.model, claude_dir, sign=-1.0)
                del Z_tr, Z_te
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            trunk3.free_memory()
            del trunk3
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # ==============================================================================
        # TRUNK 4: DeBERTa-v3-Large
        # ==============================================================================
        for deb_key in ("deberta_p2", "deberta_inhouse"):
            if deb_key not in ckpts:
                continue
            deb_dir = ckpts[deb_key]
            base_deb_dir = verified_bases.get("deberta_v3_large") or find_base_model(input_dir, "deberta", 1024, 24)
            print(f"[Pipeline] Loading Trunk 4: DeBERTa-v3 ({deb_key}) from {deb_dir}...")
            if run_submission and N_test > 0 and run_benchmark and N_unseen > 0:
                deb_texts = test_texts + unseen_texts
                Z_test_all, Z_tr_deb, W_p_deb, b_p_deb, W_c_deb, b_c_deb, deb_labels = extract_deberta_checkpoint(
                    deb_dir, base_deb_dir, deb_texts, train_sub_texts, y_sub
                )
                Z_te_deb = Z_test_all[:N_test]
                Z_unseen_deb = Z_test_all[N_test : N_test + N_unseen]
            elif run_submission and N_test > 0:
                Z_test_all, Z_tr_deb, W_p_deb, b_p_deb, W_c_deb, b_c_deb, deb_labels = extract_deberta_checkpoint(
                    deb_dir, base_deb_dir, test_texts, train_sub_texts, y_sub
                )
                Z_te_deb = Z_test_all
                Z_unseen_deb = np.zeros((0, Z_te_deb.shape[1]), dtype=np.float32)
            elif run_benchmark and N_unseen > 0:
                Z_test_all, Z_tr_deb, W_p_deb, b_p_deb, W_c_deb, b_c_deb, deb_labels = extract_deberta_checkpoint(
                    deb_dir, base_deb_dir, unseen_texts, train_sub_texts, y_sub
                )
                Z_unseen_deb = Z_test_all
                Z_te_deb = np.zeros((0, Z_unseen_deb.shape[1]), dtype=np.float32)
            else:
                continue

            if N_unseen > 0 and len(Z_unseen_deb) == N_unseen:
                features_unseen_dict[deb_key] = Z_unseen_deb.astype(np.float16)

            if W_p_deb is None and not is_smoke:
                raise RuntimeError(f"Required classification head W_p is None for {deb_dir}")
            update_hybrid_groups(Z_te_deb, prior_scales.get(deb_key, 0.80), Z_unseen_deb if N_unseen > 0 else None)

            w_main_deb = model_weights.get(deb_key, {}).get("plain_weight", main_members.get("deberta_v3_large", {}).get("plain_weight", 0.25))
            w_comp_deb = comp_members.get("deberta_v3_large", {}).get("plain_weight", 0.18)
            w_comp_c = model_weights.get(deb_key, {}).get("twin_weight", comp_members.get("deberta_v3_large", {}).get("twin_weight", 0.06))
            lp_p_deb = None
            lp_c_blend = np.zeros((N_test, K), dtype=np.float32)
            lp_u_deb = np.zeros((0, K), dtype=np.float32)
            lp_u_c_deb = np.zeros((0, K), dtype=np.float32)

            if run_submission and len(Z_te_deb) > 0:
                if W_p_deb is not None:
                    lp_p_deb = log_softmax(align_cols(Z_te_deb @ W_p_deb.T + b_p_deb, deb_labels))
                    LOGIT_MAIN += lp_p_deb * w_main_deb
                    comp_lora_posteriors.append((w_comp_deb, softmax(lp_p_deb, axis=1)))
                if W_c_deb is not None and tracker_test is not None:
                    lp_c_blend = compute_multires_centered_blend(Z_te_deb, tracker_test.groups, W_c_deb, b_c_deb, deb_labels, weights=(w85, w72, w60))
                    comp_lora_posteriors.append((w_comp_c, softmax(lp_c_blend, axis=1)))

            if run_benchmark and len(Z_unseen_deb) > 0:
                if W_p_deb is not None:
                    lp_u_deb = log_softmax(align_cols(Z_unseen_deb @ W_p_deb.T + b_p_deb, deb_labels))
                if W_c_deb is not None and tracker_unseen is not None:
                    lp_u_c_deb = compute_multires_centered_blend(Z_unseen_deb, tracker_unseen.groups, W_c_deb, b_c_deb, deb_labels, weights=(w85, w72, w60))

            if W_p_deb is not None:
                w_tot_deb = w_main_deb + (w_comp_c if W_c_deb is not None else 0.0)
                lp_m_te = _combine_plain_and_twin_lp(lp_p_deb, lp_c_blend if W_c_deb is not None else None, w_main_deb, w_comp_c) if (run_submission and lp_p_deb is not None and len(lp_p_deb) == N_test) else np.zeros((0, K), dtype=np.float32)
                lp_m_u = _combine_plain_and_twin_lp(lp_u_deb, lp_u_c_deb if W_c_deb is not None else None, w_main_deb, w_comp_c) if (run_benchmark and len(lp_u_deb) == N_unseen) else np.zeros((0, K), dtype=np.float32)
                aud_name = "DeBERTa-v3-Large" if deb_key == "deberta_p2" else "DeBERTa-InHouse"
                auditor.add_model(aud_name, tier="Main", weight=w_tot_deb, unseen_log_probs=lp_m_u, test_log_probs=lp_m_te)

            apply_wpca_and_spectral(
                model_key=deb_key,
                Z_te_mat=Z_te_deb,
                Z_tr_mat=Z_tr_deb,
                Z_u_mat=Z_unseen_deb if N_unseen > 0 else None,
                dp_w=dp_weights.get("deberta", 0.20),
                prior_scale=prior_scales.get(deb_key, 0.80),
                d_target=256,
                se_floor=se_floors.get(deb_key, 0.10)
            )

            del Z_tr_deb, Z_te_deb, Z_test_all
            if "Z_unseen_deb" in locals():
                del Z_unseen_deb
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # ==============================================================================
        # OPTIONAL CAUSAL LLM TRUNKS: Gemma-2-9B, Llama-3.1-8B, Mistral-7B-v0.3
        # ==============================================================================
        opt_causal_specs = [
            ("gemma2_9b_ts", "gemma2", 3584, 42, 24, "Gemma-2-9B"),
            ("llama31_8b_ts", "llama", 4096, 32, 24, "Llama-3.1-8B"),
            ("mistral_7b_ts", "mistral", 4096, 32, 24, "Mistral-7B"),
        ]
        for opt_key, opt_family, opt_hidden, opt_layers, opt_default_keep, opt_disp_name in opt_causal_specs:
            if opt_key not in ckpts:
                continue
            opt_ckpt_dir = ckpts[opt_key]
            base_opt_dir = verified_bases.get(opt_key) or find_base_model(input_dir, opt_family, opt_hidden, opt_layers)
            if not base_opt_dir:
                print(f"[Pipeline] Base model for {opt_disp_name} ({opt_family}, h={opt_hidden}, l={opt_layers}) not found; skipping {opt_key}.")
                continue

            print(f"[Pipeline] Loading Trunk: {opt_disp_name} ({opt_key}) from base {base_opt_dir}...")
            meta_opt = read_checkpoint_meta(opt_ckpt_dir, default_keep=opt_default_keep, default_layers=[2, 4, 6, 8, 12, 16, 20, 24], default_max_len=447)
            opt_trunk = UniversalLoRATrunkLoader(
                base_opt_dir,
                keep_layers=meta_opt["keep"],
                layers=meta_opt["layers"],
                max_len=meta_opt["max_len"],
            )

            merged = merge_lora(opt_trunk.model, opt_ckpt_dir, sign=+1.0)
            if not merged:
                print(f"[Pipeline] Failed to merge LoRA for {opt_key}; skipping.")
                opt_trunk.free_memory()
                del opt_trunk
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue

            heads_sd = load_head_file(opt_ckpt_dir)
            opt_trunk.load_norm_weights(heads_sd)
            W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd)

            w_comp_p = model_weights.get(opt_key, {}).get("plain_weight", 0.15)
            w_comp_c = model_weights.get(opt_key, {}).get("twin_weight", 0.05)
            w_comp_tot = w_comp_p + (w_comp_c if W_c is not None else 0.0)

            opt_labels = meta_opt["labels"]
            gate_passed = True

            # Single pass for test + unseen benchmark (Pure Inference)
            Z_te, Z_unseen, Z_tr = _extract_trunk(opt_trunk, end_id=meta_opt.get("end_id"))
            if N_unseen > 0:
                features_unseen_dict[opt_key] = Z_unseen.astype(np.float16)

            # Auto-align label order directly on Z_unseen
            cand_orders = [opt_labels]
            if sorted(LABELS) not in cand_orders:
                cand_orders.append(sorted(LABELS))
            if list(LABELS) not in cand_orders:
                cand_orders.append(list(LABELS))

            best_align_order = opt_labels
            best_solo_bacc = -1.0
            best_lp_u = None
            if W_p is not None and N_unseen > 0:
                for cand_o in cand_orders:
                    try:
                        trial_lp_p = log_softmax(align_cols(Z_unseen @ W_p.T + b_p, cand_o))
                        if W_c is not None:
                            trial_lp_c = compute_multires_centered_blend(Z_unseen, tracker_unseen.groups, W_c, b_c, cand_o, weights=(w85, w72, w60))
                            trial_lp = _combine_plain_and_twin_lp(trial_lp_p, trial_lp_c, w_comp_p, w_comp_c)
                        else:
                            trial_lp = trial_lp_p
                        trial_acc = float(balanced_accuracy_score(y_unseen, trial_lp.argmax(axis=1)))
                        if trial_acc > best_solo_bacc:
                            best_solo_bacc = trial_acc
                            best_align_order = cand_o
                            best_lp_u = trial_lp
                    except Exception:
                        continue

            solo_bacc = best_solo_bacc if best_solo_bacc >= 0 else 0.0
            lp_u = best_lp_u if best_lp_u is not None else (log_softmax(align_cols(Z_unseen @ W_p.T + b_p, opt_labels)) if (W_p is not None and N_unseen > 0) else np.zeros((0, K), dtype=np.float32))
            opt_labels = best_align_order
            align_name = "canonical" if best_align_order == list(LABELS) else "alphabetical" if best_align_order == sorted(LABELS) else "custom"
            print(f"[Pipeline] {opt_key}: solo unseen BAcc = {solo_bacc:.4f} (alignment: {align_name})")

            # Free trunk memory immediately before downstream DP/Spectral math
            opt_trunk.free_memory()
            del opt_trunk
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            gate_passed = True
            if len(auditor.models) > 0 and N_unseen > 0:
                lift = auditor.evaluate_marginal_lift(candidate_unseen_lp=lp_u, weight=w_comp_tot)
                if lift < -0.002:
                    print(f"[Pipeline] {opt_key}: marginal lift {lift:+.4f} < -0.002 -> skipping blend.")
                    gate_passed = False

            if gate_passed and ("Z_te" in locals() or "Z_unseen" in locals()):
                pr_sc = prior_scales.get(opt_key, 0.40)
                update_hybrid_groups(Z_te if 'Z_te' in locals() else None, pr_sc, Z_unseen if ('Z_unseen' in locals() and N_unseen > 0) else None)

                lp_p = None
                lp_c_blend = np.zeros((N_test, K), dtype=np.float32)
                if run_submission and 'Z_te' in locals() and len(Z_te) > 0:
                    lp_p = log_softmax(align_cols(Z_te @ W_p.T + b_p, opt_labels)) if W_p is not None else None
                    if W_c is not None and tracker_test is not None:
                        lp_c_blend = compute_multires_centered_blend(Z_te, tracker_test.groups, W_c, b_c, opt_labels, weights=(w85, w72, w60))
                    if W_p is not None and lp_p is not None:
                        comp_lora_posteriors.append((w_comp_p, softmax(lp_p, axis=1)))
                    if W_c is not None:
                        comp_lora_posteriors.append((w_comp_c, softmax(lp_c_blend, axis=1)))

                apply_wpca_and_spectral(
                    model_key=opt_key,
                    Z_te_mat=Z_te if 'Z_te' in locals() else np.zeros((0, 4096), dtype=np.float32),
                    Z_tr_mat=Z_tr if 'Z_tr' in locals() else np.zeros((0, 4096), dtype=np.float32),
                    Z_u_mat=Z_unseen if (N_unseen > 0 and 'Z_unseen' in locals() and len(Z_unseen) == N_unseen) else None,
                    dp_w=dp_weights.get(opt_key, 0.05),
                    prior_scale=pr_sc,
                    d_target=256,
                    se_floor=se_floors.get(opt_key, 0.25)
                )

                if W_p is not None:
                    w_tot = w_comp_p + (w_comp_c if W_c is not None else 0.0)
                    lp_m_te = _combine_plain_and_twin_lp(lp_p, lp_c_blend if W_c is not None else None, w_comp_p, w_comp_c) if (run_submission and lp_p is not None and len(lp_p) == N_test) else np.zeros((0, K), dtype=np.float32)
                    lp_m_u = _combine_plain_and_twin_lp(lp_u, None, w_comp_p, w_comp_c) if (run_benchmark and N_unseen > 0 and 'lp_u' in locals() and lp_u is not None and len(lp_u) == N_unseen) else np.zeros((0, K), dtype=np.float32)
                    auditor.add_model(opt_disp_name, tier="Comp", weight=w_tot, unseen_log_probs=lp_m_u, test_log_probs=lp_m_te)

            if "heads_sd" in locals():
                del heads_sd
            if "Z_tr" in locals():
                del Z_tr
            if "Z_te" in locals():
                del Z_te
            if "Z_unseen" in locals():
                del Z_unseen
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # ==============================================================================
        # TRUNK 8: BGE-M3 (Dense 1024-dim Bidirectional CLS Encoder)
        # ==============================================================================
        if "bge_m3" in ckpts:
            bge_dir = ckpts["bge_m3"]
            print(f"[Pipeline] Loading Trunk 8: BGE-M3 from {bge_dir}...")
            try:
                from transformers import AutoTokenizer, AutoModel
                bge_tok = AutoTokenizer.from_pretrained(str(bge_dir))
                bge_model = AutoModel.from_pretrained(
                    str(bge_dir),
                    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                )
                bge_dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
                bge_model = bge_model.to(bge_dev)
                bge_model.eval()

                def _extract_bge_dense(texts: list[str], bsize: int = 16) -> np.ndarray:
                    embs = []
                    with torch.no_grad():
                        for i in range(0, len(texts), bsize):
                            batch = texts[i : i + bsize]
                            enc = bge_tok(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(bge_dev)
                            out = bge_model(**enc)
                            cls_repr = out.last_hidden_state[:, 0]
                            cls_norm = F.normalize(cls_repr, p=2, dim=-1)
                            embs.append(cls_norm.cpu().float().numpy())
                    return np.vstack(embs) if embs else np.zeros((0, 1024), dtype=np.float32)

                Z_bge_te = _extract_bge_dense(test_texts) if (run_submission and N_test > 0) else np.zeros((0, 1024), dtype=np.float32)
                Z_bge_u = _extract_bge_dense(unseen_texts) if (run_benchmark and N_unseen > 0) else np.zeros((0, 1024), dtype=np.float32)
                Z_bge_tr = _extract_bge_dense(train_sub_texts) if (extract_train_wpca and N_tr_sub > 0) else np.zeros((0, 1024), dtype=np.float32)

                if N_unseen > 0 and len(Z_bge_u) == N_unseen:
                    features_unseen_dict["bge_m3"] = Z_bge_u.astype(np.float16)

                # Free BGE model from GPU immediately
                del bge_model, bge_tok
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                # Check if trained classification head exists for BGE-M3
                heads_sd = load_head_file(bge_dir) if ((bge_dir / "heads.safetensors").exists() or (bge_dir / "heads.pt").exists()) else {}
                W_p, b_p, W_c, b_c = resolve_head_tensors(heads_sd) if heads_sd else (None, None, None, None)

                w_bge_p = model_weights.get("bge_m3", {}).get("plain_weight", 0.12)
                w_bge_c = model_weights.get("bge_m3", {}).get("twin_weight", 0.04)

                pr_sc = prior_scales.get("bge_m3", 0.60)
                update_hybrid_groups(Z_bge_te, pr_sc, Z_bge_u if N_unseen > 0 else None)

                lp_p_bge = None
                lp_c_blend = np.zeros((N_test, K), dtype=np.float32)
                if run_submission and len(Z_bge_te) > 0:
                    if W_p is not None:
                        lp_p_bge = log_softmax(align_cols(Z_bge_te @ W_p.T + b_p, LABELS))
                        comp_lora_posteriors.append((w_bge_p, softmax(lp_p_bge, axis=1)))
                    if W_c is not None and tracker_test is not None:
                        lp_c_blend = compute_multires_centered_blend(Z_bge_te, tracker_test.groups, W_c, b_c, LABELS, weights=(w85, w72, w60))
                        comp_lora_posteriors.append((w_bge_c, softmax(lp_c_blend, axis=1)))

                apply_wpca_and_spectral(
                    model_key="bge_m3",
                    Z_te_mat=Z_bge_te,
                    Z_tr_mat=Z_bge_tr,
                    Z_u_mat=Z_bge_u if N_unseen > 0 else None,
                    dp_w=dp_weights.get("bge_m3", 0.10),
                    prior_scale=pr_sc,
                    d_target=256,
                    se_floor=se_floors.get("bge_m3", 0.20)
                )

                if W_p is not None:
                    w_tot_bge = w_bge_p + (w_bge_c if W_c is not None else 0.0)
                    lp_m_te = _combine_plain_and_twin_lp(lp_p_bge, lp_c_blend if W_c is not None else None, w_bge_p, w_bge_c) if (run_submission and lp_p_bge is not None and len(lp_p_bge) == N_test) else np.zeros((0, K), dtype=np.float32)
                    lp_m_u = np.zeros((0, K), dtype=np.float32)
                    if run_benchmark and N_unseen > 0 and len(Z_bge_u) == N_unseen:
                        lp_u_bge = log_softmax(align_cols(Z_bge_u @ W_p.T + b_p, LABELS))
                        lp_u_c_bge = compute_multires_centered_blend(Z_bge_u, tracker_unseen.groups, W_c, b_c, LABELS, weights=(w85, w72, w60)) if (W_c is not None and tracker_unseen is not None) else None
                        lp_m_u = _combine_plain_and_twin_lp(lp_u_bge, lp_u_c_bge, w_bge_p, w_bge_c)
                    auditor.add_model("BGE-M3", tier="Comp", weight=w_tot_bge, unseen_log_probs=lp_m_u, test_log_probs=lp_m_te)

                del Z_bge_tr, Z_bge_te, Z_bge_u
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
            except Exception as e:
                print(f"[Pipeline] Warning: BGE-M3 extraction failed ({e}); continuing with remaining models.")

        # Add Secondary LoRAs to COMP_LIST
        if comp_lora_posteriors:
            tot_w = sum(w for w, _ in comp_lora_posteriors)
            P_lora_comb = sum(w * P for w, P in comp_lora_posteriors) / max(1e-8, tot_w)
            COMP_LIST.append((comp_streams.get("lora_and_twins", 0.35), P_lora_comb))

        # Add DP Posteriors to COMP_LIST
        if dp_posteriors:
            tot_dp_w = sum(w for w, _ in dp_posteriors)
            P_dp_comb = sum(w * P_dp for w, P_dp in dp_posteriors) / max(1e-8, tot_dp_w)
            COMP_LIST.append((comp_streams.get("dp_posteriors", 0.30), P_dp_comb))

        # Export fitted wPCA bundle if any models were fitted from scratch
        if len(wpca_bundle_export) > 0 and not (working_dir / "wpca_bundle.safetensors").exists():
            try:
                save_wpca_bundle(wpca_bundle_export, working_dir / "wpca_bundle.safetensors")
            except Exception as e:
                print(f"[Pipeline] Note: Could not export wpca_bundle: {e}")

        # Joint Spectral Bank (Predict from pre-trained params or fit)
        P_spectral = None
        P_spectral_u = None
        sb_cfg = comp_cfg.get("spectral_bank", {})
        plain_C = sb_cfg.get("plain_C", 0.04)
        center_C = sb_cfg.get("center_C", 0.03)
        center_blend = sb_cfg.get("center_blend", 0.35)

        if spectral_params is not None:
            print("[Pipeline] Computing Spectral Bank inference from pre-trained parameters...")
            if run_submission and spectral_blocks_te and tracker_test is not None:
                P_spectral = predict_spectral_bank(
                    spectral_blocks=spectral_blocks_te,
                    prm=spectral_params,
                    D_tfidf=tracker_test.D_tfidf,
                    D_neural_sum=tracker_test.D_neural_sum,
                    D_neural_weight_total=tracker_test.D_neural_weight_total,
                    max_cluster_size=max_cl,
                )
                COMP_LIST.append((comp_streams.get("spectral_bank", 0.15), P_spectral))
            if run_benchmark and spectral_blocks_u and tracker_unseen is not None:
                P_spectral_u = predict_spectral_bank(
                    spectral_blocks=spectral_blocks_u,
                    prm=spectral_params,
                    D_tfidf=tracker_unseen.D_tfidf,
                    D_neural_sum=tracker_unseen.D_neural_sum,
                    D_neural_weight_total=tracker_unseen.D_neural_weight_total,
                    max_cluster_size=max_cl,
                )
        elif spectral_blocks_tr:
            res_spec = fit_joint_spectral_bank(
                spectral_tr_blocks=spectral_blocks_tr,
                spectral_te_blocks=spectral_blocks_te if (run_submission and spectral_blocks_te) else [np.zeros((N_test, b.shape[1]), dtype=np.float32) for b in spectral_blocks_tr],
                y_sub=y_sub,
                prompts_sub=prompts_sub,
                D_tfidf=tracker_test.D_tfidf if tracker_test is not None else np.zeros((0, 0), dtype=np.float32),
                D_neural_sum=tracker_test.D_neural_sum if tracker_test is not None else np.zeros((0, 0), dtype=np.float32),
                D_neural_weight_total=tracker_test.D_neural_weight_total if tracker_test is not None else 0.0,
                N_test=N_test if run_submission else 0,
                plain_C=plain_C,
                center_C=center_C,
                center_blend=center_blend,
                max_cluster_size=max_cl,
                spectral_u_blocks=spectral_blocks_u if (run_benchmark and spectral_blocks_u) else None,
                D_tfidf_u=tracker_unseen.D_tfidf if tracker_unseen is not None else None,
                D_neural_sum_u=tracker_unseen.D_neural_sum if tracker_unseen is not None else None,
                D_neural_weight_total_u=tracker_unseen.D_neural_weight_total if tracker_unseen is not None else 0.0,
                N_unseen=N_unseen if run_benchmark else 0,
                return_unseen=run_benchmark,
                save_path=working_dir / "spectral_heads.safetensors",
            )
            if run_benchmark:
                P_spectral, oof_score, P_spectral_u = res_spec
            else:
                P_spectral, oof_score = res_spec
                P_spectral_u = None
            print(f"[Pipeline] Joint Spectral Bank fitted (OOF BAcc = {oof_score:.4f})")
            if run_submission and P_spectral is not None:
                COMP_LIST.append((comp_streams.get("spectral_bank", 0.15), P_spectral))

    elif is_smoke:
        print("[Pipeline] Smoke mode active: synthesizing high-fidelity mock representations...")
        np.random.seed(42)
        mock_p_lora = softmax(np.random.randn(N_test, K), axis=1)
        mock_p_dp = softmax(np.random.randn(N_test, K), axis=1)
        mock_p_spec = softmax(np.random.randn(N_test, K), axis=1)
        COMP_LIST.append((comp_streams.get("lora_and_twins", 0.35), mock_p_lora))
        COMP_LIST.append((comp_streams.get("dp_posteriors", 0.30), mock_p_dp))
        COMP_LIST.append((comp_streams.get("spectral_bank", 0.15), mock_p_spec))
        dp_posteriors.append((comp_streams.get("dp_posteriors", 0.30), mock_p_dp))
        P_spectral = mock_p_spec

    # 4. MODULE 4: Flat Per-Model Log-Space Ensemble & Dynamic Unseen Calibration
    print("[Pipeline] Module 4: Flat Per-Model Log-Space Ensemble & Dynamic Unseen Calibration...")
    extra_streams: List[Tuple[str, float, np.ndarray]] = []
    extra_streams_u: List[Tuple[str, float, np.ndarray]] = []
    w_dp = model_weights.get("wpca_dp", {}).get("plain_weight", 0.22)
    if run_submission and dp_posteriors:
        tot_dp_w = sum(w for w, _ in dp_posteriors)
        P_dp_comb = sum(w * P_dp for w, P_dp in dp_posteriors) / max(1e-8, tot_dp_w)
        extra_streams.append(("wpca_dp", w_dp, np.log(np.clip(P_dp_comb, 1e-9, 1.0))))
    elif is_smoke:
        extra_streams.append(("wpca_dp", w_dp, np.log(np.clip(mock_p_dp, 1e-9, 1.0))))

    if run_benchmark and dp_posteriors_u and N_unseen > 0:
        tot_dp_u_w = sum(w for w, _ in dp_posteriors_u)
        P_dp_u_comb = sum(w * P_dp for w, P_dp in dp_posteriors_u) / max(1e-8, tot_dp_u_w)
        extra_streams_u.append(("wpca_dp", w_dp, np.log(np.clip(P_dp_u_comb, 1e-9, 1.0))))
    elif is_smoke and run_benchmark:
        extra_streams_u.append(("wpca_dp", w_dp, np.log(np.clip(mock_p_dp, 1e-9, 1.0))))

    w_sb = model_weights.get("spectral_bank", {}).get("plain_weight", 0.12)
    if run_submission and "P_spectral" in locals() and P_spectral is not None:
        extra_streams.append(("spectral_bank", w_sb, np.log(np.clip(P_spectral, 1e-9, 1.0))))
    elif is_smoke:
        extra_streams.append(("spectral_bank", w_sb, np.log(np.clip(mock_p_spec, 1e-9, 1.0))))

    if run_benchmark and "P_spectral_u" in locals() and P_spectral_u is not None and N_unseen > 0:
        extra_streams_u.append(("spectral_bank", w_sb, np.log(np.clip(P_spectral_u, 1e-9, 1.0))))
    elif is_smoke and run_benchmark:
        extra_streams_u.append(("spectral_bank", w_sb, np.log(np.clip(mock_p_spec, 1e-9, 1.0))))

    if len(auditor.models) > 0:
        optimize_on_unseen = (run_benchmark and N_unseen >= 36 and not is_smoke)
        P_mix, active_weights_map, df_ablation = auditor.optimize_weights_and_blend(
            working_dir=working_dir,
            extra_test_log_probs=extra_streams,
            extra_unseen_log_probs=extra_streams_u,
            optimize_on_unseen=optimize_on_unseen,
        )
        print(f"[Pipeline] Flat log-space ensemble active weights: {active_weights_map}")
        R_main = P_mix
        P_mix_u = getattr(auditor, "P_mix_unseen", np.zeros((0, K), dtype=np.float64)) if run_benchmark else np.zeros((0, K), dtype=np.float64)
    else:
        main_ratio = fusion_cfg.get("main_ratio", 0.85)
        comp_ratio = fusion_cfg.get("comp_ratio", 0.15)
        P_mix, R_main = blend_main_and_complementary(LOGIT_MAIN, COMP_LIST, main_ratio=main_ratio, comp_ratio=comp_ratio)
        P_mix_u = np.zeros((0, K), dtype=np.float64)

    train_prior = np.bincount(y_sub, minlength=18).astype(np.float64) + 1.0 if len(y_sub) > 0 else None
    train_ref_texts = train_sub_texts if len(train_sub_texts) > 0 else (df_train["text"].fillna("").astype(str).tolist() if df_train is not None and "text" in df_train.columns else [])
    train_ref_y = y_sub if len(y_sub) > 0 else (df_train["model"].values if df_train is not None and "model" in df_train.columns else np.array([], dtype=int))

    # Run full inference pipeline on TEST (if run_submission)
    if run_submission and N_test > 0:
        P_final = run_full_inference_pipeline(
            P_mix=P_mix,
            texts=test_texts,
            post_cfg=post_cfg,
            R_main=R_main,
            claude_spec_logits=claude_spec_logits if "claude_spec_logits" in locals() else None,
            claude_classes=claude_classes if "claude_classes" in locals() else None,
            log_p_handwriting=log_p_handwriting if "log_p_handwriting" in locals() else None,
            rule_flags=rule_flags_test if "rule_flags_test" in locals() else None,
            train_texts=train_ref_texts,
            y_train=train_ref_y,
            train_prior=train_prior,
        )
    else:
        P_final = np.ones((N_test, K), dtype=np.float32) / float(K)

    # Run full inference pipeline on UNSEEN BENCHMARK (if run_benchmark)
    P_final_u = np.zeros((0, K), dtype=np.float32)
    if run_benchmark and N_unseen > 0:
        P_final_u = run_full_inference_pipeline(
            P_mix=P_mix_u,
            texts=unseen_texts,
            post_cfg=post_cfg,
            R_main=P_mix_u,
            claude_spec_logits=claude_spec_logits_u if "claude_spec_logits_u" in locals() else None,
            claude_classes=claude_classes if "claude_classes" in locals() else None,
            log_p_handwriting=log_p_handwriting_unseen if "log_p_handwriting_unseen" in locals() else None,
            rule_flags=rule_flags_unseen if "rule_flags_unseen" in locals() else None,
            train_texts=train_ref_texts,
            y_train=train_ref_y,
            train_prior=train_prior,
        )

        raw_unseen_bacc = float(balanced_accuracy_score(y_unseen, P_mix_u.argmax(axis=1)))
        post_unseen_bacc = float(balanced_accuracy_score(y_unseen, P_final_u.argmax(axis=1)))
        lift_post = post_unseen_bacc - raw_unseen_bacc
        print("=" * 70)
        print(f"[Pipeline Benchmark Validation] 100% Symmetrical Pipeline Parity Evaluation:")
        print(f"  - Pre-Postprocessing Ensemble Raw BAcc : {raw_unseen_bacc:.4f} ({raw_unseen_bacc*100:.2f}%)")
        print(f"  - Full Postprocessed Benchmark BAcc     : {post_unseen_bacc:.4f} ({post_unseen_bacc*100:.2f}%)")
        print(f"  - Downstream Postprocessing Lift       : {lift_post:+.4f} ({lift_post*100:+.2f}%)")
        print("=" * 70)

        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(y_unseen, P_final_u.argmax(axis=1), labels=range(K))
        print("[Pipeline Benchmark Validation] Class-by-Class Accuracies:")
        for c_idx, c_name in enumerate(LABELS):
            denom = cm[c_idx].sum()
            c_acc = cm[c_idx, c_idx] / denom if denom > 0 else 0.0
            print(f"  [{c_idx:02d}] {c_name:<28s}: {c_acc*100:5.1f}% ({cm[c_idx, c_idx]}/{denom})")

        cm_err = cm.copy()
        np.fill_diagonal(cm_err, 0)
        top_err_indices = np.dstack(np.unravel_index(np.argsort(cm_err.ravel())[::-1], cm_err.shape))[0][:10]
        print("[Pipeline Benchmark] Top 10 confused class pairs:")
        for r_idx, c_idx in top_err_indices:
            if cm_err[r_idx, c_idx] > 0:
                print(f"    True: {LABELS[r_idx]:<22} -> Pred: {LABELS[c_idx]:<22} : {cm_err[r_idx, c_idx]} errors")

        # Save clean_benchmark_1800_predictions.npz
        bench_pred_file = working_dir / "clean_benchmark_1800_predictions.npz"
        np.savez_compressed(
            bench_pred_file,
            P_mix=P_mix_u.astype(np.float32),
            P_final=P_final_u.astype(np.float32),
            y_unseen=y_unseen.astype(np.int32),
            raw_bacc=np.array([raw_unseen_bacc], dtype=np.float32),
            post_bacc=np.array([post_unseen_bacc], dtype=np.float32),
            lift=np.array([lift_post], dtype=np.float32),
            labels=np.array(LABELS, dtype=object),
        )
        print(f"[Pipeline Benchmark] Exported clean benchmark predictions: {bench_pred_file} ({bench_pred_file.stat().st_size} bytes)")

    # Auditing and forward stepwise ablation report
    test_final_preds = P_final.argmax(axis=1) if len(P_final) > 0 else np.array([])
    auditor.print_and_save_report(working_dir, final_test_preds=test_final_preds)

    # Generate Submission Artifacts
    df_sub, out_parquet, out_csv = generate_submission(P_final, test_ids, working_dir)
    elapsed = time.time() - start_time
    print(f"[Pipeline] Submission generated successfully in {elapsed:.2f}s: {out_parquet} ({len(df_sub)} rows)")

    # Export compact local evaluation cache for sub-second offline local tuning
    try:
        cache_dict = {
            "N_test": np.array([N_test], dtype=np.int32),
            "N_unseen": np.array([N_unseen], dtype=np.int32),
            "y_unseen": y_unseen.astype(np.int32) if len(y_unseen) > 0 else np.array([], dtype=np.int32),
            "groups_unseen": tracker_unseen.groups.get(0.85, np.zeros(0, dtype=np.int32)) if tracker_unseen else np.zeros(0, dtype=np.int32),
            "groups_085_u": tracker_unseen.groups.get(0.85, np.zeros(0, dtype=np.int32)) if tracker_unseen else np.zeros(0, dtype=np.int32),
            "groups_072_u": tracker_unseen.groups.get(0.72, np.zeros(0, dtype=np.int32)) if tracker_unseen else np.zeros(0, dtype=np.int32),
            "groups_065_u": tracker_unseen.groups.get(0.65, np.zeros(0, dtype=np.int32)) if tracker_unseen else np.zeros(0, dtype=np.int32),
            "groups_060_u": tracker_unseen.groups.get(0.60, np.zeros(0, dtype=np.int32)) if tracker_unseen else np.zeros(0, dtype=np.int32),
            "D_tfidf": tracker_test.D_tfidf.astype(np.float32) if (tracker_test and tracker_test.D_tfidf is not None) else np.zeros((0, 0), dtype=np.float32),
            "D_hybrid": tracker_test.D_hybrid.astype(np.float32) if (tracker_test and tracker_test.D_hybrid is not None) else np.zeros((0, 0), dtype=np.float32),
            "D_tfidf_u": tracker_unseen.D_tfidf.astype(np.float32) if (tracker_unseen and tracker_unseen.D_tfidf is not None) else np.zeros((0, 0), dtype=np.float32),
            "D_hybrid_u": tracker_unseen.D_hybrid.astype(np.float32) if (tracker_unseen and tracker_unseen.D_hybrid is not None) else np.zeros((0, 0), dtype=np.float32),
            "groups_085": tracker_test.groups.get(0.85, np.zeros(0, dtype=np.int32)) if tracker_test else np.zeros(0, dtype=np.int32),
            "groups_072": tracker_test.groups.get(0.72, np.zeros(0, dtype=np.int32)) if tracker_test else np.zeros(0, dtype=np.int32),
            "groups_065": tracker_test.groups.get(0.65, np.zeros(0, dtype=np.int32)) if tracker_test else np.zeros(0, dtype=np.int32),
            "groups_060": tracker_test.groups.get(0.60, np.zeros(0, dtype=np.int32)) if tracker_test else np.zeros(0, dtype=np.int32),
            "model_names": np.array([m["name"] for m in auditor.models], dtype=object),
            "model_tiers": np.array([m["tier"] for m in auditor.models], dtype=object),
            "model_prior_weights": np.array([m.get("prior_weight", m["weight"]) for m in auditor.models], dtype=np.float64),
            "model_learned_weights": np.array([m["weight"] for m in auditor.models], dtype=np.float64),
            "P_mix": P_mix.astype(np.float32),
            "P_final": P_final.astype(np.float32),
            "P_mix_unseen": P_mix_u.astype(np.float32) if len(P_mix_u) > 0 else np.zeros((0, K), dtype=np.float32),
            "P_final_unseen": P_final_u.astype(np.float32) if len(P_final_u) > 0 else np.zeros((0, K), dtype=np.float32),
            "test_texts": np.array(test_texts, dtype=object),
            "unseen_texts": np.array(unseen_texts, dtype=object),
        }
        for idx, m in enumerate(auditor.models):
            cache_dict[f"m_{idx}_unseen_lp"] = m["unseen_lp"].astype(np.float32)
            cache_dict[f"m_{idx}_test_lp"] = m["test_lp"].astype(np.float32)
        if "rule_flags_test" in locals() and rule_flags_test is not None:
            cache_dict["rule_flags_test"] = rule_flags_test.astype(np.float32)
        if "rule_flags_unseen" in locals() and rule_flags_unseen is not None:
            cache_dict["rule_flags_unseen"] = rule_flags_unseen.astype(np.float32)
        if "log_p_handwriting" in locals() and log_p_handwriting is not None:
            cache_dict["log_p_handwriting"] = log_p_handwriting.astype(np.float32)
        if "log_p_handwriting_unseen" in locals() and log_p_handwriting_unseen is not None:
            cache_dict["log_p_handwriting_unseen"] = log_p_handwriting_unseen.astype(np.float32)
        if "claude_spec_logits" in locals() and claude_spec_logits is not None:
            cache_dict["claude_spec_logits"] = claude_spec_logits.astype(np.float32)
        if "claude_spec_logits_u" in locals() and claude_spec_logits_u is not None:
            cache_dict["claude_spec_logits_u"] = claude_spec_logits_u.astype(np.float32)
        if "claude_classes" in locals() and claude_classes is not None:
            cache_dict["claude_classes"] = np.array(claude_classes, dtype=object)
        for ex_idx, (ex_name, ex_w, ex_lp) in enumerate(extra_streams):
            cache_dict[f"extra_{ex_idx}_name"] = np.array([ex_name], dtype=object)
            cache_dict[f"extra_{ex_idx}_weight"] = np.array([ex_w], dtype=np.float64)
            cache_dict[f"extra_{ex_idx}_lp"] = ex_lp.astype(np.float32)
        for ex_idx, (ex_name, ex_w, ex_lp_u) in enumerate(extra_streams_u):
            cache_dict[f"extra_{ex_idx}_unseen_lp"] = ex_lp_u.astype(np.float32)
        cache_dict["num_extra"] = np.array([len(extra_streams)], dtype=np.int32)

        cache_file_path = working_dir / "local_eval_cache.npz"
        np.savez_compressed(cache_file_path, **cache_dict)
        print(f"[Pipeline] Exported compact local evaluation cache: {cache_file_path} ({cache_file_path.stat().st_size} bytes)")

        # Export versioned, standardized benchmark feature bundle
        if len(features_unseen_dict) > 0 and N_unseen > 0:
            version_str = cfg.get("version") or os.environ.get("EXPERIMENT_VERSION") or os.environ.get("PIPELINE_VERSION") or "latest"
            date_str = time.strftime("%Y%m%d")
            clean_v = version_str.lower() if version_str.lower().startswith("v") else f"v{version_str}"
            bundle_dir_name = f"{clean_v}_{date_str}"

            # Standard consolidated features file for working_dir
            feat_save_dict = {
                "labels": y_unseen.astype(np.int32),
                "groups_085": tracker_unseen.groups.get(0.85, np.zeros(0, dtype=np.int32)),
                "groups_072": tracker_unseen.groups.get(0.72, np.zeros(0, dtype=np.int32)),
                "groups_065": tracker_unseen.groups.get(0.65, np.zeros(0, dtype=np.int32)),
                "groups_060": tracker_unseen.groups.get(0.60, np.zeros(0, dtype=np.int32)),
                "D_tfidf": tracker_unseen.D_tfidf.astype(np.float32) if tracker_unseen.D_tfidf is not None else np.zeros((0, 0), dtype=np.float32),
                "D_hybrid": tracker_unseen.D_hybrid.astype(np.float32) if tracker_unseen.D_hybrid is not None else np.zeros((0, 0), dtype=np.float32),
            }
            for k, v in features_unseen_dict.items():
                feat_save_dict[f"Z_unseen_{k}"] = v
            feat_file_path = working_dir / "benchmark_1800_features.npz"
            np.savez_compressed(feat_file_path, **feat_save_dict)
            print(f"[Pipeline] Exported {len(features_unseen_dict)} benchmark feature representations to: {feat_file_path} ({feat_file_path.stat().st_size} bytes)")

            # Save versioned bundle per model to target directory if available
            target_dirs = [working_dir / "benchmark_features" / bundle_dir_name]
            local_features_root = Path(__file__).resolve().parent.parent.parent / "local_data" / "benchmark_features"
            if local_features_root.parent.exists():
                target_dirs.append(local_features_root / bundle_dir_name)

            for bundle_dir in target_dirs:
                try:
                    bundle_dir.mkdir(parents=True, exist_ok=True)
                    models_meta = {}
                    for model_name, Z_val in features_unseen_dict.items():
                        model_file = f"Z_unseen_{model_name}_{clean_v}_{date_str}.npz"
                        m_path = bundle_dir / model_file
                        np.savez_compressed(
                            m_path,
                            Z_unseen=Z_val.astype(np.float16),
                            labels=y_unseen.astype(np.int32),
                            groups=tracker_unseen.groups.get(0.85, np.zeros(0, dtype=np.int32)),
                            model_name=model_name,
                            version=clean_v,
                            date=date_str,
                        )
                        models_meta[model_name] = {
                            "filename": model_file,
                            "shape": list(Z_val.shape),
                            "dtype": str(Z_val.dtype),
                            "size_mb": round(m_path.stat().st_size / (1024 * 1024), 2),
                        }

                    # Consolidated bundle archive
                    master_file = f"benchmark_1800_features_all_models_{clean_v}_{date_str}.npz"
                    np.savez_compressed(bundle_dir / master_file, **feat_save_dict)

                    # Metadata JSON
                    meta_content = {
                        "version": clean_v,
                        "date": date_str,
                        "N_samples": int(N_unseen),
                        "models": models_meta,
                        "consolidated_file": master_file,
                    }
                    (bundle_dir / "metadata.json").write_text(json.dumps(meta_content, indent=2), encoding="utf-8")
                    print(f"[Pipeline] Standardized benchmark feature bundle saved to: {bundle_dir}")
                except Exception as ex_bundle:
                    print(f"[Pipeline] Note: Could not save bundle to {bundle_dir}: {ex_bundle}")
    except Exception as e:
        print(f"[Pipeline] Note: Could not write local evaluation outputs: {e}")

    return df_sub, out_parquet


In [ ]:
if __name__ == '__main__':
    run_submission_pipeline(ENSEMBLE_CONFIG)
